# LARA-Q on INSTANCE — P-aligned validation of the 110 study checkpoints

This notebook measures the checkpoints produced by `LARA_Q_12102026.ipynb` on
the INSTANCE release. It does **not** train anything, and it does **not** reuse
the settings of the earlier `LARA_Validation_Instance_19102026` notebook.

The previous first-1500 run is kept as **pre-event evidence**: on every selected
trace its window ended before P. This notebook therefore uses a fixed, P-aligned
window and measures whether the window contains the arrivals.

## What is kept from the study

| Item | Study value | Used here |
|---|---|---|
| Component | Z (component 2) | yes |
| Input length | 1500 samples | yes |
| Normalisation | per-trace min–max in `EfficientSeismicDataset` | yes |
| Magnitude filter | `source_magnitude > 2.5` | yes |
| Distance filter | `<= 60 km` | yes |
| Split | 80/10/10, `SEED = 42` | yes |
| Calibration | 2000 training traces, per-depth `set_calibrated_delta` | yes |
| Scored set | 200 test traces | yes |
| Depths | 2, 3, 4, 5, 6, 8, 10 | yes |
| Coders | fixed, adaptive, learned | yes |
| Checkpoints | 3 learned series × 10 widths × 3 λ + 2 baselines × 10 widths | 110 |
| Bit-accurate wavelets | `db4`, `sym8`, `coif3` at 7 target rates | yes |
| Architecture 3 | classical control at 7 target rates | yes |

## What is different, and why

1. **P-aligned window.** The input is still 1500 samples of component 2, but the
   start is `clip(P - 500, 0, len(trace) - 1500)`. P is required; S is never
   filtered. S coverage is measured and gated. No bandpass and no padding are
   introduced.
2. **The data are INSTANCE, not STEAD.** The study's filter is applied to the
   INSTANCE metadata (`source_magnitude`, and the more complete of
   `source_distance_km` / `path_ep_distance_km`, preferring the study's
   `source_distance_km` on a tie).
3. **Δ is recalibrated on INSTANCE.** `meta.calib` in each checkpoint is the
   calibration from *before* training and is stale even on STEAD. It is never
   used for measurement. `set_calibrated_delta` is called per depth on latents
   from the INSTANCE training split.
4. **1c low-count policy.** If fewer than 2000 calibration traces survive, the
   run does **not** stop: it uses the available training traces, reduces the
   scored set the same way, prints a warning, and records the actual counts in
   `run_manifest.json` and in every CSV. Acceptance marks such a run
   `INVALID_LOW_COUNT`, not `COMPLETE`.
5. **Window gate.** A run with P coverage below 99 % or S coverage below 80 %
   (provisional) is written for inspection but marked `INVALID_WINDOW` or
   `PILOT_INVALID_WINDOW`, never `COMPLETE`.
6. **Protocol hash.** Every CSV carries a `protocol_hash` derived from the
   segment rule, `P_OFFSET`, coverage thresholds, input definition, split,
   actual counts, wavelet/arch-3/legacy target lists, and data paths/sizes. A
   CSV is reused only when every row has the current hash and the expected row
   set is complete.
7. **Legacy Final_LARA wavelet.** The old top-K retention curves are reproduced
   at the 10 nominal CRs and plotted on `snr_vs_cr` as `Wavelet_DB4`,
   `Wavelet_SYM8`, `Wavelet_COIF3`. They are idealized: positions, signs and
   quantisation are not charged. The bit-accurate wavelet codec remains a
   separate arm in the RD and measured-CR figures.

## Outputs

Everything is written under `<results folder>/instance_validation_24102026_p_aligned/`:

```
coded_rates/            one CSV per checkpoint
wavelet_equal_rate/     one CSV per bit-accurate wavelet
legacy_wavelet/         one CSV per legacy Final_LARA wavelet
arch3/                  arch3_rates.csv
figures/                RD, nominal-CR, measured-CR, pipeline and diagnostics
diagnostics/            checkpoint analysis, per-trace metrics, reconstruction
summary.csv             learned/baseline, bit-accurate wavelet and arch-3 rows
                        (legacy wavelet is separate)
run_manifest.json       funnel, split, counts, window coverage
acceptance.json         full-run status; acceptance_pilot.json / _quick.json
                        for those modes
provenance.json         paths and source-notebook hashes
```

## Running

Set `RESULTS_DIR` at the top of the first cell if the automatic search cannot
find the folder that contains `ckpts_baselines/`. On a GPU host the full run is
the default. For a six-checkpoint pilot set `LARA_PILOT=1`; for a short smoke run
set `QUICK = True`. On a CPU-only host set `ALLOW_CPU = True` (the notebook then
forces `QUICK`). The `.py` export runs the same cells top to bottom.


## Environment, the one path, and the study protocol


In [ ]:
"""Configuration for the P-aligned LARA-Q validation on INSTANCE (24 10 2026).

This notebook measures the 110 checkpoints that were trained by
``LARA_Q_12102026.ipynb`` on STEAD, on the INSTANCE release.  It does NOT
train anything and it does NOT inherit any setting from the earlier
``LARA_Validation_Instance_19102026`` notebook.  Every protocol constant below
is the one used by the study, except the segment rule:

``w[:1500, 2]`` is replaced by a **fixed, P-aligned window**:
``start = clip(P - P_OFFSET, 0, len(trace) - 1500)``.  P is required; the
whole 1500-sample window is taken after min-max normalisation, exactly as the
study did.  This is a separate protocol from the earlier first-1500 run, which
is kept as pre-event evidence and is never overwritten.

Only one folder has to be located on the external host: the results folder
copied from the training host, i.e. the folder that contains
``ckpts_baselines/``.  Leave ``RESULTS_DIR`` empty and the notebook walks up
from its own directory until it finds that folder.  ``LARA_RESULTS_DIR`` is
honoured as an override.

The INSTANCE release stays at the fixed paths below.  ``LARA_INSTANCE_META``
and ``LARA_INSTANCE_HDF5`` exist only so the smoke harness (and a relocated
copy of the release) can point at a synthetic/local file; a normal run does
not need them.
"""
import os
import sys
import json
import math
import time
import glob
import hashlib
import csv
import re
import types

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# The inherited plotting helpers print Unicode ticks and degree signs.  A
# Windows console pinned to cp1252 would raise from inside a figure helper.
try:
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")
    sys.stderr.reconfigure(encoding="utf-8", errors="replace")
except Exception:
    pass

# ============================================================ the one path
RESULTS_DIR = os.environ.get("LARA_RESULTS_DIR", r"")

# ============================================================ INSTANCE (fixed)
INSTANCE_META = os.environ.get(
    "LARA_INSTANCE_META",
    r"/home/sadalyom/Data_Seismic/Instance/metadata_Instance_events_v2.csv.bz2")
INSTANCE_HDF5 = os.environ.get(
    "LARA_INSTANCE_HDF5",
    r"/home/sadalyom/Data_Seismic/Instance/Instance_events_counts.hdf5")

# ============================================================ study protocol
# Exactly the study's data definition and measurement protocol.
SEED = 42
COMPONENT = 2
INPUT_LEN = 1500
MIN_MAGNITUDE = 2.5
MAX_DISTANCE_KM = 60.0

# The study's 10 widths and the CR each implies at input length 1500.
WIDTHS = [750, 500, 300, 150, 100, 75, 50, 30, 25, 15]
CRS = [max(2, int(round(INPUT_LEN / d))) for d in WIDTHS]   # 2,3,5,10,15,20,30,50,60,100
LAMBDAS = [1e-5, 1e-4, 1e-3]
LAM_HEADLINE = 1e-4

# Study measurement: 2000 calibration traces, 200 scored traces, 7 depths.
N_CALIB_TARGET = 2000
N_EVAL_TARGET = 200
N_CALIB = N_CALIB_TARGET
N_EVAL = N_EVAL_TARGET
DEPTHS = [2, 3, 4, 5, 6, 8, 10]

# Study codec / rate model.
TARGET_BITS = 5
RATE_MODE = "prior"
PRIOR_BITS = 32
DELTA_SIDE_BITS = 16
SOURCE_BITS_PER_SAMPLE = 32.0
LARA_VARIANT = "pyramid_funnel32"

# Study wavelets and targets.
WAVELETS = ["db4", "sym8", "coif3"]
WAVE_TARGETS_BPS = [0.05, 0.1, 0.2, 0.4, 0.8, 1.6, 3.0]

# Legacy Final_LARA wavelet curves: top-K retention at the 10 nominal CRs.
# These are deliberately idealized (positions, signs, quantisation and entropy
# coding are not charged).  They are plotted ONLY on snr_vs_cr; the
# bit-accurate wavelet codec remains the arm in the RD and measured-CR figures.
LEGACY_WAVELET_CRS = None          # set from CRS after any QUICK override
DO_LEGACY_WAVELET = True

# Study architecture-3 (classical) targets.
ARCH3_TARGETS_BPS = [0.05, 0.1, 0.25, 0.5, 1.0, 2.0, 4.0]
DO_ARCH3 = True
DO_ARCH1 = True

ARCH2_SERIES = ["LARA-Q2"]
ARCH1_SERIES = ["LARA-Q1", "Light_HARA-Q1"]
LEARNED_SERIES = ARCH2_SERIES + ARCH1_SERIES
BASELINE_NAMES = ["GeneralizedAutoencoder", "AE_PureConcat"]

# ============================================================ INSTANCE segment rule
# P-aligned window.  The study's `w[:1500, 2]` window is moved so that the P
# arrival sits at a fixed position inside the window.  500 samples before P
# leaves 1000 samples after P for the S arrival and the early coda.  The start
# is clipped at both ends of the trace, so a trace whose P is very early or very
# late still yields exactly 1500 samples.
#
# P is REQUIRED for this window.  A trace with a missing/non-finite/out-of-range
# P is skipped; there is no post-selection P/S *content* filter, and S is never
# used to include or exclude a trace.  S coverage is measured and gated because
# it tells the reader whether the window actually contains the S arrival.
P_OFFSET = 500
P_REQUIRED = True
P_COVERAGE_MIN = 99.0
S_COVERAGE_MIN = 80.0

SEGMENT_MODE = "p_aligned_fixed_offset"
VALIDATION_VERSION = "p_aligned_offset500_20261024"

# Counts below these are reported as INVALID_LOW_COUNT, not COMPLETE.  The run
# still writes its outputs (the 1c policy is not an abort), but it must not be
# mistaken for a protocol-sized comparison.
MIN_CALIB_VALID = N_CALIB_TARGET
MIN_EVAL_VALID = N_EVAL_TARGET

# 1c: never abort on a small pool.  If fewer than N_CALIB_TARGET calibration
# traces survive, use every available training trace and record the actual
# count; the evaluation set is reduced in the same way.
LOW_COUNT_POLICY = "reduce_calib"

# ============================================================ runtime policy
ALLOW_CPU = False
if os.environ.get("LARA_ALLOW_CPU") == "1":
    ALLOW_CPU = True
QUICK = False
if os.environ.get("LARA_QUICK") == "1":
    QUICK = True
FORCE = os.environ.get("LARA_FORCE") == "1"   # True: redo existing CSVs

# Pilot: exactly six headline checkpoints, all lambda=1e-4.  It shares the
# protocol hash and output folder with the full run, so the wavelet, legacy
# wavelet and arch-3 rows written by the pilot are reused by the full run.
# QUICK takes precedence: the CPU smoke test is never reported as a pilot.
PILOT = os.environ.get("LARA_PILOT") == "1"
PILOT_SPECS = [
    ("LARA-Q2", 750, 1e-4),
    ("LARA-Q2", 75, 1e-4),
    ("LARA-Q2", 15, 1e-4),
    ("LARA-Q1", 75, 1e-4),
    ("LARA-Q1", 15, 1e-4),
    ("Light_HARA-Q1", 75, 1e-4),
]
if QUICK:
    PILOT = False

if QUICK:
    # Smoke mode only.  The shipped default is the full grid above.
    WIDTHS = [75, 15]
    CRS = [max(2, int(round(INPUT_LEN / d))) for d in WIDTHS]
    LAMBDAS = [1e-4]
    N_CALIB = 64
    N_EVAL = 24
    DEPTHS = [2, 3, 5]
    WAVELETS = ["db4", "sym8"]
    WAVE_TARGETS_BPS = [0.1, 1.6]
    ARCH3_TARGETS_BPS = [0.25, 1.0]
    print("[QUICK] smoke mode: CRs", CRS, "| lambdas", LAMBDAS,
          "| calib", N_CALIB, "| eval", N_EVAL, "| depths", DEPTHS)


# ============================================================ resolve the folder
def _is_results_dir(d):
    """A results folder is the folder that holds ckpts_baselines/."""
    return os.path.isdir(os.path.join(d, "ckpts_baselines"))


def resolve_results_dir(explicit):
    """Return (results_dir, tried_list).

    Tries, in order: the explicit path, $LARA_RESULTS_DIR, the working
    directory, up to four parents, and the first-level sub-folders of each.
    At every level the ``Seismic 13 10 2026`` sub-folder is tried too.
    """
    cands = []
    if explicit:
        cands.append(explicit)
    env = os.environ.get("LARA_RESULTS_DIR")
    if env:
        cands.append(env)

    here = os.path.abspath(os.getcwd())
    roots = []
    for _ in range(5):
        roots.append(here)
        here = os.path.dirname(here)

    for root in roots:
        cands.append(root)
        cands.append(os.path.join(root, "Seismic 13 10 2026"))
        try:
            for name in sorted(os.listdir(root)):
                sub = os.path.join(root, name)
                if os.path.isdir(sub):
                    cands.append(sub)
                    cands.append(os.path.join(sub, "Seismic 13 10 2026"))
        except OSError:
            pass

    tried, seen = [], set()
    for c in cands:
        c = os.path.abspath(c)
        if c in seen:
            continue
        seen.add(c)
        tried.append(c)
        if _is_results_dir(c):
            return c, tried
    return None, tried


_results, _tried = resolve_results_dir(RESULTS_DIR)
if _results is None:
    raise FileNotFoundError(
        "Could not find a folder containing ckpts_baselines/.\n"
        "  Set RESULTS_DIR at the top of this cell to the copied results folder,\n"
        "  or set the LARA_RESULTS_DIR environment variable.\n"
        "  Tried:\n" + "\n".join(f"    {c}" for c in _tried[:20])
        + (f"\n    ... and {len(_tried) - 20} more" if len(_tried) > 20 else ""))
RESULTS_DIR = _results
CKPT_DIR = os.path.join(RESULTS_DIR, "ckpts_baselines")

# Outputs live inside the located folder, so the copied folder stays
# self-contained and a rerun on any machine finds the previous results.
#
# This is a FRESH directory for the P-aligned protocol.  The earlier
# instance_validation_20102026/ (invalid B1 run) and
# instance_validation_20102026_first1500/ (pre-event evidence) folders are
# deliberately left untouched.
OUT_DIR = os.environ.get(
    "LARA_OUT_DIR",
    os.path.join(RESULTS_DIR, "instance_validation_24102026_p_aligned"))
RATE_DIR = os.path.join(OUT_DIR, "coded_rates")
WAV_DIR = os.path.join(OUT_DIR, "wavelet_equal_rate")
ARCH3_DIR = os.path.join(OUT_DIR, "arch3")
LEGACY_DIR = os.path.join(OUT_DIR, "legacy_wavelet")
FIG_DIR = os.path.join(OUT_DIR, "figures")
for _d in (OUT_DIR, RATE_DIR, WAV_DIR, ARCH3_DIR, LEGACY_DIR, FIG_DIR):
    os.makedirs(_d, exist_ok=True)

# ============================================================ device
torch.manual_seed(SEED)
np.random.seed(SEED)
HAS_CUDA = torch.cuda.is_available()
if HAS_CUDA:
    DEVICE = torch.device("cuda")
elif ALLOW_CPU:
    DEVICE = torch.device("cpu")
    print("!" * 78)
    print("WARNING: no CUDA device found and ALLOW_CPU=True.")
    print("The arithmetic coder runs in Python and the wavelet search is a")
    print("(depth, K) sweep, so on CPU this is hours, not minutes.  QUICK mode")
    print("is therefore FORCED.")
    print("!" * 78)
    if not QUICK:
        QUICK = True
        PILOT = False
        WIDTHS = [75, 15]
        CRS = [max(2, int(round(INPUT_LEN / d))) for d in WIDTHS]
        LAMBDAS = [1e-4]
        N_CALIB = 64
        N_EVAL = 24
        DEPTHS = [2, 3, 5]
        WAVELETS = ["db4", "sym8"]
        WAVE_TARGETS_BPS = [0.1, 1.6]
        ARCH3_TARGETS_BPS = [0.25, 1.0]
else:
    raise RuntimeError(
        "A CUDA device is required unless ALLOW_CPU=True.\n"
        f"  torch      {torch.__version__}\n"
        f"  built with {torch.version.cuda}\n"
        "  pip install torch --index-url https://download.pytorch.org/whl/cu128\n"
        "  ...or set ALLOW_CPU = True at the top of this cell to run anyway.")

if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(SEED)
    GPU_NAME = torch.cuda.get_device_name(DEVICE)
    GPU_MEM_GB = torch.cuda.get_device_properties(DEVICE).total_memory / 1024 ** 3
else:
    GPU_NAME, GPU_MEM_GB = "CPU", 0.0

# Set after QUICK / CPU overrides so the legacy curve always uses the CR grid
# actually measured in this run.
LEGACY_WAVELET_CRS = list(CRS)

# A pilot or smoke run must never overwrite the full-run acceptance record.
if PILOT:
    ACCEPTANCE_FILE = "acceptance_pilot.json"
elif QUICK:
    ACCEPTANCE_FILE = "acceptance_quick.json"
else:
    ACCEPTANCE_FILE = "acceptance.json"

# ============================================================ preflight
for _p, _what in ((INSTANCE_META, "metadata"),
                  (INSTANCE_HDF5, "HDF5 events")):
    if not os.path.isfile(_p):
        raise FileNotFoundError(
            f"INSTANCE {_what} file not found: {_p}\n"
            "This notebook expects the INSTANCE release at the fixed paths at "
            "the top of this cell.  Copy/relocate it there (or set "
            "LARA_INSTANCE_META / LARA_INSTANCE_HDF5) and rerun.")
    print(f"instance {_what:9s} {_p}  ({os.path.getsize(_p) / 1e6:.1f} MB)")

if not os.path.isdir(CKPT_DIR):
    raise FileNotFoundError(f"checkpoint folder not found: {CKPT_DIR}")

# ============================================================ provenance
SOURCES = {
    "LARA_Q_12102026.ipynb":
        "study settings, codec mixins, make_laraq2 / make_laraq1, "
        "TopKCode, QuantizedBaseline, coders, wavelet, arch 3",
    "Final_LARA_27092026.ipynb":
        "class LARA, GeneralizedAutoencoder, AE_PureConcat, "
        "EfficientSeismicDataset, legacy WaveletCompressor",
    "Final_Ablation_27092026.ipynb":
        "AttentionBlock / EnhancedResidualBlock / PlainConvBlock, "
        "Light_Capacity_HARA",
}


def sha16(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()[:16]


def find_source(name):
    """Best-effort: the source notebooks are NOT needed at run time."""
    here = os.path.abspath(os.getcwd())
    for _ in range(4):
        for cand in (os.path.join(here, name),
                     os.path.join(here, "Models", name)):
            if os.path.isfile(cand):
                return cand
        here = os.path.dirname(here)
    return None


prov = {"assembled": time.strftime("%Y-%m-%d %H:%M:%S"),
        "results_dir": RESULTS_DIR, "ckpt_dir": CKPT_DIR, "out_dir": OUT_DIR,
        "instance_meta": INSTANCE_META, "instance_hdf5": INSTANCE_HDF5,
        "quick": bool(QUICK), "pilot": bool(PILOT),
        "pilot_specs": PILOT_SPECS, "allow_cpu": bool(ALLOW_CPU),
        "segment_mode": SEGMENT_MODE, "validation_version": VALIDATION_VERSION,
        "p_offset": P_OFFSET, "p_required": bool(P_REQUIRED),
        "p_coverage_min_pct": P_COVERAGE_MIN,
        "s_coverage_min_pct": S_COVERAGE_MIN,
        "low_count_policy": LOW_COUNT_POLICY,
        "n_calib_target": N_CALIB_TARGET, "n_eval_target": N_EVAL_TARGET,
        "min_calib_valid": MIN_CALIB_VALID, "min_eval_valid": MIN_EVAL_VALID,
        "widths": WIDTHS, "crs": CRS, "lambdas": LAMBDAS,
        "depths": DEPTHS, "wavelets": WAVELETS,
        "wave_targets_bps": WAVE_TARGETS_BPS,
        "arch3_targets_bps": ARCH3_TARGETS_BPS,
        "legacy_wavelet_crs": LEGACY_WAVELET_CRS,
        "sources": {}}
for name, role in SOURCES.items():
    path = find_source(name)
    prov["sources"][name] = {"role": role, "path": path,
                             "sha256_16": sha16(path) if path else None,
                             "found": bool(path)}
with open(os.path.join(OUT_DIR, "provenance.json"), "w", encoding="utf-8") as fh:
    json.dump(prov, fh, indent=1)

print("=" * 78)
print("LARA-Q validation on INSTANCE  (P-aligned, 24 10 2026)")
print("=" * 78)
print(f"device   {DEVICE}  {GPU_NAME}"
      + (f"  ({GPU_MEM_GB:.1f} GB)" if DEVICE.type == "cuda" else ""))
print(f"torch    {torch.__version__}  cuda {torch.version.cuda}")
print(f"results  {RESULTS_DIR}")
print(f"ckpts    {CKPT_DIR}")
print(f"out      {OUT_DIR}")
print(f"accept   {ACCEPTANCE_FILE}")
print(f"instance {INSTANCE_HDF5}")
print(f"split    {N_CALIB} calibration + {N_EVAL} scored traces, disjoint")
print(f"series   {', '.join(ARCH2_SERIES + ARCH1_SERIES)} x {len(LAMBDAS)} lambda"
      f"  vs  {', '.join(BASELINE_NAMES)}  vs  best of {', '.join(WAVELETS)}")
print(f"depths   {DEPTHS}   wavelets targets {WAVE_TARGETS_BPS}")
print(f"mode     {'QUICK' if QUICK else ('PILOT' if PILOT else 'FULL')}")
print(f"segment  {SEGMENT_MODE}  P_OFFSET={P_OFFSET}  ({VALIDATION_VERSION})")
print(f"gates    P coverage >= {P_COVERAGE_MIN:g}%   "
      f"S coverage >= {S_COVERAGE_MIN:g}%")
print(f"legacy   {', '.join(f'Wavelet_{w.upper()}' for w in WAVELETS)} "
      f"at nominal CR {LEGACY_WAVELET_CRS}")
for name in SOURCES:
    s = prov["sources"][name]
    print(f"source   {name:34s} {s['sha256_16'] or 'not on this host'}")


## Model code, inlined verbatim (architecture 2 and the baselines)


In [ ]:
"""The model code, inlined verbatim.  Nothing here is imported at run time.

Three sources, three jobs:

  Final_LARA_27092026.ipynb   class LARA, the two published baselines and the
                              AE_PureConcat architecture table
  LARA_Q_12102026.ipynb       the codec mixins, make_laraq2 and QuantizedBaseline

`class LARA` alone is not enough.  LARA-Q2 is a *different class*: make_laraq2
mixes SharedDeltaCodec and GatedEncoder into LARA's bases, so its state_dict has
gate_p, prior.* and _calib_span_t keys that no LARA ever had.  Loading a study
checkpoint into a bare LARA silently loads nothing but the trunk.
"""

# --------------------------------------------------------------------------
# from Final_LARA_27092026.ipynb -- LARA builds its optional
# variants from AttentionBlock / EnhancedResidualBlock, so these two
# come along even though pyramid_funnel32 does not use them.
# --------------------------------------------------------------------------

class AttentionBlock(nn.Module):
    """Simplified channel attention mechanism - EXACT ABLATION VERSION"""
    def __init__(self, channels):
        super().__init__()
        self.attention = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Conv1d(channels, max(4, channels//8), kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv1d(max(4, channels//8), channels, kernel_size=1),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        attention_weights = self.attention(x)
        return x * attention_weights

class EnhancedResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, use_attention=False):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.elu = nn.ELU(inplace=True)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm1d(out_channels)
        
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm1d(out_channels)
            )
        
        self.attention = AttentionBlock(out_channels) if use_attention else nn.Identity()
            
    def forward(self, x):
        residual = self.shortcut(x)
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.elu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        out += residual
        out = self.elu(out)
        out = self.attention(out)
        return out

model_2_architecture = {
    100: {"layers": 2, "feature_maps": [16, 16]},
     60: {"layers": 2, "feature_maps": [16, 16]},
     50: {"layers": 2, "feature_maps": [16, 16]},
     30: {"layers": 2, "feature_maps": [8,  8 ]},
     20: {"layers": 2, "feature_maps": [16, 16]},
     15: {"layers": 2, "feature_maps": [16, 16]},
     10: {"layers": 2, "feature_maps": [16, 16]},
      5: {"layers": 4, "feature_maps": [16, 16, 16, 16]},
      3: {"layers": 4, "feature_maps": [16, 16, 16, 16]},
      2: {"layers": 4, "feature_maps": [16, 16, 16, 16]},
}

class GeneralizedAutoencoder(nn.Module):
    def __init__(self, cr):
        super().__init__()
        self.cr = cr
        latent_dim = 1500 // cr
        
        self.encoder = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=9, stride=2, padding=4), nn.ReLU(),
            nn.Conv1d(16, 32, kernel_size=9, stride=2, padding=4), nn.ReLU(),
            nn.Flatten(),
            nn.Linear(32 * 375, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 32 * 375),
            nn.Unflatten(1, (32, 375)),
            nn.ConvTranspose1d(32, 16, kernel_size=9, stride=2, padding=4, output_padding=1), nn.ReLU(),
            nn.ConvTranspose1d(16, 1, kernel_size=9, stride=2, padding=4, output_padding=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)

class AE_PureConcat(nn.Module):
    def __init__(
        self,
        cr: int,
        input_len: int = 1500,
        kernel_size: int = 3,
        activation: nn.Module = nn.ELU,
        dropout: float = 0.1,
    ):
        super().__init__()
        cfg       = model_2_architecture[cr]
        layers    = cfg["layers"]
        fmaps     = cfg["feature_maps"]
        total_lat = input_len // cr

        # Distribute latent dimensions across branches
        base_dim  = total_lat // layers
        rem       = total_lat % layers
        latent_sizes = [base_dim + (1 if i < rem else 0) for i in range(layers)]

        # Encoder branches
        self.branches = nn.ModuleList()
        for i in range(layers):
            branch = nn.Sequential(
                nn.Conv1d(1, fmaps[i], kernel_size, padding=1),
                activation(inplace=True),
                nn.Dropout(dropout),
                nn.Flatten(),
                nn.Linear(input_len * fmaps[i], latent_sizes[i]),
                activation(inplace=True),
            )
            self.branches.append(branch)

        # Decoder: single linear layer to reconstruct full signal
        self.decoder = nn.Linear(total_lat, input_len)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x shape: (batch, 1, input_len)
        zs = [branch(x) for branch in self.branches]     # list of (batch, latent_i)
        z  = torch.cat(zs, dim=1)                        # (batch, total_lat)
        out = self.decoder(z)                            # (batch, input_len)
        return out.unsqueeze(1)                          # → (batch, 1, input_len)

# --------------------------------------------------------------------------
# from Final_LARA_27092026.ipynb -- the trained trunk.  Self-contained:
# only nn / torch / np / math.
# --------------------------------------------------------------------------

class LARA(nn.Module):
    """
    LARA - Light Autoencoder for Reconstruction & Analysis.

    Single production configuration: variant='pyramid_funnel32' (see VARIANT in the CONFIG
    cell). Deep residual + channel-attention encoder, filled projection funnel, learned
    channel-pyramid decoder, conv refinement head.

    variant='pyramid_funnel32' (LARA): FILLS THE FUNNEL - T = L (no min(latent, L) clamp),
                               so the projection width no longer collapses as CR grows, and
                               BOTTLE_CHANNELS=32 -> 6016 projection columns (cr<=30) /
                               3008 (cr>30). cr<10 is identical to 'pyramid'.
    Compression ratio is preserved exactly: latent == input_len // cr (asserted).
    MSE loss only -> fair against the verbatim baseline recipe.

    Other internal tokens kept for provenance / ablations:
    variant='pyramid'   : single latent path; channel pyramid (16->32->64) + refinement head.
    variant='pyramid_v1': exact original 'pyramid' behaviour (archival alias).
    variant='pyramid_b32'    : 32-channel bottleneck spine for cr>=10 (16 for cr<10).
    variant='pyramid_b32_hd' : b32 + high-CR decoder polish for cr>=30 (ceiling 96, k9).
    variant='multiscale': latent split coarse (ceil half) + detail (rest), finest feature
                          carried in its own code, both decoded and fused.
    variant='pyramid_funnel64': same T = L fix with B=64 (cr<=30) / 128 (cr>30) -> 12032
                               columns at EVERY CR (baseline-parity width).
    """

    BOTTLE_CHANNELS = 16

    def __init__(self, cr, input_len=1500, variant="pyramid"):
        super().__init__()
        self.cr = cr
        self.input_len = input_len
        self.variant = variant
        self.latent_dim = input_len // cr

        # ---- variant knobs ----
        b32 = variant in ("pyramid_b32", "pyramid_b32_hd") and cr >= 10
        hd = variant == "pyramid_b32_hd" and cr >= 30
        funnel = variant in ("pyramid_funnel32", "pyramid_funnel64") and cr >= 10
        if variant == "pyramid_funnel32" and funnel:
            self.BOTTLE_CHANNELS = 32
        elif variant == "pyramid_funnel64" and funnel:
            self.BOTTLE_CHANNELS = 64 if cr <= 30 else 128
        else:
            self.BOTTLE_CHANNELS = 32 if b32 else 16
        self.pyramid_ceiling = max(64, self.BOTTLE_CHANNELS) if funnel else (96 if hd else 64)
        self.final_kernel = 9 if hd else 5

        self.L = 188 if cr <= 30 else 94
        self.T = self.L if funnel else min(self.latent_dim, self.L)

        if variant == "multiscale":
            self.latent_coarse = (self.latent_dim + 1) // 2
            self.latent_detail = self.latent_dim - self.latent_coarse
            self.T2 = min(32, max(2, self.latent_dim // 8))
        else:
            self.latent_coarse = self.latent_dim
            self.latent_detail = 0
            self.T2 = None
        assert self.latent_coarse + self.latent_detail == self.latent_dim, "CR must stay exact"

        # ---- encoder stages ----
        self.stage0 = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=15, stride=2, padding=7),
            nn.BatchNorm1d(32),
            nn.ELU(inplace=True),
            AttentionBlock(32),
        )
        self.stage1 = EnhancedResidualBlock(32, 64, stride=2, use_attention=True)
        self.stage2 = EnhancedResidualBlock(64, 128, stride=2, use_attention=True)
        self.stage3 = EnhancedResidualBlock(128, 256, stride=2, use_attention=True) if cr > 30 else None
        final_channels = 256 if self.stage3 is not None else 128

        # ---- coarse bottleneck (original Light path) ----
        self.bot_conv = nn.Sequential(
            nn.Conv1d(final_channels, self.BOTTLE_CHANNELS, kernel_size=1),
            nn.ELU(inplace=True),
        )
        self.bot_pool = nn.Sequential(
            nn.AdaptiveAvgPool1d(self.T),
            nn.Flatten(),
        )
        self.coarse_head = nn.Linear(self.BOTTLE_CHANNELS * self.T, self.latent_coarse)

        # ---- detail bottleneck (multiscale only): finest 32ch @ 750 feature ----
        self.detail_block = nn.Sequential()
        self.detail_head = nn.Identity()
        if variant == "multiscale":
            self.detail_block = nn.Sequential(
                nn.Conv1d(32, 32, kernel_size=3, padding=1),
                nn.BatchNorm1d(32),
                nn.ELU(inplace=True),
                nn.AdaptiveAvgPool1d(self.T2),
                nn.Flatten(),
            )
            self.detail_head = nn.Linear(32 * self.T2, self.latent_detail)

        # ---- decoders (channel pyramid + learned upsampling) ----
        self.coarse_decoder = nn.Sequential(
            nn.Linear(self.latent_coarse, self.BOTTLE_CHANNELS * self.T),
            nn.Unflatten(1, (self.BOTTLE_CHANNELS, self.T)),
            nn.ELU(inplace=True),
            self._pyramid(self.BOTTLE_CHANNELS, self.T),
        )
        self.detail_decoder = nn.Sequential()
        if variant == "multiscale":
            self.detail_decoder = nn.Sequential(
                nn.Linear(self.latent_detail, self.BOTTLE_CHANNELS * self.T2),
                nn.Unflatten(1, (self.BOTTLE_CHANNELS, self.T2)),
                nn.ELU(inplace=True),
                self._pyramid(self.BOTTLE_CHANNELS, self.T2),
            )

        # ---- refinement head ----
        in_ch = 128 if variant == "multiscale" else self.pyramid_ceiling
        self.head = nn.Sequential(
            nn.Conv1d(in_ch, 32, kernel_size=5, padding=2),
            nn.ELU(inplace=True),
            nn.Conv1d(32, 1, kernel_size=15, padding=7),
            nn.Sigmoid(),
        )

        self.test_dimensions()

    # ------------------------------------------------------------------ #
    # encoding / decoding
    # ------------------------------------------------------------------ #
    def encoder(self, x):
        """Concatenated latent (coarse + detail) - the actual transmitted code."""
        zc = self.encode_coarse(x)
        if self.variant == "multiscale":
            zd = self.encode_detail(x)
            return torch.cat([zc, zd], dim=-1)
        return zc

    def encode_coarse(self, x):
        h = self.stage0(x)
        h = self.stage1(h)
        h = self.stage2(h)
        if self.stage3 is not None:
            h = self.stage3(h)
        f = self.bot_conv(h)
        f = self.bot_pool(f)
        return self.coarse_head(f)

    def encode_detail(self, x):
        f = self.stage0(x)
        d = self.detail_block(f)
        return self.detail_head(d)

    def forward(self, x):
        zc = self.encode_coarse(x)
        cf = self.coarse_decoder(zc)
        if self.variant == "multiscale":
            zd = self.encode_detail(x)
            df = self.detail_decoder(zd)
            fused = torch.cat([cf, df], dim=1)
            return self.head(fused)
        return self.head(cf)

    # ------------------------------------------------------------------ #
    # helpers
    # ------------------------------------------------------------------ #
    def _pyramid(self, start_channels, start_len):
        """Channel-pyramid upsampler from start_len to input_len using learned
        (nearest + conv) upsampling. Channel ramp up to the variant ceiling."""
        nlev = 6
        ratio = self.input_len / start_len
        levels = []
        cch = start_channels
        prev_len = start_len
        for k in range(nlev):
            tgt = int(round(start_len * (ratio ** ((k + 1) / nlev))))
            tgt = max(tgt, prev_len + 1)
            if k == nlev - 1:
                tgt = self.input_len
            ksize = self.final_kernel if k == nlev - 1 else 5
            nc = min(self.pyramid_ceiling, int(cch * 2))
            levels.append(nn.Sequential(
                nn.Upsample(size=tgt, mode='nearest'),
                nn.Conv1d(cch, nc, kernel_size=ksize, padding=ksize // 2, bias=False),
                nn.BatchNorm1d(nc),
                nn.ELU(inplace=True),
            ))
            cch = nc
            prev_len = tgt
        return nn.Sequential(*levels)

    def test_dimensions(self):
        try:
            with torch.no_grad():
                test_input = torch.randn(1, 1, self.input_len)
                zc = self.encode_coarse(test_input)
                assert zc.numel() == self.latent_coarse, f"coarse latent {zc.numel()} != {self.latent_coarse}"
                if self.variant == "multiscale":
                    zd = self.encode_detail(test_input)
                    assert zd.numel() == self.latent_detail, f"detail latent {zd.numel()} != {self.latent_detail}"
                total = self.encoder(test_input).numel()
                assert total == self.latent_dim, f"total latent {total} != {self.latent_dim}"
                decoded = self(test_input)
                assert decoded.shape[2] == self.input_len, f"output {decoded.shape[2]} != {self.input_len}"
                actual_cr = self.input_len / total
                print(f"[OK] Light-HARA-V2 ({self.variant}) CR={self.cr}: Input {self.input_len} -> Latent {total} "
                      f"(coarse {self.latent_coarse} + detail {self.latent_detail}) -> Output {decoded.shape[2]} "
                      f"(True CR = {actual_cr:.2f})")
        except Exception as e:
            print(f"[FAIL] Light-HARA-V2 ({self.variant}) CR={self.cr}: {e}")
            self.create_fallback_architecture()

    def create_fallback_architecture(self):
        print("Creating fallback architecture...")
        self.stage0 = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=15, stride=2, padding=7), nn.ELU(),
            nn.Conv1d(32, 64, kernel_size=9, stride=2, padding=4), nn.ELU(),
        )
        self.stage1 = nn.Identity()
        self.stage2 = nn.Identity()
        self.stage3 = None
        self.bot_conv = nn.Conv1d(64, self.BOTTLE_CHANNELS, kernel_size=1)
        self.bot_pool = nn.Sequential(nn.AdaptiveAvgPool1d(self.T), nn.Flatten())
        self.coarse_head = nn.Linear(self.BOTTLE_CHANNELS * self.T, self.latent_dim)
        self.latent_coarse = self.latent_dim
        self.latent_detail = 0
        self.variant = "pyramid"
        self.detail_block = nn.Sequential()
        self.detail_head = nn.Identity()
        self.coarse_decoder = nn.Sequential(
            nn.Linear(self.latent_dim, self.BOTTLE_CHANNELS * self.T),
            nn.Unflatten(1, (self.BOTTLE_CHANNELS, self.T)), nn.ELU(),
            nn.Upsample(size=self.input_len, mode='linear', align_corners=False),
            nn.Conv1d(self.BOTTLE_CHANNELS, 1, kernel_size=15, padding=7), nn.Sigmoid(),
        )
        self.detail_decoder = nn.Sequential()
        self.head = nn.Identity()
        print("[OK] Fallback architecture created")

# --------------------------------------------------------------------------
# from LARA_Q_12102026.ipynb -- the codec layer.
# --------------------------------------------------------------------------

class UniformQuantizer(nn.Module):
    """Single shared step size.  Additive noise in train, rounding in eval.

    Delta is a scalar Parameter so a trained step costs one number, and its
    gradient reaches it through the additive-noise path -- no straight-through
    estimator, because the training path is already linear in Delta.
    """

    def __init__(self, delta_init: float = 1.0, learn_delta: bool = False):
        super().__init__()
        if not (delta_init > 0):
            raise ValueError("delta_init must be positive")
        self.delta = nn.Parameter(torch.tensor(float(delta_init)),
                                  requires_grad=bool(learn_delta))

    def indices(self, z):
        """Integer symbol index per code position.  This is what gets coded."""
        return torch.round(z / self.delta)

    def dequantize(self, idx):
        return idx * self.delta

    def forward(self, z):
        if self.training:
            return z + self.delta * torch.empty_like(z).uniform_(-0.5, 0.5)
        return self.dequantize(self.indices(z))

    def extra_repr(self):
        return f"delta={float(self.delta.detach()):.6g}, learn_delta={self.delta.requires_grad}"

def _laplace_cdf(x, scale):
    s = scale.clamp_min(1e-9)
    xa = (x / s).clamp(-30.0, 30.0)
    return torch.where(xa <= 0, 0.5 * torch.exp(xa), 1.0 - 0.5 * torch.exp(-xa))

class CausalLaplacePrior(nn.Module):
    """Scale-only context model over the code positions.

    z has shape (B, d), read as a length-d, one-channel sequence.  Position i's
    scale depends on z_{i-K..i-1} ONLY, enforced by a left-padded causal
    convolution -- there is no path from z_i to s_i, which is what makes the
    rate non-trivial.  p(z_i) is the Laplace mass integrated across that
    position's bin, so the training rate is the rate the matching range coder
    actually achieves.
    """

    def __init__(self, n_ctx: int = 16, kernel: int = 5):
        super().__init__()
        if kernel < 2:
            raise ValueError("kernel must be >= 2 for a context to exist")
        self.kernel = int(kernel)
        self.n_ctx = int(n_ctx)
        self.ctx = nn.Conv1d(1, n_ctx, kernel_size=kernel)
        self.head = nn.Conv1d(n_ctx, 1, kernel_size=1)

    def scales(self, z):
        h = self.ctx(F.pad(z.unsqueeze(1), (self.kernel - 1, 0)))
        return F.softplus(self.head(h)).squeeze(1) + 1e-6

    def bin_prob(self, z_q, scales, delta):
        half = 0.5 * delta
        return (_laplace_cdf(z_q + half, scales) - _laplace_cdf(z_q - half, scales)).clamp_min(1e-9)

    def rate_bits(self, z, delta, floor_bits: float = 1e-9):
        s = self.scales(z)
        p = self.bin_prob(z, s, delta)
        return (-torch.log2(p).sum(dim=1)).clamp_min(floor_bits)

class SharedDeltaCodec:
    """Shared-scalar Delta calibration.

    Delta is ONE scalar: 16 bits per model.  Per-dimension float16 ranges cost
    2*d*16 bits -- 2400 bits at CR=20 against a 265-bit payload -- so a
    per-dimension convention would compare bookkeeping, not codecs.
    """

    _q_armed = False

    def quantize(self, z):
        if not self._q_armed:
            return z
        return self.quantizer(z)

    def bits_per_symbol(self):
        """Depth implied by the current step size.

        The alphabet is SYMMETRIC: depth b holds indices -2**(b-1) .. 2**(b-1)-1,
        spanning 2**b steps of width Delta and reaching +-2**(b-1)*Delta.  The
        calibrated span S is a max |z|, i.e. a HALF-width, so

            h = round(S / Delta)   positive levels
            b = ceil(log2(h + 1)) + 1

        Getting this factor of two wrong makes Delta twice as coarse as the
        nominal depth claims, and every symbol past +-2**(b-1) overflows.
        """
        span = float(self._calib_span_t.detach())
        if span <= 0:
            raise RuntimeError("step size not calibrated: call set_calibrated_delta(z)")
        h = int(round(span / float(self.quantizer.delta.detach())))
        return max(1, int(math.ceil(math.log2(h + 1))) + 1)

    def calibrate_prior(self, z_calib):
        """Bias the prior head to the calibrated latent scale.

        Without this the head starts at softplus(0)=0.693 while the latent is
        O(1e-2), so the discretised Laplace is flat across the bin: every symbol
        costs the same, the rate is identical for every trace, and the context
        contributes nothing.  Only head.bias is touched -- zeroing the conv would
        kill the gradient into it.
        """
        if getattr(self, "prior", None) is None:
            return None
        with torch.no_grad():
            target = max(float(z_calib.detach().float().abs().mean().item()), 1e-6)
            inv = math.log(math.expm1(target)) if target < 20.0 else target
            self.prior.head.bias.fill_(float(inv))
        return target

    def set_calibrated_delta(self, z_calib, target_bits=None):
        """Fit the shared step size on TRAINING latents only.  Never on val or test."""
        with torch.no_grad():
            z = z_calib.detach().float().reshape(-1)
            span = float(z.abs().max().item())
            self._calib_span_t.fill_(span)
            if target_bits is not None:
                half_levels = max(1, (1 << (int(target_bits) - 1)) - 1)
                self.quantizer.delta.data.fill_(float(span / half_levels))
            self.calibrate_prior(z_calib)
        return (float(self.quantizer.delta.detach()), self.bits_per_symbol(),
                float(self._calib_span_t.detach()))

    def rate_bits(self, z_q):
        """Expected rate in bits for an ALREADY quantized latent."""
        d = float(z_q.shape[-1])
        if getattr(self, "prior", None) is None:
            return torch.full((z_q.shape[0],), d * self.bits_per_symbol(),
                              device=z_q.device, dtype=z_q.dtype)
        return self.prior.rate_bits(z_q, self.quantizer.delta)

class LaraTrunk:
    """class LARA's conv trunk, down to the flattened B*T feature vector."""

    def _features_impl(self, x):
        h = self.stage0(x)
        h = self.stage1(h)
        h = self.stage2(h)
        if self.stage3 is not None:
            h = self.stage3(h)
        return self.bot_pool(self.bot_conv(h))

class SparseCode(nn.Module):
    """B*T quantized values pruned to the d largest-magnitude ones.

    Straight-through top-k: the mask is hard forward (so training and coding see
    the same code) and identity backward (so the encoder still gets gradient).
    Per trace, so one mask per sample.
    """

    def __init__(self, d):
        super().__init__()
        self.d = int(d)

    def forward(self, f):
        """f: (B, N) -> (masked (B, N), survivor indices (B, k))."""
        B, N = f.shape
        k = min(self.d, N)
        if k == N:
            idx = torch.arange(N, device=f.device).expand(B, N)
            return f, idx
        idx = f.abs().topk(k, dim=1).indices
        mask = torch.zeros_like(f).scatter_(1, idx, 1.0)
        out = f * mask + f * (1.0 - mask) if self.training else f * mask
        return out, idx

class GatedEncoder:
    """Identity-initialised elementwise gate on the flattened B*T feature vector.

    LARA's AttentionBlock re-weights *channels* through AdaptiveAvgPool1d(1) and
    never touches the flattened pre-projection vector, so it cannot express
    "this (channel, time) feature matters at this CR".
    """

    _gate_ready = False

    def _init_gate(self, ft, gate_mode="elem", gate_rank=64):
        if gate_mode not in ("elem", "lowrank", "none"):
            raise ValueError(f"unknown gate_mode {gate_mode!r}")
        self.gate_mode = gate_mode
        self.gate_rank = int(gate_rank)
        if gate_mode == "elem":
            self.gate_p = nn.Parameter(torch.zeros(int(ft)))
            self.gate_p_used = True
        elif gate_mode == "lowrank":
            self.gate_l1 = nn.Linear(int(ft), self.gate_rank)
            self.gate_l2 = nn.Linear(self.gate_rank, int(ft))
            nn.init.zeros_(self.gate_l2.weight)
            nn.init.zeros_(self.gate_l2.bias)
            self.gate_p = None
        else:
            self.gate_p = None
        self.feature_gate = nn.Identity()
        self.gate_dim = int(ft)
        # Arms the gate.  Without this the gate is never applied, the
        # "identity at init" check passes VACUOUSLY, and the parameter
        # silently receives no gradient -- so selftest() asserts the gate
        # is really in use, not merely present.
        self._gate_ready = True

    def _gate(self, f):
        if self.gate_mode == "elem":
            return f * (1.0 + torch.tanh(self.gate_p))
        if self.gate_mode == "lowrank":
            return f * (1.0 + torch.tanh(self.gate_l2(F.relu(self.gate_l1(f)))))
        return f

    def _features_impl(self, x):
        raise NotImplementedError

    def features(self, x):
        """Flattened, gated B*T feature vector.  The unit the gate acts on."""
        f = self._features_impl(x)
        return self._gate(f) if self._gate_ready else f

def make_laraq2(LARA):
    """LARA + feature gate + shared-Delta quantizer + causal Laplace prior."""

    # LaraTrunk must precede GatedEncoder in the bases: GatedEncoder defines
    # _features_impl as abstract, and Python resolves same-name methods
    # left-to-right, so the reverse order would shadow the real implementation.
    class LARAQ2(SharedDeltaCodec, LaraTrunk, GatedEncoder, LARA):
        def __init__(self, cr, input_len=INPUT_LEN, variant=LARA_VARIANT,
                     gate_mode="elem", gate_rank=64, delta_init=1.0,
                     learn_delta=False, prior_ctx=16, prior_kernel=5,
                     use_prior=True):
            super().__init__(cr, input_len=input_len, variant=variant)
            self._init_gate(self.BOTTLE_CHANNELS * self.T, gate_mode, gate_rank)
            self.quantizer = UniformQuantizer(delta_init, learn_delta)
            self.register_buffer("_calib_span_t", torch.zeros(()))
            self.prior = CausalLaplacePrior(prior_ctx, prior_kernel) if use_prior else None
            self.arch = 2
            self._q_armed = True
            self.test_dimensions()

        def encode_coarse(self, x):
            return self.coarse_head(self.features(x))

        def decode_latent(self, z_q):
            return self.head(self.coarse_decoder(z_q))

        def forward(self, x):
            return self.decode_latent(self.quantize(self.encode_coarse(x)))

        def describe(self):
            return _describe(self)

    return LARAQ2

# --------------------------------------------------------------------------
# from LARA_Q_12102026.ipynb -- the baseline adapter.  The base.
# prefix on GeneralizedAutoencoder / AE_PureConcat weights comes
# from this wrapper.
# --------------------------------------------------------------------------

def baseline_encode(model, name, x):
    """The code vector each baseline actually transmits."""
    if name == "AE_PureConcat":
        return torch.cat([b(x) for b in model.branches], dim=1)
    return model.encoder(x)

def baseline_decode(model, name, z):
    y = model.decoder(z)
    return y if y.dim() == 3 else y.unsqueeze(1)

def latent_dim_of(base, cr, input_len=INPUT_LEN):
    """Code length of a baseline.

    `class LARA` and `Light_Capacity_HARA` store `self.latent_dim`, but
    `GeneralizedAutoencoder` and `AE_PureConcat` keep it as a *local* in
    __init__ and never attach it, so attribute access on them raises
    AttributeError.  Fall back to the definition both classes use.
    """
    d = getattr(base, "latent_dim", None)
    return int(d) if d else int(input_len) // int(cr)

class QuantizedBaseline(SharedDeltaCodec, nn.Module):
    """Adapter putting a published architecture behind the codec contract.

    Inheriting SharedDeltaCodec is what lets the *same* trainer, the same
    calibration and the same evaluation run for every series -- one code path,
    not five.
    """

    def __init__(self, base, name, cr, input_len=INPUT_LEN, delta_init=1.0,
                 learn_delta=False):
        super().__init__()
        self.base = base
        self.name = name
        self.cr = int(cr)
        self.input_len = input_len
        self.latent_dim = latent_dim_of(base, cr, input_len)
        self.quantizer = UniformQuantizer(delta_init, learn_delta)
        self.register_buffer("_calib_span_t", torch.zeros(()))
        self.prior = None                      # published baselines have no prior
        self.arch = 2
        self._q_armed = True

    def encode_coarse(self, x):
        return baseline_encode(self.base, self.name, x)

    def decode_latent(self, z_q):
        return baseline_decode(self.base, self.name, z_q)

    def forward(self, x):
        return self.decode_latent(self.quantize(self.encode_coarse(x)))

    def describe(self):
        return {"name": self.name, "latent_dim": self.latent_dim,
                "B_times_T": int(self.latent_dim), "gate_mode": "none",
                "gate_params": 0, "quantizer_params": 1, "prior_params": 0,
                "total_params": sum(p.numel() for p in self.parameters())}



# ---------------------------------------------------------------- factories
# build_baseline_base is rewritten: the study notebook reaches into a sibling
# notebook for these two classes, which the GPU host does not have.  The classes
# themselves are the study's, copied above.


def build_baseline_base(name, cr):
    """Instantiate a published baseline architecture, untrained."""
    if name == "GeneralizedAutoencoder":
        return GeneralizedAutoencoder(cr)
    if name == "AE_PureConcat":
        return AE_PureConcat(cr, INPUT_LEN)
    raise ValueError(f"unknown baseline {name!r}")


def class_for(series):
    """The trained class for a series name, exactly as the study built it."""
    if series == "LARA-Q2":
        return make_laraq2(LARA)
    if series in BASELINE_NAMES:
        return None                      # wrapped by QuantizedBaseline instead
    raise ValueError(f"unknown series {series!r}")


def build_model(series, cr, name=None):
    """Construct an untrained model of `series` at compression ratio `cr`."""
    if series == "LARA-Q2":
        return make_laraq2(LARA)(cr, variant=LARA_VARIANT, gate_mode="elem")
    name = name or series
    base = build_baseline_base(name, cr)
    return QuantizedBaseline(base, name, cr, delta_init=1.0, learn_delta=False)


print(f"inlined: LARA, GeneralizedAutoencoder, AE_PureConcat, "
      f"LARAQ2 = make_laraq2(LARA), QuantizedBaseline")


## Architecture 1: TopKCode, LARA-Q1 and Light_HARA-Q1


In [ ]:
"""Architecture-1 support: TopKCode / make_laraq1, and Light_HARA-Q1.

Everything below is inlined verbatim from the source notebooks; the external
host does not have them.  The Light_Capacity_HARA class and its building blocks
come from Final_Ablation_27092026.ipynb, so make_light_hara_q1 can name
Base = Light_Capacity_HARA exactly as the study did.

Nothing here is imported at run time.
"""

# ---------------------------------------------------------------------------
# from LARA_Q_12102026.ipynb -- architecture 1: top-k selection + positions.
# ---------------------------------------------------------------------------
class TopKCode:
    """Architecture 1: no learned projection; the code is the feature map.

    To land on exactly d transmitted values the B*T features are quantized and
    the d largest-magnitude survivors kept -- which means the coder must also
    transmit WHERE they are.  That position cost is the whole point of the
    architecture, and it is charged exactly by code_positions() in the wavelet
    cell.
    """

    def _init_topk(self, d, n_slots, delta_init=1.0, learn_delta=False,
                    prior_ctx=16, prior_kernel=5, use_prior=True):
        self.sparse = SparseCode(d)
        self.quantizer = UniformQuantizer(delta_init, learn_delta)
        self.register_buffer("_calib_span_t", torch.zeros(()))
        self.prior = CausalLaplacePrior(prior_ctx, prior_kernel) if use_prior else None
        self.latent_BT, self.latent_d = int(n_slots), int(d)
        self.arch = 1
        self._q_armed = True

    def forward(self, x):
        f = self.encode_coarse(x)
        fm, idx = self.sparse(f)
        return self.decode_latent(self.quantize(fm))

    def _assert_topk_shape(self):
        with torch.no_grad():
            f = self.features(torch.randn(2, 1, self.input_len))
            assert f.shape == (2, self.BOTTLE_CHANNELS * self.T), \
                f"features {tuple(f.shape)} != {(2, self.BOTTLE_CHANNELS * self.T)}"
            fm, idx = self.sparse(f)
            assert idx.shape[1] == self.latent_d, \
                f"arch 1 kept {idx.shape[1]} values, need exactly {self.latent_d}"
            y = self.decode_latent(self.quantize(fm))
            assert y.shape == (2, 1, self.input_len), tuple(y.shape)
        for name in ("coarse_head", "coarse_decoder"):
            if hasattr(self, name):
                raise AssertionError(f"arch 1 must not retain {name}")

def make_laraq1(LARA):
    """LARA with architecture 1: coarse_head removed, top-k selection instead.

    Identical to LARA-Q2 except for how the code is cut from the signal, so the
    two differ only in the bottleneck and the positions it costs.
    """

    class LARAQ1(SharedDeltaCodec, TopKCode, LaraTrunk, GatedEncoder, LARA):
        def __init__(self, cr, input_len=INPUT_LEN, variant=LARA_VARIANT,
                     gate_mode="elem", gate_rank=64, delta_init=1.0,
                     learn_delta=False, prior_ctx=16, prior_kernel=5,
                     use_prior=True):
            # LARA.__init__ ends with test_dimensions(), which asserts the
            # latent is exactly d -- false for arch 1, whose code is the B*T
            # feature map.  Neutralise it for the base call and run our own.
            saved = LARA.test_dimensions
            LARA.test_dimensions = lambda self: None
            try:
                super().__init__(cr, input_len=input_len, variant=variant)
            finally:
                LARA.test_dimensions = saved

            del self._modules["coarse_head"]
            self.pyramid_tail = nn.Sequential(*list(self.coarse_decoder.children())[1:])
            del self._modules["coarse_decoder"]

            ft = self.BOTTLE_CHANNELS * self.T
            d = self.latent_dim
            if ft < d:
                raise ValueError(f"arch 1 needs B*T >= d, got B*T={ft} < d={d} at CR={cr}")
            self._init_gate(ft, gate_mode, gate_rank)
            self._init_topk(d, ft, delta_init, learn_delta, prior_ctx, prior_kernel, use_prior)
            self._assert_topk_shape()

        def encode_coarse(self, x):
            return self.features(x)          # (B, B*T), gated

        def decode_latent(self, z):
            return self.head(self.pyramid_tail(z))

        def describe(self):
            return _describe(self)

    return LARAQ1


# ---------------------------------------------------------------------------
# from Final_Ablation_27092026.ipynb -- Light_Capacity_HARA and its blocks.
# These are the ablation notebook's own definitions; they are functionally the
# same AttentionBlock / EnhancedResidualBlock the LARA cell above uses.
# ---------------------------------------------------------------------------
# =============================
# 🔹 SHARED BUILDING BLOCKS
# =============================

class AttentionBlock(nn.Module):
    """Simplified channel attention mechanism - EXACT ABLATION VERSION"""
    def __init__(self, channels):
        super().__init__()
        self.attention = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Conv1d(channels, max(4, channels//8), kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv1d(max(4, channels//8), channels, kernel_size=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        attention_weights = self.attention(x)
        return x * attention_weights


class EnhancedResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, use_attention=False):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.elu = nn.ELU(inplace=True)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm1d(out_channels)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm1d(out_channels)
            )

        self.attention = AttentionBlock(out_channels) if use_attention else nn.Identity()

    def forward(self, x):
        residual = self.shortcut(x)
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.elu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        out += residual
        out = self.elu(out)
        out = self.attention(out)
        return out


class PlainConvBlock(nn.Module):
    """Plain convolutional block without residual connections (for ablation)"""
    def __init__(self, in_channels, out_channels, stride=1, use_attention=False):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.elu = nn.ELU(inplace=True)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm1d(out_channels)

        self.attention = AttentionBlock(out_channels) if use_attention else nn.Identity()

    def forward(self, x):
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.elu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        out = self.elu(out)
        out = self.attention(out)
        return out

# =============================
# 🔹 HARA (Heavy Advanced Residual Autoencoder) - Ablation Base
# =============================
class HARA_AblationBase(nn.Module):
    """
    HARA with a TRUE information bottleneck (transferred from Final_HARA_20092026.py).

    Encoder backbone: strided conv -> attention + residual blocks (unchanged).
    Bottleneck: channel reduction -> pool -> flatten -> linear projection
    to a flat 1D latent with EXACTLY 1500//cr values.
    Decoder: linear projection -> upsample + residual blocks -> final conv.

    Ablation knobs (override in subclasses):
      USE_ATTENTION   - False removes every channel-attention block
      USE_RESIDUAL    - False replaces residual blocks with plain conv blocks
      BASE_CHANNELS   - capacity scaling (Light=32, Heavy=128)
      BOTTLE_CHANNELS - bottleneck width (Light=16, Heavy=64)
    """

    BOTTLE_CHANNELS = 32
    BASE_CHANNELS = 64
    USE_ATTENTION = True
    USE_RESIDUAL = True

    def __init__(self, cr, input_len=1500):
        super().__init__()
        self.cr = cr
        self.input_len = input_len
        self.latent_dim = input_len // cr

        self.L = 188 if cr <= 30 else 94
        self.T = min(self.latent_dim, self.L)

        block = EnhancedResidualBlock if self.USE_RESIDUAL else PlainConvBlock
        use_att = self.USE_ATTENTION

        c0 = self.BASE_CHANNELS
        c1 = c0 * 2
        c2 = c0 * 4

        # ---- encoder backbone ----
        encoder_layers = [
            nn.Conv1d(1, c0, kernel_size=15, stride=2, padding=7),
            nn.BatchNorm1d(c0),
            nn.ELU(inplace=True),
        ]
        if use_att:
            encoder_layers.append(AttentionBlock(c0))

        encoder_layers.append(block(c0, c1, stride=2, use_attention=use_att))
        encoder_layers.append(block(c1, c2, stride=2, use_attention=use_att))

        if cr > 30:
            encoder_layers.append(block(c2, c2 * 2, stride=2, use_attention=use_att))
            final_channels = c2 * 2
        else:
            final_channels = c2

        # ---- true compression bottleneck ----
        bottleneck = [
            nn.Conv1d(final_channels, self.BOTTLE_CHANNELS, kernel_size=1),
            nn.ELU(inplace=True),
            nn.AdaptiveAvgPool1d(self.T),
            nn.Flatten(),
            nn.Linear(self.BOTTLE_CHANNELS * self.T, self.latent_dim),
        ]

        self.encoder = nn.Sequential(*(encoder_layers + bottleneck))

        # ---- decoder ----
        self.upsample_factors = self.calculate_upsample_factors()

        decoder_layers = [
            nn.Linear(self.latent_dim, self.BOTTLE_CHANNELS * self.T),
            nn.Unflatten(1, (self.BOTTLE_CHANNELS, self.T)),
            nn.ELU(inplace=True),
        ]

        for factor in self.upsample_factors:
            decoder_layers.extend([
                nn.Upsample(scale_factor=factor, mode='linear', align_corners=False),
                block(self.BOTTLE_CHANNELS, self.BOTTLE_CHANNELS, stride=1, use_attention=use_att),
            ])

        decoder_layers.extend([
            nn.Conv1d(self.BOTTLE_CHANNELS, 1, kernel_size=15, padding=7),
            nn.Sigmoid(),
        ])

        self.decoder = nn.Sequential(*decoder_layers)

        self.test_dimensions()

    # ------------------------------------------------------------------ #
    # helpers
    # ------------------------------------------------------------------ #

    def calculate_upsample_factors(self):
        current_length = self.T
        factors = []
        while current_length < self.input_len:
            remaining_ratio = self.input_len / current_length
            factor = min(2.0, remaining_ratio)
            factors.append(factor)
            current_length = int(current_length * factor)
            if len(factors) >= 5:
                break
        if current_length != self.input_len:
            factors.append(self.input_len / current_length)
        return factors

    def test_dimensions(self):
        try:
            with torch.no_grad():
                test_input = torch.randn(1, 1, self.input_len)
                encoded   = self.encoder(test_input)
                latent_n  = encoded.numel() // encoded.shape[0]

                assert latent_n == self.latent_dim, f"Latent {latent_n} != {self.latent_dim}"

                decoded = self.decoder(encoded)
                if decoded.shape[2] != self.input_len:
                    self.decoder = nn.Sequential(
                        self.decoder,
                        nn.Upsample(size=self.input_len, mode='linear', align_corners=False),
                    )
                    decoded = self.decoder(encoded)

                assert decoded.shape[2] == self.input_len, f"Output {decoded.shape[2]} != {self.input_len}"
                actual_cr = self.input_len / latent_n
                print(f"[OK] {self.__class__.__name__} CR={self.cr}: Input {self.input_len} -> Latent {latent_n} -> Output {decoded.shape[2]} (True CR = {actual_cr:.2f})")
        except Exception as e:
            print(f"[FAIL] {self.__class__.__name__} CR={self.cr}: {e}")
            self.create_fallback_architecture()

    def create_fallback_architecture(self):
        print("Creating fallback architecture...")
        self.encoder = nn.Sequential(
            nn.Conv1d(1, 64, kernel_size=15, stride=2, padding=7), nn.ELU(),
            nn.Conv1d(64, 128, kernel_size=9, stride=2, padding=4), nn.ELU(),
            nn.Conv1d(128, self.BOTTLE_CHANNELS, kernel_size=1), nn.ELU(),
            nn.AdaptiveAvgPool1d(self.T), nn.Flatten(),
            nn.Linear(self.BOTTLE_CHANNELS * self.T, self.latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(self.latent_dim, self.BOTTLE_CHANNELS * self.T),
            nn.Unflatten(1, (self.BOTTLE_CHANNELS, self.T)), nn.ELU(),
            nn.Upsample(size=self.input_len, mode='linear', align_corners=False),
            nn.Conv1d(self.BOTTLE_CHANNELS, 1, kernel_size=15, padding=7), nn.Sigmoid(),
        )
        print("[OK] Fallback architecture created")

    def forward(self, x):
        return self.decoder(self.encoder(x))

    def count_params(self):
        return sum(p.numel() for p in self.parameters())


# =============================
# 🔹 ABLATION 1: BASE MODEL (Full HARA)
# =============================
class Base_Residual_Attention(HARA_AblationBase):
    """Full HARA structure (residual + attention + full capacity)."""
    BOTTLE_CHANNELS = 32
    BASE_CHANNELS = 64
    USE_ATTENTION = True
    USE_RESIDUAL = True


# =============================
# 🔹 ABLATION 2: NO ATTENTION
# =============================
class No_Attention_HARA(HARA_AblationBase):
    """HARA with every channel-attention block removed."""
    USE_ATTENTION = False
    USE_RESIDUAL = True


# =============================
# 🔹 ABLATION 3: PLAIN CONVOLUTION (No Residual)
# =============================
class Plain_Conv_HARA(HARA_AblationBase):
    """HARA with residual connections removed (attention kept)."""
    USE_ATTENTION = True
    USE_RESIDUAL = False


# =============================
# 🔹 ABLATION 4: LIGHT CAPACITY (50% channels)
# =============================
class Light_Capacity_HARA(HARA_AblationBase):
    """HARA with halved channel dimensions."""
    BOTTLE_CHANNELS = 16
    BASE_CHANNELS = 32
    USE_ATTENTION = True
    USE_RESIDUAL = True


# =============================
# 🔹 ABLATION 5: HEAVY CAPACITY (2x channels)
# =============================
class Heavy_Capacity_HARA(HARA_AblationBase):
    """HARA with doubled channel dimensions."""
    BOTTLE_CHANNELS = 64
    BASE_CHANNELS = 128
    USE_ATTENTION = True
    USE_RESIDUAL = True


# =============================
# 🔹 EXACT HARA ALIAS (matches Final_HARA_20092026.py)
# =============================
class HeavyAdvancedResidualAutoencoder(HARA_AblationBase):
    """Alias of the full HARA exactly as in Final_HARA_20092026.py."""
    BOTTLE_CHANNELS = 32
    BASE_CHANNELS = 64
    USE_ATTENTION = True
    USE_RESIDUAL = True


# ---------------------------------------------------------------------------
# from LARA_Q_12102026.ipynb -- architecture 1 on Light_Capacity_HARA.
# ---------------------------------------------------------------------------
def make_light_hara_q1(light_ns):
    """Architecture 1 on Light_Capacity_HARA.

    Built by REPLACING the bottleneck of a real Light_Capacity_HARA, so the
    encoder, its attention blocks and its upsampling decoder are the published
    ones -- only Linear(B*T -> d) and Linear(d -> B*T) are removed.  Gate, step
    size and prior match LARA-Q1 so the two architecture-1 arms differ only in
    their encoder.
    """
    Base = light_ns["Light_Capacity_HARA"]

    def _boom(self):
        raise RuntimeError(
            "Light_Capacity_HARA.test_dimensions() caught a shape failure and tried "
            "to install create_fallback_architecture(). Refusing: a silent swap to a "
            "32/64-channel, pyramid-free graph would invalidate every measurement.")

    class LightHARAQ1(SharedDeltaCodec, TopKCode, GatedEncoder, Base):
        def __init__(self, cr, input_len=INPUT_LEN, delta_init=1.0, learn_delta=False,
                     gate_mode="elem", gate_rank=64, prior_ctx=16, prior_kernel=5,
                     use_prior=True):
            # The base __init__ runs its own test_dimensions against the FULL
            # encoder; let it, then strip the bottleneck and re-test.
            super().__init__(cr, input_len)
            type(self).create_fallback_architecture = _boom
            self._fallback_disabled = True

            enc, dec = self.encoder, self.decoder
            lin_idx = max(i for i, m in enumerate(enc) if isinstance(m, nn.Linear))
            self.encoder_feature = nn.Sequential(*[enc[i] for i in range(lin_idx)])
            del self._modules["encoder"]
            self.decoder_tail = nn.Sequential(*[dec[i] for i in range(1, len(dec))])
            del self._modules["decoder"]
            self.upsample_factors = None

            ft = self.BOTTLE_CHANNELS * self.T
            d = self.latent_dim
            if ft < d:
                raise ValueError(f"arch 1 needs B*T >= d, got B*T={ft} < d={d} at CR={cr}")
            self._init_gate(ft, gate_mode, gate_rank)
            self._init_topk(d, ft, delta_init, learn_delta, prior_ctx, prior_kernel, use_prior)
            self._assert_topk_shape()

        def _features_impl(self, x):
            return self.encoder_feature(x)

        def encode_coarse(self, x):
            return self.features(x)

        def decode_latent(self, z):
            return self.decoder_tail(z)

        def describe(self):
            return _describe(self)

    return LightHARAQ1


LightHARAQ1 = make_light_hara_q1({"Light_Capacity_HARA": Light_Capacity_HARA})


def _describe(m):
    """Model summary for the log only; not a correctness dependency."""
    g = lambda mod: sum(p.numel() for p in mod.parameters())
    latent = getattr(m, "latent_dim", getattr(m, "latent_d", None))
    return {"arch": getattr(m, "arch", None), "latent_dim": latent,
            "B_times_T": getattr(m, "gate_dim", None),
            "gate_mode": getattr(m, "gate_mode", "none"),
            "prior_params": g(m.prior) if getattr(m, "prior", None) is not None else 0,
            "total_params": sum(p.numel() for p in m.parameters())}


# ---------------------------------------------------------------------------
# build_model is overridden so load_series() (verbatim, in the next cell) can
# rebuild every family this validation loads, not just LARA-Q2 and the two
# baselines.
# ---------------------------------------------------------------------------
def class_for(series):
    if series == "LARA-Q2":
        return make_laraq2(LARA)
    if series == "LARA-Q1":
        return make_laraq1(LARA)
    if series == "Light_HARA-Q1":
        return LightHARAQ1
    if series in BASELINE_NAMES:
        return None                      # wrapped by QuantizedBaseline instead
    raise ValueError(f"unknown series {series!r}")


def build_model(series, cr, name=None):
    """Untrained model of `series` at `cr`, exactly as the study built it."""
    if series == "LARA-Q2":
        return make_laraq2(LARA)(cr, variant=LARA_VARIANT, gate_mode="elem")
    if series == "LARA-Q1":
        return make_laraq1(LARA)(cr, variant=LARA_VARIANT, gate_mode="elem")
    if series == "Light_HARA-Q1":
        return LightHARAQ1(cr)
    name = name or series
    base = build_baseline_base(name, cr)
    return QuantizedBaseline(base, name, cr, delta_init=1.0, learn_delta=False)


print("arch-1 support ready for:", ", ".join(LEARNED_SERIES))


## Adaptive arithmetic coder


In [ ]:
"""Adaptive arithmetic coder used by the fixed/adaptive coding arms."""
import numpy as np
import constriction

_Cat = constriction.stream.model.Categorical
_q = constriction.stream.queue


def _probs(counts):
    return counts / counts.sum()


class AdaptiveCoder:
    """Adaptive arithmetic coder; the alphabet model is reset per call (per trace).

    `perfect=False` is deliberate and load-bearing.  constriction's "perfect"
    mode reports the exact theoretical bit count, but at a 2**14 alphabet it
    costs 23 s for 1500 symbols versus 0.05 s -- measured 300-430x slower, which
    made the arch #3 depth sweep run for hours.  The discarded probability
    precision is a fraction of a bit per symbol, the round trip is still
    verified exactly, and every arm uses the same setting, so the comparison
    stays fair.
    """

    def __init__(self, alphabet, perfect=False):
        self.A = alphabet
        self.perfect = perfect

    def encode(self, seq):
        counts = np.ones(self.A)
        enc = _q.RangeEncoder()
        for s in seq:
            enc.encode(int(s), _Cat(_probs(counts), perfect=self.perfect))
            counts[int(s)] += 1
        return enc.num_bits(), enc.get_decoder()

    def decode(self, decoder, n):
        counts = np.ones(self.A)
        out = np.empty(n, dtype=np.int64)
        for i in range(n):
            v = int(decoder.decode(_Cat(_probs(counts), perfect=self.perfect)))
            out[i] = v
            counts[v] += 1
        return out


def zeroth_order_entropy(syms, A):
    counts = np.bincount(np.asarray(syms, dtype=np.int64), minlength=A).astype(np.float64)
    p = counts / counts.sum()
    nz = p[p > 0]
    return float(-(nz * np.log2(nz)).sum())


## The three coders and the measurement pass


In [ ]:
"""The three coders, and the measurement pass that writes the rate/SNR CSVs.

The device handling here is the fix for a bug that silently voided every
measurement in the previous revision: `z_eval` lives on the GPU, so `.numpy()`
on it raised and the driver's `except Exception` swallowed it 30 times.  Every
GPU tensor is now moved explicitly, in both directions.
"""
import os
import csv

import numpy as np
import torch

DELTA_SIDE_BITS = 16          # one shared scalar, sent once per model


def nominal_alphabet(bits):
    """Symmetric alphabet of exactly 2**bits symbols, centred on zero."""
    half = 1 << (bits - 1)
    return -half, half - 1, 1 << bits, half


def clamp_to_alphabet(indices, bits):
    """Clamp symbols into the nominal alphabet; report how many needed it.

    The latent comes out of a Linear, so it has no bounding nonlinearity --
    unlike the reconstruction path, which ends in a Sigmoid.  A step size
    calibrated on the maximum of the TRAINING latents therefore does not
    strictly bound what TEST latents reach.  Clamp and COUNT; a non-zero count
    is a finding about the range estimate, not something to hide.
    """
    lo, hi, A, off = nominal_alphabet(bits)
    n_out = int(((indices < lo) | (indices > hi)).sum())
    return np.clip(indices, lo, hi), n_out, A, off


# --------------------------------------------------------------------- coders
def code_fixed(indices, bits, lat):
    return float(bits * lat)


def code_adaptive(indices, bits, lat, verify=True):
    """Adaptive arithmetic coder, alphabet restarted per trace."""
    _, _, A, off = nominal_alphabet(bits)
    q = (indices + off).astype(np.int64)
    coder = AdaptiveCoder(A)
    total = 0.0
    out = np.empty((indices.shape[0], lat), dtype=np.int64)
    for t in range(indices.shape[0]):
        nb, dec = coder.encode(q[t])
        total += nb
        if verify:
            back = coder.decode(dec, lat)
            if not np.array_equal(back - off, indices[t]):
                raise RuntimeError(f"adaptive coder round trip failed at trace {t}")
            out[t] = back
    ent = float(np.mean([zeroth_order_entropy(q[t], A) * lat for t in range(q.shape[0])]))
    return total / q.shape[0], ent, A


def code_learned(indices, scales_symbols, verify=True):
    """Quantised-Laplace range coder driven by the model's causal context prior."""
    import constriction
    QL = constriction.stream.model.QuantizedLaplace
    q = constriction.stream.queue

    lo, hi, A, _ = nominal_alphabet(int(np.abs(indices).max()).bit_length() + 1)
    n, lat = indices.shape
    total = 0.0
    for t in range(n):
        enc = q.RangeEncoder()
        enc.encode(indices[t].astype(np.int32), QL(lo, hi),
                   np.zeros(lat, dtype=np.float32), scales_symbols[t].astype(np.float32))
        total += enc.num_bits()
        if verify:
            back = enc.get_decoder().decode(QL(lo, hi),
                                            np.zeros(lat, dtype=np.float32),
                                            scales_symbols[t].astype(np.float32))
            if not np.array_equal(np.asarray(back).astype(np.int64), indices[t]):
                raise RuntimeError(f"learned coder round trip failed at trace {t}")
    return total / n, A


# ------------------------------------------------------------------ measuring
def snr_db(x, y):
    """Per-trace SNR then averaged -- not a pooled expectation."""
    out = []
    for i in range(x.shape[0]):
        a, b = x[i].reshape(-1), y[i].reshape(-1)
        mse = float(np.mean((a - b) ** 2))
        p = float(np.mean(a ** 2))
        out.append(10.0 * np.log10(p / (mse + 1e-12)) if mse > 0 else float("inf"))
    v = np.asarray(out, dtype=float)
    return float(np.mean(v[np.isfinite(v)]))


def evaluate(model, z_eval, x_eval, z_calib, n_traces, side_bits, depths,
             verify, extra=None, eval_device=None):
    """One row per (depth, coder), all measured on the SAME latents."""
    rows = []
    model.eval()
    dev = eval_device or z_eval.device
    x_eval_np = x_eval[:, 0, :]

    for b in depths:
        delta, depth, span = model.set_calibrated_delta(z_calib, target_bits=b)
        delta, span = float(delta), float(span)
        if depth != b:
            raise RuntimeError(f"calibration gave depth {depth}, asked {b}")

        # GPU -> host, explicitly.  .numpy() cannot convert a CUDA tensor.
        idx_raw = model.quantizer.indices(z_eval).cpu().numpy().astype(np.int64)
        idx, n_clamped, A_nom, _ = clamp_to_alphabet(idx_raw, b)
        n_sym = int(idx.size)
        lat = idx.shape[1]

        # decode from the CLAMPED symbols, so SNR is what a decoder achieves
        z_q = torch.as_tensor(idx, dtype=torch.float32, device=dev) * delta
        with torch.no_grad():
            y = model.decode_latent(z_q).cpu().numpy()
        s_db = snr_db(x_eval_np, y[:, 0, :])

        if getattr(model, "prior", None) is not None:
            with torch.no_grad():
                scales = (model.prior.scales(z_q) / delta).cpu().numpy()
        else:
            scales = None

        pay_fixed = code_fixed(idx, b, lat)
        pay_ad, ent_bound, A_ad = code_adaptive(idx, b, lat, verify=verify)
        pay_ln, A_ln = (code_learned(idx, scales, verify=verify)
                        if scales is not None else (float("nan"), 0))

        print(f"  b={b:2d}  clamped {n_clamped}/{n_sym} "
              f"({100.0 * n_clamped / max(1, n_sym):.3f}%)  SNR {s_db:6.2f} dB   "
              f"fixed {pay_fixed:7.1f}  adaptive {pay_ad:7.1f}  learned {pay_ln:7.1f}")

        for mode, payload, alpha in (("fixed", pay_fixed, A_ad),
                                     ("adaptive", pay_ad, A_ad),
                                     ("learned", pay_ln, A_ln)):
            if not np.isfinite(payload):
                continue
            rows.append({
                "n_clamped": n_clamped,
                "pct_clamped": round(100.0 * n_clamped / max(1, n_sym), 4),
                "latent_dim": lat, "target_bits": b, "delta": delta,
                "latent_span": span, "coder": mode, "alphabet": alpha,
                "payload_bits_per_trace": round(payload, 2),
                "bps_payload": round(payload / INPUT_LEN, 6),
                "side_bits_per_model": side_bits,
                "bps_side_amortised": round(side_bits / (n_traces * INPUT_LEN), 8),
                "bps_total_amortised": round((payload + side_bits / n_traces) / INPUT_LEN, 6),
                "compression_vs_source": round(
                    SOURCE_BITS_PER_SAMPLE / ((payload + side_bits / n_traces) / INPUT_LEN), 2),
                "entropy_bound_bits_per_trace": (round(ent_bound, 2) if mode == "adaptive" else ""),
                "snr_db": round(s_db, 3),
                **(extra or {}),
            })
    return rows


def write_rows(path, rows):
    """Write a CSV whose header is the UNION of every row's keys.

    Taking fieldnames from rows[0] alone is a latent crash: one row carrying an
    extra key aborts the write, and with it the whole run.
    """
    if not rows:
        return
    fields = list(rows[0].keys())
    for r in rows:
        fields += [k for k in r if k not in fields]
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", newline="", encoding="utf-8") as fh:
        w = csv.DictWriter(fh, fieldnames=fields, restval="")
        w.writeheader()
        w.writerows(rows)
    print(f"  [saved] {path}  ({len(rows)} rows)")


## The two published baselines, under the same shared Delta


In [ ]:
"""The two published baselines, measured exactly as the study measured them.

The baselines have no prior, so the codec is the shared-Delta quantizer plus
the fixed/adaptive coder; ``evaluate`` is the same function the study used.
"""


def measure_baseline(ckpt, name, cr, train_ds, test_data, depths=None,
                     device=None):
    """Coded rate and SNR for one baseline under the shared-Delta quantizer."""
    dev = device or DEVICE
    blob = torch.load(ckpt, map_location=dev, weights_only=False)
    meta = blob.get("meta") or {}
    base = build_baseline_base(name, cr)
    model = QuantizedBaseline(base, name, cr).to(dev)
    model.load_state_dict(blob["state_dict"], strict=True)
    model.eval()

    n_cal = min(N_CALIB, len(train_ds))
    acc = []
    for i in range(0, n_cal, 64):
        xb = torch.stack([train_ds[j] for j in range(i, min(i + 64, n_cal))]).to(dev)
        acc.append(model.encode_coarse(xb).reshape(-1).cpu())
    z_calib = torch.cat(acc)

    x_eval = test_data[:min(N_EVAL, len(test_data))]
    z_eval = model.encode_coarse(
        torch.as_tensor(x_eval, dtype=torch.float32, device=dev)).detach()

    prov = {"method": name, "model": name, "series": name, "arch": 2,
            "cr": cr, "cr_implied": INPUT_LEN / model.latent_dim,
            "latent_dim": model.latent_dim, "lam": 0.0,
            "trained_here": True}
    rows = evaluate(model, z_eval, x_eval, z_calib, len(x_eval),
                    DELTA_SIDE_BITS, depths or DEPTHS, verify=True,
                    extra=prov, eval_device=dev)
    return rows


## Wavelet top-K codec at equal coded rate


In [ ]:
"""Wavelet top-K codec at equal coded rate.

Position coding is EXACT here.  The previous revision coded positions with an
adaptive coder restarted from a *uniform* prior over all n_slots every trace,
which cost 2-4x the information-theoretic floor and made the wavelet arm a
strawman -- exactly what the manuscript says this cell exists to avoid.

The replacement is a colex-rank combinatorial code costing log2 C(n_slots, k):
rank the sorted K-subset via r = sum C(p_i, i), split r into mixed-radix digits
whose widths come from C(n,k).bit_length() alone (so the decoder needs no side
information), and range-code each digit.

Positions, signs and every subband index stream share ONE range encoder per
trace, as they would in a real bitstream.  Charging each field its own flush
would invent an overhead no codec pays.
"""
import math
import numpy as np
import pywt
import torch

LEVEL = 6
SOURCE_BITS_PER_SAMPLE = 32.0
STEP_BITS = 16                    # one scalar per subband, sent once per model
K_GRID = [1, 2, 3, 5, 8, 12, 18, 25, 35, 50, 70, 100, 140, 200,
          280, 380, 500, 650, 800, 950, 1100, 1300]

# constriction's Uniform model tops out at alphabet 2**24.
CHUNK_BITS = 24


# ================================================= exact position-set coding
def combinadic_rank(positions, n_slots):
    """Colexicographic rank of a sorted k-subset of {0 .. n_slots-1}.

    Exact Python int: C(n,k) can exceed 10**400, which no fixed width holds, so
    the rank is only ever split into bounded digits below.
    """
    return sum(math.comb(int(p), i) for i, p in enumerate(positions, start=1))


def _largest_p(r, i, n_slots):
    """Largest p with C(p, i) <= r.  Monotone in p, so binary search."""
    lo, hi = i - 1, n_slots - 1
    if math.comb(lo, i) > r:
        return None
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if math.comb(mid, i) <= r:
            lo = mid
        else:
            hi = mid - 1
    return lo


def combinadic_unrank(r, n_slots, k):
    """Inverse of combinadic_rank."""
    out = []
    for i in range(k, 0, -1):
        p = _largest_p(r, i, n_slots)
        if p is None:
            raise ValueError(f"rank {r} is not a valid {k}-subset of {n_slots}")
        out.append(p)
        r -= math.comb(p, i)
    return np.array(sorted(out), dtype=np.int64)


def position_floor_bits(n_slots, k):
    """The information-theoretic cost the coder works toward."""
    if k >= n_slots or k <= 0:
        return 0.0
    return math.lgamma(n_slots + 1) - math.lgamma(k + 1) - math.lgamma(n_slots - k + 1)


def _digit_widths(n_slots, k):
    """Mixed-radix widths covering [0, C(n_slots, k)).

    Full CHUNK_BITS digits with a narrower top digit sized from bit_length(), so
    the digit space always contains every valid rank.  Depends only on
    (n_slots, k), so the decoder rebuilds it with no side information.
    """
    n_bits = math.comb(n_slots, k).bit_length()
    m = max(1, -(-n_bits // CHUNK_BITS))
    top = max(1, min(CHUNK_BITS, n_bits - CHUNK_BITS * (m - 1)))
    return [1 << CHUNK_BITS] * (m - 1) + [1 << top]


def position_digits(keep, n_slots):
    """Rank a sorted position set and split it into bounded digits."""
    k = int(keep.size)
    if k <= 0 or k >= n_slots:
        return [], []
    widths = _digit_widths(n_slots, k)
    rem, digits = combinadic_rank(keep, n_slots), []
    for w in widths:
        digits.append(rem % w)
        rem //= w
    return digits, widths


def position_cost_bits(keep, n_slots):
    """Exact coded cost of a position set: log2 C(n_slots, k) plus byte rounding.

    Used by the wavelet arm and by arch #1, so both are charged the same model.
    """
    import constriction
    digits, widths = position_digits(np.asarray(keep, dtype=np.int64), n_slots)
    if not digits:
        return 0.0
    enc = constriction.stream.queue.RangeEncoder()
    for d, w in zip(digits, widths):
        enc.encode(int(d), constriction.stream.model.Uniform(int(w)))
    return float(enc.num_bits())


# --------------------------------------------------- one stream, many fields
class AdaptiveStream:
    """Adaptive symbol model over one field, reset per trace."""

    def __init__(self, alphabet):
        self.alphabet = int(alphabet)
        self.counts = np.ones(self.alphabet, dtype=np.float64)

    def probs(self):
        return self.counts / self.counts.sum()

    def update(self, s):
        self.counts[int(s)] += 1.0


def _band_indices(flat, band, keep, steps, depth):
    """Magnitude indices of the kept coefficients, per subband."""
    A_idx = 1 << depth
    out = np.zeros(keep.size, dtype=np.int64)
    kb = band[keep]
    for b in np.unique(kb):
        sel = np.nonzero(kb == b)[0]
        out[sel] = np.clip(np.rint(np.abs(flat[keep][sel]) / steps[b]),
                           0, A_idx - 1).astype(np.int64)
    return out


def encode_trace(flat, band, keep, steps, depth, n_slots, verify=True):
    """Encode one whole trace into a single bitstream.

    Returns (bits, signs, indices).  Positions, signs and every subband index
    stream share one range encoder, which is what a real codec does; giving each
    field its own flush would invent overhead no codec pays.
    """
    import constriction
    q = constriction.stream.queue
    M = constriction.stream.model

    A_idx = 1 << depth
    enc = q.RangeEncoder()

    digits, widths = position_digits(keep.astype(np.int64), n_slots)
    for d, w in zip(digits, widths):
        enc.encode(int(d), M.Uniform(int(w)))

    sgn = (flat[keep] >= 0).astype(np.int64)
    ss = AdaptiveStream(2)
    for v in sgn:
        enc.encode(int(v), M.Categorical(ss.probs(), perfect=False))
        ss.update(v)

    idx_all = _band_indices(flat, band, keep, steps, depth)
    kb = band[keep]
    for b in np.unique(kb):
        sel = np.nonzero(kb == b)[0]
        bs = AdaptiveStream(A_idx)
        for v in idx_all[sel]:
            enc.encode(int(v), M.Categorical(bs.probs(), perfect=False))
            bs.update(v)

    bits = float(enc.num_bits())
    if verify:
        dec = enc.get_decoder()
        back_d, back_w = [], []
        for _ in range(len(widths)):
            back_d.append(dec.decode(M.Uniform(int(widths[len(back_d)]))))
            back_w.append(widths[len(back_d) - 1])
        rank, mul = 0, 1
        for d, w in zip(back_d, back_w):
            rank += int(d) * mul
            mul *= int(w)
        if widths and not np.array_equal(combinadic_unrank(rank, n_slots, keep.size), keep):
            raise RuntimeError(f"position coder round trip failed for {keep.size}/{n_slots}")
        ss = AdaptiveStream(2)
        for i in range(sgn.size):
            v = int(dec.decode(M.Categorical(ss.probs(), perfect=False)))
            if v != int(sgn[i]):
                raise RuntimeError("sign round trip failed")
            ss.update(v)
        for b in np.unique(kb):
            sel = np.nonzero(kb == b)[0]
            bs = AdaptiveStream(A_idx)
            for j, i in enumerate(sel):
                v = int(dec.decode(M.Categorical(bs.probs(), perfect=False)))
                if v != int(idx_all[i]):
                    raise RuntimeError("index round trip failed")
                bs.update(v)
    return bits, sgn, idx_all


def _selftest_positions():
    print("\n--- exact position coder " + "-" * 40)
    import constriction
    q = constriction.stream.queue
    M = constriction.stream.model
    rng = np.random.default_rng(0)
    worst = 0.0
    for n_slots, k in ((1539, 8), (1539, 18), (1539, 35), (1539, 200),
                       (3008, 75), (6016, 75), (240, 15)):
        keep = np.sort(rng.choice(n_slots, size=k, replace=False)).astype(np.int64)
        rank = combinadic_rank(keep, n_slots)
        assert np.array_equal(combinadic_unrank(rank, n_slots, k), keep), "unrank mismatch"
        digits, widths = position_digits(keep, n_slots)
        enc = q.RangeEncoder()
        for d, w in zip(digits, widths):
            enc.encode(int(d), M.Uniform(int(w)))
        floor = position_floor_bits(n_slots, k)
        oh = enc.num_bits() / floor
        worst = max(worst, oh)
        print(f"    C({n_slots:5d},{k:3d})  floor {floor:8.1f} bits  "
              f"coded {enc.num_bits():8d}  {oh:.4f}x")
    print(f"    exact bijection + round trip; worst overhead {worst:.3f}x "
          f"(byte alignment, not a modelling choice)")


# ------------------------------------------------------------------ transform
def dwt(sig, wavelet, level=LEVEL):
    return pywt.wavedec(sig, wavelet, level=level)


def flatten(coeffs):
    flat = np.concatenate([c.reshape(-1) for c in coeffs])
    band = np.concatenate([np.full(c.size, i, dtype=np.int64) for i, c in enumerate(coeffs)])
    return flat, band


def unflatten(flat, shapes):
    out, off = [], 0
    for shp in shapes:
        n = int(np.prod(shp))
        out.append(flat[off:off + n].reshape(shp))
        off += n
    return out


def topk(flat, k):
    if k >= flat.size:
        return np.arange(flat.size)
    return np.sort(np.argsort(-np.abs(flat), kind="stable")[:k])


# --------------------------------------------------------------- calibration
def fit_band_max(cal_pairs):
    """Per-band max |c| RANGE from calibration traces.

    This is a RANGE, not a quantizer step.  Using it directly as the step makes
    |c|/step fall in [0,1] for every coefficient, so rint() emits only 0 or 1 and
    the quantizer is effectively 1 bit at every nominal depth.
    """
    n_bands = int(max(int(b.max()) for _, b in cal_pairs)) + 1
    mx = np.zeros(n_bands, dtype=np.float64)
    for flat, band in cal_pairs:
        a = np.abs(flat)
        for b in range(n_bands):
            sel = band == b
            if sel.any():
                mx[b] = max(mx[b], float(a[sel].max()))
    return np.maximum(mx, 1e-12)


def steps_for(band_max, depth):
    """Per-band step.  Only magnitudes are quantized (signs travel separately),
    so a band spans [0, max] and needs 2**depth levels: step = max/(2**depth-1)."""
    return np.maximum(band_max / ((1 << depth) - 1), 1e-12)


# ------------------------------------------------------------------ measure
SHAPES = None


def decode_trace(flat_n, band, shapes, keep, steps, sgn, idx_all, wavelet, sig_len):
    """Rebuild from the transmitted symbols only."""
    kb = band[keep]
    vals = np.zeros(keep.size, dtype=np.float64)
    for b in np.unique(kb):
        sel = np.nonzero(kb == b)[0]
        vals[sel] = idx_all[sel] * steps[b]
    vals = np.where(sgn > 0, vals, -vals)
    flat = np.zeros(flat_n, dtype=np.float64)
    flat[keep] = vals
    y = pywt.waverec(unflatten(flat, shapes), wavelet)
    return y[:sig_len] if len(y) >= sig_len else np.pad(y, (0, sig_len - len(y)))


def measure_wavelet(wavelet, traces, sigs, k, depth, steps, n_bands, verify=True):
    tot_bits = 0.0
    recs = []
    n_coeff = traces[0][0].size
    n_slots = n_coeff
    for t, (flat, band) in enumerate(traces):
        keep = topk(flat, k)
        bits, sgn, idx_all = encode_trace(flat, band, keep, steps, depth,
                                          n_slots, verify=verify)
        tot_bits += bits
        if verify:
            recs.append(decode_trace(flat.size, band, SHAPES, keep, steps, sgn,
                                     idx_all, wavelet, len(sigs[t])))
    n = len(traces)
    payload = tot_bits / n
    side = STEP_BITS * n_bands
    total = (payload + side / n) / INPUT_LEN
    floor = position_floor_bits(n_slots, k)
    # ideal = exact positions + 1 sign bit each + entropy of the indices
    ideal = floor + k
    return {"wavelet": wavelet, "K": k, "depth": depth, "n_bands": n_bands,
            "n_coeff": int(n_coeff), "n_slots": int(n_slots),
            "payload_bits_per_trace": round(payload, 2),
            "bps_payload": round(payload / INPUT_LEN, 6),
            "position_floor_bits": round(floor, 2),
            "coder_overhead_vs_ideal": round(payload / ideal, 4) if ideal > 0 else "",
            "side_bits_per_model": side,
            "bps_total_amortised": round(total, 6),
            "compression_vs_source": round(SOURCE_BITS_PER_SAMPLE / total, 2),
            "snr_db": round(snr_db(np.asarray(sigs), np.asarray(recs)) if verify
                            else float("nan"), 3)}


def search(wavelet, traces, sigs, target_bps, depths, band_max, n_bands,
           verify=True, k_grid=K_GRID, tol=0.08):
    """Best-SNR (depth, K) whose coded BPS lands within `tol` of the target.

    Several (depth, K) pairs land near the same rate and are NOT equivalent, so
    every in-band candidate is measured and the best SNR wins.  If nothing is in
    band the nearest is returned, so the shortfall stays visible in abs_err.
    """
    rows = []
    ceil = target_bps * (1 + tol)
    for depth in depths:
        steps = steps_for(band_max, depth)
        for k in k_grid:
            r = measure_wavelet(wavelet, traces, sigs, k, depth, steps, n_bands, verify)
            r["target_bps"] = target_bps
            r["abs_err"] = abs(r["bps_total_amortised"] - target_bps)
            rows.append(r)
            if r["bps_total_amortised"] > ceil:
                break

    near = [r for r in rows if r["bps_total_amortised"] <= target_bps * (1 + tol)]
    if near:
        best = max(near, key=lambda r: (r["snr_db"] if np.isfinite(r["snr_db"]) else -1e9,
                                        -r["bps_total_amortised"]))
    else:
        best = min(rows, key=lambda r: r["abs_err"])
        print(f"    [warn] {wavelet}: nothing within {tol:.0%} of {target_bps:.3f} bps; "
              f"closest {best['bps_total_amortised']:.4f} bps "
              f"(short by {best['abs_err']:.4f}). Reported, not hidden.")
    best["n_candidates"] = len(rows)
    best["n_in_band"] = len(near)
    return best


_selftest_positions()


## Architecture 1 measurement and architecture 3 (classical)


In [ ]:
# ---------------------------------------------------------------- arch 1 measure
def measure_topk(ckpt, factory, test_data, train_ds, name, depths=None, device=None):
    """Coded rate and SNR for any architecture-1 model, positions charged exactly.

    The transmitted payload is d quantized values PLUS log2 C(B*T, d) bits of
    positions, so arch 1 is charged the same combinatorial model the wavelet arm
    uses rather than being handed a free reduction.
    """
    dev = device or DEVICE
    blob = torch.load(ckpt, map_location=dev, weights_only=False)
    meta = blob.get("meta") or {}
    cr = meta.get("cr", 20)
    model = factory(cr).to(dev)
    model.load_state_dict(blob["state_dict"], strict=True)
    model.eval()

    n_cal = min(N_CALIB, len(train_ds))
    acc = []
    with torch.no_grad():
        for i in range(0, n_cal, 64):
            xb = torch.stack([train_ds[j] for j in range(i, min(i + 64, n_cal))]).to(dev)
            acc.append(model.encode_coarse(xb).reshape(-1).cpu())
    z_calib = torch.cat(acc)

    # the encoder needs the channel axis; SNR needs the squeezed version
    x_t = torch.as_tensor(test_data[:min(N_EVAL, len(test_data))],
                          dtype=torch.float32, device=dev)
    x_eval = np.asarray(test_data[:min(N_EVAL, len(test_data))])[:, 0, :]
    with torch.no_grad():
        f_eval = model.encode_coarse(x_t).cpu().numpy()

    n_slots = f_eval.shape[1]
    d = model.latent_d
    rows = []
    for b in (depths or DEPTHS):
        delta, depth, _ = model.set_calibrated_delta(z_calib, target_bits=b)
        delta = float(delta)
        payloads, pos_bits, recs = [], [], []
        for t in range(f_eval.shape[0]):
            f = torch.as_tensor(f_eval[t], dtype=torch.float32, device=dev).unsqueeze(0)
            fm, idx = model.sparse(f)
            q_all = model.quantizer.indices(fm.squeeze(0)).cpu().numpy().astype(np.int64)
            sel = np.sort(idx.squeeze(0).cpu().numpy())
            qv = q_all[sel]
            pb = position_cost_bits(sel.astype(np.int64), n_slots)
            pos_bits.append(pb)
            payloads.append(qv.size * b + pb)
            full = np.zeros(n_slots, dtype=np.float32)
            full[sel] = qv * delta
            with torch.no_grad():
                y = model.decode_latent(torch.as_tensor(full, device=dev).unsqueeze(0))
            recs.append(y.cpu().numpy()[0, 0])
        s_db = snr_db(x_eval, np.asarray(recs))
        pay = float(np.mean(payloads))
        floor = position_floor_bits(n_slots, d)
        side = DELTA_SIDE_BITS + (sum(p.numel() for p in model.prior.parameters())
                                  if getattr(model, "prior", None) is not None else 0) * PRIOR_BITS
        rows.append({"method": name, "K": d, "depth": b, "latent_slots": n_slots,
                     "payload_bits_per_trace": round(pay, 2),
                     "bps_payload": round(pay / INPUT_LEN, 6),
                     "position_floor_bits": round(floor, 2),
                     "position_bits_measured": round(float(np.mean(pos_bits)), 2),
                     "coder_overhead_vs_ideal": round(
                         float(np.mean(pos_bits)) / (floor + d), 4) if floor else "",
                     "side_bits_per_model": side,
                     "bps_total_amortised": round(
                         (pay + side / max(1, len(recs))) / INPUT_LEN, 6),
                     "compression_vs_source": round(
                         SOURCE_BITS_PER_SAMPLE / ((pay + side / max(1, len(recs))) / INPUT_LEN), 2),
                     "snr_db": round(s_db, 3),
                     "cr": cr, "cr_implied": INPUT_LEN / d,
                     "coder": "topk+exact-positions"})
    return rows


# ==================================================== arch 3 classical control
class Arch3Classical(nn.Module):
    """Architecture 3: no encoder, no transform.  Raw trace in, quantized out."""

    def __init__(self, input_len=INPUT_LEN, delta_init=1.0):
        super().__init__()
        self.input_len = int(input_len)
        self.quantizer = UniformQuantizer(delta_init, False)
        self.arch = 3

    def forward(self, x):
        return self.quantizer(x)

    def count_params(self):
        return 0


def measure_arch3(test_data, train_ds, targets=None, device=None):
    """Rate ladder for arch 3.  Depth is the only knob; cr is NaN by length."""
    dev = device or DEVICE
    n_cal = min(N_CALIB, len(train_ds))
    with torch.no_grad():
        cal = torch.stack([train_ds[i] for i in range(n_cal)]).to(dev)
        span = float(cal.abs().max())
    x_eval = np.asarray(test_data[:min(N_EVAL, len(test_data))])[:, 0, :]
    A = len(x_eval)
    rows = []
    for tgt in (targets or ARCH3_TARGETS_BPS):
        best = None
        # fixed-width rate is `depth` bits/sample, and adaptive coding gains at
        # most ~3x, so 1..13 brackets every reachable target without sweeping an
        # unreachable alphabet.
        for depth in range(1, 14):
            delta = span / max(1, (1 << (depth - 1)) - 1)
            idx = np.clip(np.rint(x_eval / delta),
                          -(1 << (depth - 1)), (1 << (depth - 1)) - 1).astype(np.int64)
            pay = float(np.mean([code_adaptive(idx[t:t + 1], depth, INPUT_LEN,
                                               verify=False)[0] for t in range(A)]))
            bps = (pay + DELTA_SIDE_BITS / A) / INPUT_LEN
            if best is None or abs(bps - tgt) < abs(best[0] - tgt):
                best = (bps, depth, delta, idx)
        bps, depth, delta, idx = best
        off = 1 << (depth - 1)
        coder = AdaptiveCoder(1 << depth)
        recs = []
        for t in range(A):
            _, dec = coder.encode(idx[t] + off)
            recs.append((coder.decode(dec, INPUT_LEN) - off) * delta)
        rows.append({"method": "Arch3Classical", "K": INPUT_LEN, "depth": depth,
                     "latent_slots": INPUT_LEN,
                     "payload_bits_per_trace": round(bps * INPUT_LEN, 2),
                     "bps_payload": round(bps, 6),
                     "position_floor_bits": 0, "coder_overhead_vs_ideal": "",
                     "side_bits_per_model": DELTA_SIDE_BITS,
                     "bps_total_amortised": round(bps, 6),
                     "compression_vs_source": round(SOURCE_BITS_PER_SAMPLE / bps, 2),
                     "snr_db": round(snr_db(x_eval, np.asarray(recs)), 3),
                     "target_bps": tgt, "abs_err": round(abs(bps - tgt), 6),
                     "cr": "NaN", "cr_implied": 1.0, "coder": "adaptive"})
        print(f"  arch3 target {tgt:6.3f} bps -> depth {depth}  got {bps:.4f} bps  "
              f"SNR {rows[-1]['snr_db']:6.2f} dB")
    return rows


## INSTANCE waveforms (Option A), the 80/10/10 split, and 1c


In [ ]:
"""INSTANCE waveforms in a fixed, P-aligned 1500-sample window.

The study takes ``w[:1500, 2]`` from each STEAD trace and min-max normalises it
inside ``EfficientSeismicDataset``.  This cell keeps that per-trace min-max and
the 1500-sample length, but moves the window so P sits at a fixed offset:

    start = clip(P - P_OFFSET, 0, len(trace) - 1500)
    segment = x[start:start + 1500]

P is REQUIRED.  A trace whose P is missing, non-finite or outside the stored
trace is skipped; the next name in the same split is tried.  S is never used to
include or exclude a trace.  S coverage is measured because it is the only
honest check that the chosen offset actually contains the S arrival; a run with
too little P or S coverage is marked INVALID_WINDOW / PILOT_INVALID_WINDOW.

If the surviving pool is smaller than the study's 2000 calibration traces, the
run does NOT abort: it uses every available training trace (1c), records the
actual count in the manifest and in every CSV, and prints a warning.  The
acceptance record marks such a run INVALID_LOW_COUNT rather than COMPLETE.
"""
import h5py
import pandas as pd
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split


class EfficientSeismicDataset(Dataset):
    """The study's dataset class, verbatim: per-trace min-max normalisation."""

    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        waveform = self.data[idx]
        z_min = np.min(waveform)
        z_max = np.max(waveform)

        if z_max - z_min > 0:
            waveform = (waveform - z_min) / (z_max - z_min)
        else:
            waveform = np.zeros_like(waveform)

        return torch.tensor(waveform, dtype=torch.float32).unsqueeze(0)


def _pick_col(df, candidates, required=True):
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise KeyError(
            f"none of {candidates} found in the INSTANCE metadata; "
            f"columns are {list(df.columns)[:30]}")
    return None


def _numeric(df, col):
    df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def _pick_numeric_col(df, candidates):
    """Prefer the earliest candidate, unless it is sparse/unusable.

    The study uses ``source_magnitude`` and ``source_distance_km``.  INSTANCE
    releases sometimes carry only ``path_ep_distance_km``; if both columns are
    present, the more complete one wins and the candidate order breaks ties.
    That keeps the study column when it is populated without aborting on an
    all-empty optional column.
    """
    best, best_n = None, -1
    for c in candidates:
        if c not in df.columns:
            continue
        vals = pd.to_numeric(df[c], errors="coerce")
        n_ok = int(vals.notna().sum())
        if n_ok > best_n:
            best, best_n = c, n_ok
    if best is None or best_n < 2:
        raise KeyError(
            f"none of {candidates} has at least two numeric values; "
            f"columns are {list(df.columns)[:30]}")
    df[best] = pd.to_numeric(df[best], errors="coerce")
    return best


def _median(values):
    vals = [float(v) for v in values if v is not None and np.isfinite(v)]
    return float(np.median(vals)) if vals else float("nan")


def _orient(w):
    """Return the sample vector, whatever the on-disk orientation is."""
    if w.ndim != 2:
        return None
    if w.shape[0] >= w.shape[1]:
        if w.shape[1] < COMPONENT + 1:
            return None
        return w[:, COMPONENT]
    if w.shape[0] < COMPONENT + 1:
        return None
    return w[COMPONENT, :]


# ---------------------------------------------------------------- metadata
meta = pd.read_csv(INSTANCE_META, keep_default_na=False, low_memory=False)
name_col = _pick_col(meta, ["trace_name"])
mag_col = _pick_numeric_col(meta, ["source_magnitude"])
dist_col = _pick_numeric_col(
    meta, ["source_distance_km", "path_ep_distance_km"])
p_col = _pick_col(meta, ["trace_P_arrival_sample", "trace_p_arrival_sample"],
                  required=False)
s_col = _pick_col(meta, ["trace_S_arrival_sample", "trace_s_arrival_sample"],
                  required=False)
cat_col = _pick_col(meta, ["trace_category"], required=False)

for _c in (p_col, s_col):
    if _c is not None:
        meta = _numeric(meta, _c)
meta[name_col] = meta[name_col].astype(str)
meta = meta.drop_duplicates(subset=[name_col]).reset_index(drop=True)

n0 = len(meta)
if cat_col is not None:
    meta = meta[meta[cat_col] == "earthquake_local"]
n1 = len(meta)
meta = meta[meta[mag_col] > MIN_MAGNITUDE]
n2 = len(meta)
meta = meta[meta[dist_col] <= MAX_DISTANCE_KM]
n3 = len(meta)

# P/S are descriptive.  P is required by the alignment itself; S is never
# used to include or exclude a trace.  The first-1500 numbers are kept so the
# pre-event failure remains visible in the same funnel.
if p_col is not None and s_col is not None:
    ps_available = int(((meta[p_col] > 0) & (meta[s_col] > 0)).sum())
    p_inside = int(((meta[p_col] > 0) & (meta[p_col] < INPUT_LEN)).sum())
    s_inside = int(((meta[s_col] > 0) & (meta[s_col] < INPUT_LEN)).sum())
    ps_inside_both = int((((meta[p_col] > 0) & (meta[p_col] < INPUT_LEN))
                          & ((meta[s_col] > 0) & (meta[s_col] < INPUT_LEN))).sum())
else:
    ps_available = p_inside = s_inside = ps_inside_both = 0

print("[instance] metadata funnel (P-aligned; P required, S not filtered)")
print(f"  rows in file ............................ {n0:8d}")
if cat_col is not None:
    print(f"  trace_category == earthquake_local ...... {n1:8d}")
else:
    print("  trace_category column absent ............ (skipped)")
print(f"  source_magnitude > {MIN_MAGNITUDE:g} ................. {n2:8d}")
print(f"  {dist_col} <= {MAX_DISTANCE_KM:g} km ....... {n3:8d}   <- pool")
if p_col is not None and s_col is not None:
    print(f"  P > 0 and S > 0 (diagnostic) ............ {ps_available:8d}")
    print(f"  P and S inside [0,{INPUT_LEN}) (old pre-event check) "
          f"{ps_inside_both:8d}")
    print(f"  P inside / S inside first 1500 (diagnostic) "
          f"{p_inside:8d} / {s_inside:8d}")
else:
    print("  P/S columns absent .................... (diagnostics skipped)")
if p_col is None:
    raise RuntimeError(
        "the P-aligned window requires a P-arrival column; this metadata file "
        "has none.  Set SEGMENT_MODE back only if you intentionally want the "
        "old first-1500 evidence run.")

if n3 < 2:
    raise RuntimeError(
        "Fewer than two traces survive the INSTANCE metadata filters; a split "
        "is not possible.  Check the metadata columns and thresholds printed "
        "above.  The run is not silently continuing with an empty test set.")

# ---------------------------------------------------------------- 80/10/10 split
names = meta[name_col].tolist()
n = len(names)
test_size = max(1, int(0.1 * n))
if test_size >= n:
    test_size = n - 1
tr_val, test_names = train_test_split(names, test_size=test_size,
                                      random_state=SEED)
val_size = max(0, int(0.1 * n))
train_size = len(tr_val) - val_size
if train_size <= 0 or train_size >= len(tr_val):
    train_names, val_names = list(tr_val), []
else:
    train_names, val_names = train_test_split(tr_val, train_size=train_size,
                                              random_state=SEED)

calib_target = min(N_CALIB, len(train_names))
eval_target = min(N_EVAL, len(test_names))
print(f"[instance] split 80/10/10 (seed {SEED}): "
      f"train {len(train_names)}, val {len(val_names)}, test {len(test_names)}")
print(f"[instance] requested {calib_target} calibration + {eval_target} scored traces")

# ---------------------------------------------------------------- waveforms
p_by_name = (dict(zip(meta[name_col], meta[p_col]))
             if p_col is not None else {})
s_by_name = (dict(zip(meta[name_col], meta[s_col]))
             if s_col is not None else {})


def _arrival(value):
    """Return a finite float arrival or None; never let 'NA' become a crop."""
    try:
        v = float(value)
    except (TypeError, ValueError):
        return None
    return v if np.isfinite(v) else None


def _load_segments(candidates, target, label):
    """Load P-aligned 1500-sample windows, or stop when candidates run out.

    The split is made on metadata names, as before.  A missing/short HDF5 key
    or a missing/invalid P is skipped and the next name in the SAME split is
    tried, so the 2000/200 target is preserved whenever the split can supply it.
    S is measured, never filtered.
    """
    segs, used, metas = [], [], []
    missing = short = examined = 0
    p_skipped = p_missing = p_invalid = 0
    p_vals, s_vals = [], []
    p_rel_vals, s_rel_vals, starts = [], [], []
    p_inside = s_inside = s_missing = 0
    with h5py.File(INSTANCE_HDF5, "r") as f:
        if "data" not in f:
            raise KeyError(
                f"{INSTANCE_HDF5} has no 'data' group; keys are {list(f.keys())[:20]}")
        g = f["data"]
        for nm in candidates:
            if len(segs) >= target:
                break
            examined += 1
            try:
                w = g[nm][:]
            except KeyError:
                missing += 1
                continue
            x = _orient(w)
            if x is None or len(x) < INPUT_LEN:
                short += 1
                continue

            p = _arrival(p_by_name.get(nm))
            if P_REQUIRED:
                if p is None:
                    p_missing += 1
                    p_skipped += 1
                    continue
                if not (0.0 < p < float(len(x))):
                    p_invalid += 1
                    p_skipped += 1
                    continue
                start = int(np.clip(int(round(p)) - P_OFFSET,
                                    0, len(x) - INPUT_LEN))
            else:
                start = 0

            seg = np.asarray(x[start:start + INPUT_LEN], dtype=np.float32)
            if seg.size != INPUT_LEN:
                short += 1
                continue

            s = _arrival(s_by_name.get(nm))
            p_rel = (p - start) if p is not None else float("nan")
            s_rel = (s - start) if s is not None else float("nan")
            p_ok = bool(np.isfinite(p_rel) and 0 <= p_rel < INPUT_LEN)
            s_ok = bool(np.isfinite(s_rel) and 0 <= s_rel < INPUT_LEN)
            if p is not None and np.isfinite(p):
                p_vals.append(float(p))
            if s is not None and np.isfinite(s):
                s_vals.append(float(s))
            else:
                s_missing += 1
            if np.isfinite(p_rel):
                p_rel_vals.append(float(p_rel))
                p_inside += int(p_ok)
            if np.isfinite(s_rel):
                s_rel_vals.append(float(s_rel))
                s_inside += int(s_ok)
            starts.append(int(start))

            segs.append(seg)
            used.append(nm)
            metas.append({
                "name": nm, "start": int(start),
                "p_abs": (float(p) if p is not None else float("nan")),
                "s_abs": (float(s) if s is not None else float("nan")),
                "p_rel": float(p_rel), "s_rel": float(s_rel),
                "p_inside": bool(p_ok), "s_inside": bool(s_ok),
            })

    p_denom = len(segs) + p_skipped
    p_cov = 100.0 * len(segs) / p_denom if p_denom else 0.0
    s_cov = 100.0 * s_inside / len(segs) if segs else 0.0

    print(f"[instance] {label:11s}: target {target:5d}, examined {examined:6d}, "
          f"loaded {len(segs):5d} (missing key {missing}, shorter than "
          f"{INPUT_LEN} {short}, P skipped {p_skipped}; "
          f"P coverage {p_cov:.3f}%, S coverage {s_cov:.3f}%)")
    stats = {
        "requested": target, "examined": examined, "loaded": len(segs),
        "missing_key": missing, "short": short,
        "p_skipped": p_skipped, "p_missing": p_missing,
        "p_invalid": p_invalid,
        "p_inside": p_inside, "s_inside": s_inside,
        "s_missing": s_missing,
        "p_coverage_pct": round(p_cov, 4),
        "s_coverage_pct": round(s_cov, 4),
        "p_median": _median(p_vals), "s_median": _median(s_vals),
        "p_rel_median": _median(p_rel_vals),
        "s_rel_median": _median(s_rel_vals),
        "start_median": _median(starts),
    }
    return segs, used, metas, stats


calib_segs, calib_used, calib_trace_meta, calib_stats = _load_segments(
    train_names, calib_target, "calibration")
eval_segs, eval_used, eval_trace_meta, eval_stats = _load_segments(
    test_names, eval_target, "scored")
overlap = set(calib_used) & set(eval_used)
if overlap:
    raise RuntimeError(f"calibration and scored sets overlap on {len(overlap)} traces")

if not calib_segs or not eval_segs:
    raise RuntimeError(
        f"calibration {len(calib_segs)}, scored {len(eval_segs)}.  A run with an "
        "empty side is not meaningful; the counts and the drops are printed above.")

# The study's N_CALIB / N_EVAL are the *actual* sizes of these two sets.  Every
# measurement function reads these globals, so a reduced pool (1c) is used
# consistently by arch 1, arch 2, the baselines, the wavelet arm and arch 3.
N_CALIB = len(calib_segs)
N_EVAL = len(eval_segs)

# Coverage is computed over the traces actually selected for each side.  P is
# required, so p_coverage_pct is normally 100; it can only fall when names had
# to be skipped for a missing/invalid P.  S is the empirical check that the
# fixed offset actually contains the S arrival.
WINDOW_STATS = {
    "segment_mode": SEGMENT_MODE,
    "p_offset": P_OFFSET,
    "p_required": bool(P_REQUIRED),
    "p_coverage_min_pct": P_COVERAGE_MIN,
    "s_coverage_min_pct": S_COVERAGE_MIN,
    "calib": calib_stats,
    "eval": eval_stats,
    "p_coverage_min_observed_pct": round(min(
        calib_stats["p_coverage_pct"], eval_stats["p_coverage_pct"]), 4),
    "s_coverage_min_observed_pct": round(min(
        calib_stats["s_coverage_pct"], eval_stats["s_coverage_pct"]), 4),
}
WINDOW_OK = (
    WINDOW_STATS["p_coverage_min_observed_pct"] >= P_COVERAGE_MIN
    and WINDOW_STATS["s_coverage_min_observed_pct"] >= S_COVERAGE_MIN)
WINDOW_STATS["window_ok"] = bool(WINDOW_OK)
WINDOW_STATS["p_coverage_ok"] = bool(
    WINDOW_STATS["p_coverage_min_observed_pct"] >= P_COVERAGE_MIN)
WINDOW_STATS["s_coverage_ok"] = bool(
    WINDOW_STATS["s_coverage_min_observed_pct"] >= S_COVERAGE_MIN)

print(f"[instance] P-offset window: start = clip(P-{P_OFFSET}, 0, len-{INPUT_LEN})")
print(f"[instance] coverage  calib P {calib_stats['p_coverage_pct']:.3f}% / "
      f"S {calib_stats['s_coverage_pct']:.3f}%   scored P "
      f"{eval_stats['p_coverage_pct']:.3f}% / S {eval_stats['s_coverage_pct']:.3f}%")
print(f"[instance] window gate {'PASS' if WINDOW_OK else 'FAIL'}  "
      f"(P >= {P_COVERAGE_MIN:g}%, S >= {S_COVERAGE_MIN:g}%)")
if not WINDOW_OK:
    print("!" * 78)
    print("WARNING: the P/S coverage gate failed.  Measurement will continue so")
    print("the diagnostics and figures exist, but acceptance will be")
    print("INVALID_WINDOW (full) or PILOT_INVALID_WINDOW (pilot), never COMPLETE.")
    print("!" * 78)

if N_CALIB < N_CALIB_TARGET or N_EVAL < N_EVAL_TARGET:
    print("!" * 78)
    print("WARNING (1c): the surviving pool is smaller than the study protocol.")
    print(f"  calibration {N_CALIB} / {N_CALIB_TARGET} requested")
    print(f"  scored      {N_EVAL} / {N_EVAL_TARGET} requested")
    print("  The run continues with the available traces.  The actual counts are")
    print("  written to every CSV and to run_manifest.json; no protocol is changed")
    print("  silently and no trace is shifted or padded.  Acceptance will be")
    print(f"  INVALID_LOW_COUNT unless calibration >= {MIN_CALIB_VALID} and "
          f"scored >= {MIN_EVAL_VALID}.")
    print("!" * 78)

calib_ds = EfficientSeismicDataset(calib_segs)
eval_ds = EfficientSeismicDataset(eval_segs)
eval_np = np.stack([eval_ds[i].numpy() for i in range(len(eval_ds))]).astype(np.float32)
calib_probe = np.stack([calib_ds[i].numpy() for i in range(min(64, len(calib_ds)))])

for _label, _arr in (("calibration", calib_probe), ("scored", eval_np)):
    if not np.isfinite(_arr).all():
        raise RuntimeError(f"{_label} contains non-finite samples")
    if _arr.min() < -1e-6 or _arr.max() > 1 + 1e-6:
        raise RuntimeError(
            f"{_label} is outside [0,1] after min-max: "
            f"{_arr.min():.6g} .. {_arr.max():.6g}")

print(f"[instance] calibration {calib_ds[0].shape}  "
      f"min {calib_probe.min():.4f}  max {calib_probe.max():.4f}  n={len(calib_ds)}")
print(f"[instance] scored      {eval_np[0].shape}  "
      f"min {eval_np.min():.4f}  max {eval_np.max():.4f}  n={len(eval_np)}")
if eval_trace_meta:
    for _tag, _meta in (("calib", calib_trace_meta), ("scored", eval_trace_meta)):
        for _m in _meta[:3]:
            print(f"  {_tag:6s} {_m['name'][:40]:40s} "
                  f"start={_m['start']:3d} P={_m['p_abs']:7.1f} "
                  f"(rel {_m['p_rel']:6.1f})  S={_m['s_abs']:7.1f} "
                  f"(rel {_m['s_rel']:6.1f}, inside={_m['s_inside']})")

# A protocol hash makes stale CSVs from a different data definition unusable.
# It includes the exact split, the counts, and the input-definition constants.
PROTOCOL = {
    "validation_version": VALIDATION_VERSION,
    "segment_mode": SEGMENT_MODE,
    "p_offset": P_OFFSET,
    "p_required": bool(P_REQUIRED),
    "p_s_filter": False,
    "p_coverage_min_pct": P_COVERAGE_MIN,
    "s_coverage_min_pct": S_COVERAGE_MIN,
    "component": COMPONENT,
    "input_len": INPUT_LEN,
    "seed": SEED,
    "min_magnitude": MIN_MAGNITUDE,
    "max_distance_km": MAX_DISTANCE_KM,
    "magnitude_column": mag_col,
    "distance_column": dist_col,
    "category_column": cat_col,
    "n_calib_target": N_CALIB_TARGET,
    "n_eval_target": N_EVAL_TARGET,
    "n_calib_used": N_CALIB,
    "n_eval_used": N_EVAL,
    "quick": bool(QUICK),
    "widths": WIDTHS,
    "crs": CRS,
    "lambdas": LAMBDAS,
    "depths": DEPTHS,
    "wavelets": WAVELETS,
    "wave_targets_bps": WAVE_TARGETS_BPS,
    "arch3_targets_bps": ARCH3_TARGETS_BPS,
    "legacy_wavelet_crs": LEGACY_WAVELET_CRS,
    "instance_meta": INSTANCE_META,
    "instance_hdf5": INSTANCE_HDF5,
    "instance_meta_size": os.path.getsize(INSTANCE_META),
    "instance_hdf5_size": os.path.getsize(INSTANCE_HDF5),
    "ckpt_dir": CKPT_DIR,
    "calib_names_sha16": hashlib.sha256(
        "\n".join(calib_used).encode()).hexdigest()[:16],
    "eval_names_sha16": hashlib.sha256(
        "\n".join(eval_used).encode()).hexdigest()[:16],
}
PROTOCOL_HASH = hashlib.sha256(
    json.dumps(PROTOCOL, sort_keys=True, separators=(",", ":")).encode()
).hexdigest()[:16]
print(f"[instance] protocol {PROTOCOL_HASH}  ({SEGMENT_MODE})")

MANIFEST = {
    "generated": time.strftime("%Y-%m-%d %H:%M:%S"),
    "instance_meta": INSTANCE_META,
    "instance_hdf5": INSTANCE_HDF5,
    "instance_meta_size": PROTOCOL["instance_meta_size"],
    "instance_hdf5_size": PROTOCOL["instance_hdf5_size"],
    "segment_mode": SEGMENT_MODE,
    "validation_version": VALIDATION_VERSION,
    "pilot": bool(PILOT),
    "pilot_specs": PILOT_SPECS,
    "p_offset": P_OFFSET,
    "p_required": bool(P_REQUIRED),
    "protocol_hash": PROTOCOL_HASH,
    "protocol": PROTOCOL,
    "component": COMPONENT,
    "input_len": INPUT_LEN,
    "metadata_rows": n0,
    "after_category": n1,
    "after_magnitude": n2,
    "after_distance": n3,
    "after_ps_available": ps_available,
    "after_ps_inside_both": ps_inside_both,
    "p_inside_first1500_pct_of_pool": round(100.0 * p_inside / max(1, n3), 4),
    "s_inside_first1500_pct_of_pool": round(100.0 * s_inside / max(1, n3), 4),
    "window_stats": WINDOW_STATS,
    "window_ok": bool(WINDOW_OK),
    "p_coverage_min_observed_pct": WINDOW_STATS["p_coverage_min_observed_pct"],
    "s_coverage_min_observed_pct": WINDOW_STATS["s_coverage_min_observed_pct"],
    "legacy_wavelet_crs": LEGACY_WAVELET_CRS,
    "acceptance_file": ACCEPTANCE_FILE,
    "n_train": len(train_names),
    "n_val": len(val_names),
    "n_test": len(test_names),
    "n_calib_target": N_CALIB_TARGET,
    "n_eval_target": N_EVAL_TARGET,
    "min_calib_valid": MIN_CALIB_VALID,
    "min_eval_valid": MIN_EVAL_VALID,
    "n_calib_used": len(calib_segs),
    "n_eval_used": len(eval_segs),
    "calib_stats": calib_stats,
    "eval_stats": eval_stats,
    "calib_examined": calib_stats["examined"],
    "eval_examined": eval_stats["examined"],
    "calib_missing_key": calib_stats["missing_key"],
    "eval_missing_key": eval_stats["missing_key"],
    "magnitude_column": mag_col,
    "distance_column": dist_col,
    "category_column": cat_col,
    "split": "80/10/10",
    "split_seed": SEED,
    "min_magnitude": MIN_MAGNITUDE,
    "max_distance_km": MAX_DISTANCE_KM,
    "calib_min_max": [float(calib_probe.min()), float(calib_probe.max())],
    "eval_min_max": [float(eval_np.min()), float(eval_np.max())],
}
with open(os.path.join(OUT_DIR, "run_manifest.json"), "w", encoding="utf-8") as _fh:
    json.dump(MANIFEST, _fh, indent=1)
print(f"[instance] manifest {os.path.join(OUT_DIR, 'run_manifest.json')}")


## Checkpoint inventory and strict-load self-test


In [ ]:
"""Checkpoint inventory and strict-load self-test.

The 110 checkpoints are the study's grid, not a glob: 3 learned series x 10
widths x 3 lambdas = 90, plus 2 baselines x 10 widths = 20.  Each file is
rebuilt with the class the study built it with and loaded with ``strict=True``,
so a checkpoint trained by a different architecture cannot pass silently.

A missing file is reported and skipped; the run continues with the files that
exist.  The summary at the end reports ``found / expected``.
"""
import traceback


def _lam_tag(lam):
    return "na" if lam is None else f"{lam:g}".replace("+", "").replace(
        ".", "p").replace("e-", "em").replace("-", "m")


def expected_checkpoints():
    items = []
    learned = list(ARCH2_SERIES) + (list(ARCH1_SERIES) if DO_ARCH1 else [])
    for series in learned:
        for d in WIDTHS:
            cr = max(2, int(round(INPUT_LEN / d)))
            for lam in LAMBDAS:
                tag = f"{series}_d{d}_lam{_lam_tag(lam)}"
                items.append({
                    "series": series, "cr": cr, "d": d, "lam": lam,
                    "arch": 1 if series in ARCH1_SERIES else 2,
                    "path": os.path.join(CKPT_DIR, f"best_{tag}.pth"),
                    "rate": os.path.join(RATE_DIR, f"{tag}.csv"),
                    "tag": tag,
                })
    for name in BASELINE_NAMES:
        for cr in CRS:
            tag = f"{name}_cr{cr}"
            d = int(round(INPUT_LEN / cr))
            items.append({
                "series": name, "cr": cr, "d": d, "lam": 0.0,
                "arch": 2,
                "path": os.path.join(CKPT_DIR, f"best_{tag}.pth"),
                "rate": os.path.join(RATE_DIR, f"baseline_{tag}.csv"),
                "tag": tag,
            })
    return items


EXPECTED_FULL = expected_checkpoints()


def _is_pilot_item(it):
    for series, d, lam in PILOT_SPECS:
        if (it["series"] == series and int(it["d"]) == int(d)
                and it["lam"] is not None
                and abs(float(it["lam"]) - float(lam)) < 1e-15):
            return True
    return False


EXPECTED = ([it for it in EXPECTED_FULL if _is_pilot_item(it)]
            if PILOT else list(EXPECTED_FULL))
RUN_EXPECTED = list(EXPECTED)      # the six pilot points or the full 110
FOUND = [it for it in EXPECTED if os.path.isfile(it["path"])]
MISSING = [it for it in EXPECTED if not os.path.isfile(it["path"])]
if PILOT:
    print(f"[inventory] PILOT run: {len(EXPECTED)} of {len(EXPECTED_FULL)} "
          "checkpoints (the six headline points, lambda=1e-4)")
    for _it in EXPECTED:
        print(f"  pilot {_it['series']:24s} d={_it['d']:4d} "
              f"CR={_it['cr']:3d} lam={_it['lam']:g}")

print(f"[inventory] expected {len(EXPECTED)} checkpoints "
      f"({len(EXPECTED_FULL)} in the full study grid)")
for _s in (ARCH2_SERIES + (ARCH1_SERIES if DO_ARCH1 else []) + BASELINE_NAMES):
    _e = sum(1 for it in EXPECTED if it["series"] == _s)
    _f = sum(1 for it in FOUND if it["series"] == _s)
    print(f"  {_s:24s} {_f:3d} / {_e:3d}")
if MISSING:
    print(f"[inventory] WARNING: {len(MISSING)} expected files are missing; "
          "the run continues with the rest")
    for it in MISSING[:20]:
        print(f"  missing {os.path.basename(it['path'])}")
    if len(MISSING) > 20:
        print(f"  ... and {len(MISSING) - 20} more")

# Files that are present but not on the study grid.  They are not measured:
# the grid, not the folder, defines this validation.
_expected_paths = {os.path.abspath(it["path"]) for it in EXPECTED_FULL}
_extras = [p for p in glob.glob(os.path.join(CKPT_DIR, "best_*.pth"))
           if os.path.abspath(p) not in _expected_paths]
if _extras:
    print(f"[inventory] {len(_extras)} checkpoint(s) present but off the study grid; "
          "ignored (the grid defines the comparison)")
    for p in sorted(_extras)[:10]:
        print(f"  extra {os.path.basename(p)}")

# ---------------------------------------------------------------- strict load
CKPTS = []
FAILED = []
print("[inventory] strict-loading every checkpoint ...")
for i, it in enumerate(FOUND, 1):
    try:
        blob = torch.load(it["path"], map_location="cpu", weights_only=False)
        meta = blob.get("meta") or {}
        if "state_dict" not in blob:
            raise KeyError("checkpoint has no state_dict")

        # meta <-> filename agreement.  A mismatch means the checkpoint is not
        # the grid point the filename claims, which would silently mislabel a
        # point in every figure.
        if int(meta.get("cr", -1)) != int(it["cr"]):
            raise ValueError(f"meta cr {meta.get('cr')} != filename cr {it['cr']}")
        if it["series"] not in BASELINE_NAMES:
            if int(meta.get("latent_dim", -1)) != int(it["d"]):
                raise ValueError(
                    f"meta latent_dim {meta.get('latent_dim')} != d {it['d']}")
            if meta.get("lam") is None or abs(float(meta["lam"]) - it["lam"]) > 1e-15:
                raise ValueError(f"meta lam {meta.get('lam')} != {it['lam']}")
            if int(meta.get("arch", -1)) != int(it["arch"]):
                raise ValueError(f"meta arch {meta.get('arch')} != {it['arch']}")
        if it["series"] in BASELINE_NAMES and meta.get("arch") not in (None, 2):
            raise ValueError(f"baseline meta arch {meta.get('arch')} is not 2/None")

        model = build_model(it["series"], it["cr"])
        model.load_state_dict(blob["state_dict"], strict=True)
        model.eval()
        # The model is deliberately built on CPU.  Keep the strict-load smoke
        # input on CPU too: on a CUDA host, a DEVICE input would make every
        # checkpoint fail before the measurement (the measurement functions
        # move their own models to DEVICE).  This check is about architecture /
        # state-dict compatibility, not throughput.
        with torch.no_grad():
            _y = model(torch.rand(1, 1, INPUT_LEN))
        if tuple(_y.shape) != (1, 1, INPUT_LEN):
            raise ValueError(f"forward returned {tuple(_y.shape)}")

        it = dict(it)
        it["meta"] = meta
        it["params"] = sum(p.numel() for p in model.parameters())
        it["model_bytes"] = (
            sum(p.numel() * p.element_size() for p in model.parameters())
            + sum(b.numel() * b.element_size() for b in model.buffers()))
        it["model_size_mb"] = it["model_bytes"] / (1024 ** 2)
        it["epoch"] = blob.get("epoch")
        it["val_bps"] = blob.get("val_bps")
        CKPTS.append(it)
        del model
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
        if i % 10 == 0 or i == len(FOUND):
            print(f"  [{i:3d}/{len(FOUND)}] {it['series']:24s} "
                  f"CR={it['cr']:3d} d={it['d']:4d} lam={it['lam']:g}  "
                  f"params={it['params']:,}")
    except Exception as e:
        FAILED.append((it, repr(e)))
        print(f"  [FAIL] {os.path.basename(it['path'])}: {e}")
        traceback.print_exc(limit=1)

print(f"[inventory] loaded {len(CKPTS)} / {len(EXPECTED)} expected "
      f"({len(EXPECTED_FULL)} full grid; {len(FAILED)} failures, "
      f"{len(MISSING)} missing)")
if FAILED:
    print("[inventory] failures are omitted from the measurement; the run continues")
if not CKPTS:
    raise RuntimeError("no checkpoint could be loaded; nothing to measure")


## Measurement driver (resumable at the CSV level)


In [ ]:
"""Measurement driver: checkpoints, bit-accurate wavelet, legacy wavelet, arch 3.

Every measurement function is the study's.  The only differences are the data
(INSTANCE, fixed P-aligned window) and the calibration set (drawn from the
INSTANCE training split, not from ``meta.calib``).  ``set_calibrated_delta`` is
called per depth on those latents, exactly as the study did.

Two wavelet arms are written, deliberately:

* ``wavelet_equal_rate`` is the bit-accurate codec (positions, signs and
  quantised magnitudes are entropy coded).  It belongs on the RD and
  measured-CR figures.
* ``legacy_wavelet`` reproduces ``Final_LARA_27092026``'s top-K retention at
  the 10 nominal CRs.  It has no bitstream and therefore no measured bps.  It is
  plotted only on ``snr_vs_cr`` and never replaces the bit-accurate arm.

The cell defines the functions; the final cell calls them.  This keeps the
top-to-bottom ``.py`` export working: figures and summary are defined before
the driver runs.
"""
import time
import traceback


def _csv_has_current_protocol(path):
    """True only when every row exists and carries THIS protocol hash."""
    if not os.path.isfile(path):
        return False
    try:
        with open(path, newline="", encoding="utf-8") as fh:
            rows = list(csv.DictReader(fh))
        return bool(rows) and all(
            r.get("protocol_hash") == PROTOCOL_HASH for r in rows)
    except Exception:
        return False


def _csv_row_count(path):
    try:
        with open(path, newline="", encoding="utf-8") as fh:
            return len(list(csv.DictReader(fh)))
    except Exception:
        return -1


def _done(path, expected_rows=None):
    """Resume only when the CSV is this protocol and (optionally) complete."""
    if FORCE or not _csv_has_current_protocol(path):
        return False
    if expected_rows is None:
        return True
    return _csv_row_count(path) == int(expected_rows)


def _expected_checkpoint_rows(it):
    """Rows written by the study measurement for one checkpoint."""
    if it["series"] in BASELINE_NAMES:
        return len(DEPTHS) * 2          # fixed + adaptive per depth
    if it["series"] in ARCH1_SERIES:
        return len(DEPTHS)              # one top-k row per depth
    return len(DEPTHS) * 3              # fixed + adaptive + learned per depth


def _calib_latents(model, train_ds, device):
    n = min(N_CALIB, len(train_ds))
    acc = []
    with torch.no_grad():
        for i in range(0, n, 64):
            xb = torch.stack([train_ds[j] for j in range(i, min(i + 64, n))]).to(device)
            acc.append(model.encode_coarse(xb).reshape(-1).cpu())
    return torch.cat(acc)


def _eta(done, total, t0):
    if done <= 0:
        return "?"
    per = (time.time() - t0) / done
    return f"{per * max(0, total - done) / 60.0:.1f} min"


STAMP = {
    "protocol_hash": PROTOCOL_HASH,
    "validation_version": VALIDATION_VERSION,
    "segment_mode": SEGMENT_MODE,
    "p_offset": P_OFFSET,
    "p_required": bool(P_REQUIRED),
    "window_ok": bool(WINDOW_OK),
    "component": COMPONENT,
    "input_len": INPUT_LEN,
    "split_seed": SEED,
    "magnitude_column": PROTOCOL["magnitude_column"],
    "distance_column": PROTOCOL["distance_column"],
    "n_calib_used": N_CALIB,
    "n_eval_used": N_EVAL,
}


def _stamp_csv(path, extra=None):
    """Add the INSTANCE protocol fields to an existing CSV.

    Arch 1, arch 3 and the wavelet arm write their own rows; this makes every
    CSV in the run carry the same data-definition fields as the learned arms.
    """
    if not os.path.isfile(path):
        return
    with open(path, newline="", encoding="utf-8") as fh:
        rows = list(csv.DictReader(fh))
    if not rows:
        return
    for r in rows:
        r.update(STAMP)
        if extra:
            r.update(extra)
    write_rows(path, rows)


def measure_learned(it):
    """Measure one LARA-Q2 / LARA-Q1 / Light_HARA-Q1 checkpoint."""
    blob = torch.load(it["path"], map_location=DEVICE, weights_only=False)
    meta = blob.get("meta") or {}
    if meta.get("arch") == 1:
        rows = measure_topk(it["path"],
                            lambda c: build_model(it["series"], c),
                            eval_np, calib_ds, it["series"], DEPTHS, DEVICE)
        write_rows(it["rate"], rows)
        _stamp_csv(it["rate"], {"series": it["series"], "lam": it["lam"],
                                "latent_dim": it["d"], "arch": 1,
                                "trained_here": False,
                                "trained_by": "LARA_Q_12102026"})
        return rows

    model = build_model(it["series"], it["cr"]).to(DEVICE)
    model.load_state_dict(blob["state_dict"], strict=True)
    model.eval()
    z_calib = _calib_latents(model, calib_ds, DEVICE)
    x_eval = eval_np[:min(N_EVAL, len(eval_np))]
    with torch.no_grad():
        z_eval = model.encode_coarse(
            torch.as_tensor(x_eval, dtype=torch.float32, device=DEVICE))
    prior_params = (sum(p.numel() for p in model.prior.parameters())
                    if getattr(model, "prior", None) is not None else 0)
    side = DELTA_SIDE_BITS + prior_params * PRIOR_BITS
    prov = {"method": it["series"], "model": it["series"], "series": it["series"],
            "arch": 2, "cr": it["cr"], "cr_implied": INPUT_LEN / it["d"],
            "latent_dim": it["d"], "lam": it["lam"],
            "trained_here": False, "trained_by": "LARA_Q_12102026"}
    rows = evaluate(model, z_eval, x_eval, z_calib, len(x_eval), side, DEPTHS,
                    verify=True, extra=prov, eval_device=DEVICE)
    write_rows(it["rate"], rows)
    _stamp_csv(it["rate"])
    return rows


def measure_baseline_item(it):
    rows = measure_baseline(it["path"], it["series"], it["cr"],
                            calib_ds, eval_np, depths=DEPTHS, device=DEVICE)
    for r in rows:
        r["trained_here"] = False
        r["trained_by"] = "LARA_Q_12102026"
    write_rows(it["rate"], rows)
    _stamp_csv(it["rate"])
    return rows


def _read_csv(path):
    with open(path, newline="", encoding="utf-8") as fh:
        return list(csv.DictReader(fh))


def _fnum(v, default=None):
    try:
        f = float(v)
        return f if np.isfinite(f) else default
    except (TypeError, ValueError):
        return default


def wavelet_equal_rate(wavelet, train_ds, test_data):
    """The study's equal-rate wavelet arm, on the INSTANCE split."""
    cal = np.stack([train_ds[i].numpy().reshape(-1)
                    for i in range(min(64, len(train_ds)))])
    n_ev = min(N_EVAL, len(test_data))
    test = np.asarray(test_data[:n_ev])[:, 0, :]
    test_pairs = [flatten(dwt(s, wavelet)) for s in test]
    cal_pairs = [flatten(dwt(s, wavelet)) for s in cal]
    global SHAPES
    band_max = fit_band_max(cal_pairs)
    nb = band_max.size
    SHAPES = [c.shape for c in dwt(test[0], wavelet)]
    p = os.path.join(WAV_DIR, f"wavelet_{wavelet}_equal_rate.csv")
    if _done(p, len(WAVE_TARGETS_BPS)):
        try:
            have = [_fnum(r.get("target_bps")) for r in _read_csv(p)]
        except Exception:
            have = []
        if len(have) == len(WAVE_TARGETS_BPS) and all(
                a is not None and abs(a - b) < 1e-9
                for a, b in zip(have, WAVE_TARGETS_BPS)):
            print(f"  [skip] {os.path.basename(p)} matches this protocol "
                  f"({PROTOCOL_HASH}) and covers all targets")
            return []
    rows = []
    for tgt in WAVE_TARGETS_BPS:
        row = search(wavelet, test_pairs, test, tgt, [4, 6, 8], band_max, nb,
                     verify=True)
        rows.append(row)
        print(f"  {wavelet:5s} target {tgt:6.3f} bps -> K={row['K']:4d} "
              f"depth={row['depth']}  got {row['bps_total_amortised']:.4f} bps  "
              f"SNR {row['snr_db']:7.2f} dB  pos-overhead "
              f"{row['coder_overhead_vs_ideal']}x")
    write_rows(p, rows)
    return rows


def legacy_wavelet_label(wavelet):
    return {"db4": "Wavelet_DB4", "sym8": "Wavelet_SYM8",
            "coif3": "Wavelet_COIF3"}.get(wavelet, f"Wavelet_{wavelet.upper()}")


def legacy_wavelet(wavelet, test_data, crs=None):
    """Final_LARA top-K retention at nominal CR, on the current eval windows.

    This is intentionally the same operation as ``WaveletCompressor`` in
    ``Final_LARA_27092026``: level-6 DWT, retain coefficients with
    ``abs(c) > sorted_abs[K]`` where ``K = len(signal) // CR``, reconstruct and
    truncate/pad to the input length.  It does NOT encode positions, signs or
    quantised magnitudes, so it has no measured bps and is not a codec.  Its
    only output is SNR at the nominal CR.
    """
    crs = list(LEGACY_WAVELET_CRS if crs is None else crs)
    x_eval = np.asarray(test_data, dtype=np.float64)[:, 0, :]
    n = len(x_eval)
    if n == 0:
        raise RuntimeError("no evaluation traces for the legacy wavelet arm")
    p = os.path.join(LEGACY_DIR, f"wavelet_{wavelet}_legacy_cr.csv")
    if _done(p, len(crs)):
        try:
            have = [_fnum(r.get("cr")) for r in _read_csv(p)]
        except Exception:
            have = []
        if (len(have) == len(crs) and all(
                a is not None and abs(a - b) < 1e-9
                for a, b in zip(have, crs))):
            print(f"  [skip] {os.path.basename(p)} matches this protocol "
                  f"({PROTOCOL_HASH}) and covers all nominal CRs")
            return []

    rows = []
    for cr in crs:
        target = max(1, INPUT_LEN // int(cr))
        recs, kept = [], []
        for signal in x_eval:
            coeffs = pywt.wavedec(signal, wavelet, level=LEVEL)
            all_coeffs = np.concatenate([c.flatten() for c in coeffs])
            sorted_coeffs = np.sort(np.abs(all_coeffs))[::-1]
            threshold = (sorted_coeffs[target]
                         if len(sorted_coeffs) > target else 0.0)
            compressed = [c * (np.abs(c) > threshold) for c in coeffs]
            kept.append(int(sum(np.count_nonzero(c) for c in compressed)))
            y = pywt.waverec(compressed, wavelet)
            if len(y) > INPUT_LEN:
                y = y[:INPUT_LEN]
            elif len(y) < INPUT_LEN:
                y = np.pad(y, (0, INPUT_LEN - len(y)))
            recs.append(y)
        mean_kept = float(np.mean(kept)) if kept else float("nan")
        rows.append({
            "series": legacy_wavelet_label(wavelet),
            "method": legacy_wavelet_label(wavelet),
            "wavelet": wavelet,
            "cr": int(cr),
            "nominal_cr": int(cr),
            "K": int(target),
            "target_coeffs": int(target),
            "mean_kept": round(mean_kept, 3),
            "effective_retention_cr": (round(INPUT_LEN / mean_kept, 3)
                                       if mean_kept > 0 else ""),
            "snr_db": round(snr_db(x_eval, np.asarray(recs)), 3),
            "n_eval": int(n),
            "coder": "legacy-topk-retention",
            "cost_model": "no_bitstream_positions_signs_quantization_free",
            "bps_payload": "",
            "bps_total_amortised": "",
        })
        print(f"  {wavelet:5s} nominal CR {cr:3d} -> K={target:4d}  "
              f"kept {mean_kept:7.1f}  SNR {rows[-1]['snr_db']:7.2f} dB")
    write_rows(p, rows)
    _stamp_csv(p)
    return rows


MEASURE_FAILED = []


def run_measurements():
    t0 = time.time()
    MEASURE_FAILED.clear()
    print("\n### 1. checkpoints")
    print(f"  {len(CKPTS)} loaded / {len(EXPECTED)} expected for this run "
          f"({len(EXPECTED_FULL)} full grid); "
          f"{len(MISSING)} missing; {len(FAILED)} load failures")
    for i, it in enumerate(CKPTS, 1):
        if _done(it["rate"], _expected_checkpoint_rows(it)):
            print(f"  [skip] {os.path.basename(it['rate'])} matches this "
                  f"protocol ({PROTOCOL_HASH}) and is complete")
            continue
        try:
            if it["series"] in BASELINE_NAMES:
                rows = measure_baseline_item(it)
            else:
                rows = measure_learned(it)
            print(f"  [{i:3d}/{len(CKPTS)}] {it['series']:24s} "
                  f"CR={it['cr']:3d} lam={it['lam']:g}  {len(rows)} rows  "
                  f"elapsed {(time.time() - t0) / 60:.1f} min  "
                  f"ETA {_eta(i, len(CKPTS), t0)}")
        except Exception as e:
            MEASURE_FAILED.append((it, repr(e)))
            print(f"  [FAIL] {it['series']} CR={it['cr']} lam={it['lam']:g}: {e}")
            traceback.print_exc(limit=2)

    print("\n### 2. wavelet equal-rate arm")
    for w in WAVELETS:
        try:
            wavelet_equal_rate(w, calib_ds, eval_np)
            _stamp_csv(os.path.join(WAV_DIR, f"wavelet_{w}_equal_rate.csv"))
        except Exception as e:
            MEASURE_FAILED.append(({"series": f"wavelet:{w}"}, repr(e)))
            print(f"  [FAIL] wavelet {w}: {e}")
            traceback.print_exc(limit=2)

    print("\n### 3. legacy wavelet (Final_LARA top-K retention, nominal CR)")
    if DO_LEGACY_WAVELET:
        for w in WAVELETS:
            try:
                legacy_wavelet(w, eval_np)
            except Exception as e:
                MEASURE_FAILED.append(({"series": f"legacy_wavelet:{w}"}, repr(e)))
                print(f"  [FAIL] legacy wavelet {w}: {e}")
                traceback.print_exc(limit=2)
    else:
        print("  disabled")

    print("\n### 4. architecture 3 (classical)")
    if DO_ARCH3:
        p = os.path.join(ARCH3_DIR, "arch3_rates.csv")
        ready = False
        if _done(p, len(ARCH3_TARGETS_BPS)):
            try:
                have = [_fnum(r.get("target_bps")) for r in _read_csv(p)]
            except Exception:
                have = []
            ready = (len(have) == len(ARCH3_TARGETS_BPS) and all(
                a is not None and abs(a - b) < 1e-9
                for a, b in zip(have, ARCH3_TARGETS_BPS)))
        if ready:
            print(f"  [skip] {os.path.basename(p)} matches this protocol "
                  f"({PROTOCOL_HASH}) and covers all targets")
        else:
            try:
                rows = measure_arch3(eval_np, calib_ds, targets=ARCH3_TARGETS_BPS,
                                     device=DEVICE)
                write_rows(p, rows)
                _stamp_csv(p)
            except Exception as e:
                MEASURE_FAILED.append(({"series": "Arch3Classical"}, repr(e)))
                print(f"  [FAIL] arch 3: {e}")
                traceback.print_exc(limit=2)

    print(f"\n### measurement done in {(time.time() - t0) / 60:.1f} min")
    print(f"  failures: {len(MEASURE_FAILED)}")
    for it, err in MEASURE_FAILED:
        print(f"    {it.get('series', it.get('name', '?'))}: {err}")
    return MEASURE_FAILED


## Rate-distortion, SNR, and pipeline figures


In [ ]:
"""Block diagram and the two result figures.

The block diagram's BPS column is read straight from the sweep CSVs, so the
figure cannot drift the way a hand-maintained placeholder can.
"""
import glob
import csv
import re

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

INPUT_LEN = 1500
SOURCE_BPS = 32.0

STYLE = {
    "LARA-Q2":        dict(color="#C2410C", marker="o", zorder=6),
    "LARA-Q1":        dict(color="#B45309", marker="X", zorder=5),
    "Light_HARA-Q1":  dict(color="#7C2D12", marker="P", zorder=5),
    "Generalized AE": dict(color="#0F766E", marker="^"),
    "AE_PureConcat":  dict(color="#15803D", marker="v"),
    "Arch3Classical": dict(color="#4B5563", marker="X"),
    "Wavelet_DB4":    dict(color="#9D174D", marker="D"),
    "Wavelet_SYM8":   dict(color="#B45309", marker="D"),
    "Wavelet_COIF3":  dict(color="#5B21B6", marker="P"),
}
MODEL_LABEL = {"GeneralizedAutoencoder": "Generalized AE",
               "AE_PureConcat": "AE_PureConcat"}
ARCH_NOTE = ("architecture 1 = top-k selection + positions   |   "
             "2 = learned projection   |   3 = no transform")


def read_csv(path, current_only=True):
    with open(path, newline="", encoding="utf-8") as fh:
        rows = list(csv.DictReader(fh))
    if current_only:
        rows = [r for r in rows if r.get("protocol_hash") == PROTOCOL_HASH]
    return rows


def fnum(v, default=None):
    try:
        return float(v)
    except (TypeError, ValueError):
        return default


def glob_rows(directory, pattern):
    out = []
    for path in sorted(glob.glob(os.path.join(directory, pattern))):
        out.extend(read_csv(path))
    return [r for r in out if r]


def _series_from_baseline_path(path, warn=True):
    """baseline_AE_PureConcat_cr20.csv -> AE_PureConcat.

    The _cr suffix must come off or the label misses MODEL_LABEL and falls back
    to grey, which silently costs every baseline its colour in the legend.  A
    name that is not a known series at all is reported rather than plotted --
    otherwise a leftover CSV from an earlier revision re-adds a series this
    notebook has dropped, without any error.
    """
    stem = os.path.basename(path)[len("baseline_"):-len(".csv")]
    name = re.sub(r"_cr\d+$", "", stem)
    if warn and name not in MODEL_LABEL:
        print(f"  [warn] {os.path.basename(path)} names unknown series "
              f"{name!r}; not in BASELINE_NAMES, skipping")
        return None
    return name


def _pts(rows):
    out = []
    for r in rows:
        pay, tot, snr = (fnum(r.get("bps_payload")), fnum(r.get("bps_total_amortised")),
                         fnum(r.get("snr_db")))
        if None not in (pay, tot, snr):
            out.append((pay, tot, snr))
    return sorted(out)


# ------------------------------------------------------------------ block diagram
def make_pipeline_figure(out_dir=None):
    out_dir = out_dir or FIG_DIR
    BLUE, ORANGE, GREEN = "#1A56DB", "#D97706", "#15803D"
    PURPLE, PINK, TEAL = "#5B21B6", "#9D174D", "#0F766E"
    GATE, QUANT, CODER, DARK, GREY = "#7C3AED", "#C2410C", "#0F172A", "#111827", "#4B5563"
    BG_E, BG_L, BG_D = "#EFF6FF", "#F5F0FF", "#FFF1F2"

    fig, ax = plt.subplots(figsize=(13.6, 5.9))
    ax.set_xlim(0, 13.6); ax.set_ylim(0, 5.9); ax.axis("off")
    fig.patch.set_facecolor("white")

    def blk(x, y, w, h, t1, t2="", col=BLUE, fs=9.5, ss=7.8, lw=1.8):
        ax.add_patch(FancyBboxPatch((x, y), w, h,
                                    boxstyle="round,pad=0.03,rounding_size=0.10",
                                    fc=col, ec=col, lw=lw, zorder=3))
        ax.text(x + w / 2, y + h * (0.64 if t2 else 0.5), t1, ha="center",
                va="center", fontsize=fs, fontweight="bold", color="white", zorder=4)
        if t2:
            ax.text(x + w / 2, y + h * 0.25, t2, ha="center", va="center",
                    fontsize=ss, color="white", alpha=0.93, zorder=4)

    def ah(x1, x2, y, col, lw=1.8, lbl=""):
        ax.annotate("", xy=(x2, y), xytext=(x1, y),
                    arrowprops=dict(arrowstyle="->", color=col, lw=lw, mutation_scale=14),
                    zorder=7)
        if lbl:
            ax.text((x1 + x2) / 2, y + 0.12, lbl, ha="center", va="bottom",
                    fontsize=8.5, fontweight="bold", color=col)

    ax.text(6.8, 5.55, "Quantization-Aware LARA-Q2  (architecture 2)", ha="center",
            va="center", fontsize=16, fontweight="bold", color=DARK)
    ax.text(6.8, 5.22, "rate–distortion trained  ·  shared step size Δ  "
                       "·  round-trip-verified entropy coding",
            ha="center", va="center", fontsize=10, color=GREY)

    PY, PH = 1.55, 3.25
    for x, w, c, bgc, title in ((0.25, 3.5, BLUE, BG_E, "Encoder"),
                                (4.55, 4.3, PURPLE, BG_L, "Quantization & Coding"),
                                (9.85, 3.5, PINK, BG_D, "Decoder")):
        ax.add_patch(FancyBboxPatch((x, PY - 0.15), w, PH + 0.3,
                                    boxstyle="round,pad=0.06,rounding_size=0.14",
                                    fc=bgc, ec=c, lw=2.0, zorder=1))
        ax.text(x + w / 2, PY + PH, title, ha="center", va="bottom",
                fontsize=11.5, fontweight="bold", color=c)

    blk(0.45, PY + 1.55, 3.1, 1.25, "Conv1d 1→32, k15 s2", "+ BN + ELU + channel attention", col=BLUE, fs=8.4, ss=6.9)
    blk(0.45, PY - 0.05, 3.1, 1.25, "Residual 64→128 (+256 at CR>30)",
        "then 1×1 conv → B, AdaptiveAvgPool → T", col=GREEN, fs=8.0, ss=6.6)
    ax.text(2.0, PY + PH + 0.10, "f ∈ R^(B·T),  B·T = 6016 / 3008",
            ha="center", va="bottom", fontsize=7.4, color=BLUE, style="italic")

    for i, (t1, t2, c) in enumerate([
            ("Feature gate", "f · (1 + tanh p), identity at init", GATE),
            ("Linear  B·T → d", "architecture 2: the bottleneck", PURPLE),
            ("Quantizer  Q_Δ", "train z+u, u~U(−Δ/2,Δ/2)", QUANT),
            ("Entropy coder", "fixed / adaptive / learned", CODER)]):
        blk(4.75, PY + 2.65 - i * 0.95, 3.9, 0.78, t1, t2, col=c, fs=8.6, ss=6.9)
    ax.text(6.7, PY + PH + 0.10, "code z ∈ R^d,  d = 1500 / CR",
            ha="center", va="bottom", fontsize=7.4, color=PURPLE, style="italic")

    for i, (t, c) in enumerate([
            ("Pyramid decode  Linear(d→B·T) + Unflatten", PINK),
            ("6× (nearest upsample + Conv1d k5 + BN + ELU)", PINK),
            ("Conv1d 64→32 k5 + ELU", TEAL),
            ("Conv1d 32→1 k15 + Sigmoid", TEAL)]):
        blk(10.05, PY + 2.65 - i * 0.78, 3.1, 0.66, t, col=c, fs=7.8, ss=6.5)

    ah(3.75, 4.55, PY + 2.4, PURPLE, lw=2.2, lbl="gated features")
    ah(8.85, 9.85, PY + 1.5, CODER, lw=2.2, lbl="bits")

    ax.text(0.25, 1.20, "Code length and quantizer depth", fontsize=9,
            fontweight="bold", color=DARK, va="top")
    hdr = ["CR", "d", "bits", "measured BPS"]
    table = []
    for series in ARCH2_SERIES:
        for path in sorted(glob.glob(os.path.join(RATE_DIR, f"{series}_d*_lam*.csv"))):
            rows = read_csv(path)
            if not rows:
                continue
            d = int(fnum(rows[0].get("latent_dim"), 0) or 0)
            cr = fnum(rows[0].get("cr_implied")) or (INPUT_LEN / d if d else None)
            best = None
            for r in rows:
                b, tot = fnum(r.get("target_bits")), fnum(r.get("bps_total_amortised"))
                if b is None or tot is None:
                    continue
                if best is None or tot < best[1]:
                    best = (int(b), tot)
            if cr and best:
                table.append((cr, d, best[0], best[1]))
    table = sorted(set(table))
    tw, rh = 1.35, 0.20
    body = [[f"{cr:g}", str(d), str(b), (f"{bps:.4f}" if bps else "—")]
            for cr, d, b, bps in table]
    for ri, row in enumerate([hdr] + body):
        for ci, val in enumerate(row):
            x = 0.25 + ci * tw
            y = 1.02 - ri * rh
            bg = DARK if ri == 0 else ("#EEF2FF" if ri % 2 else "white")
            ax.add_patch(FancyBboxPatch((x, y - rh + 0.02), tw - 0.05, rh - 0.02,
                                        boxstyle="round,pad=0.01,rounding_size=0.03",
                                        fc=bg, ec=PURPLE, lw=0.4, zorder=3))
            ax.text(x + tw / 2 - 0.02, y - rh / 2 + 0.03, val, ha="center",
                    va="center", fontsize=6.8, zorder=4,
                    color="white" if ri == 0 else DARK,
                    fontweight="bold" if ri == 0 else "normal")

    ax.text(5.9, 0.95,
            "L = MSE + λ·R̂       Δ is ONE shared scalar (16 bits/model)\n"
            "R̂ = −Σ log₂ p(z | previous symbols),  causal, no lookahead\n"
            "Every reported rate is round-trip verified before it is written",
            fontsize=8.2, color=CODER, va="top", style="italic")

    os.makedirs(out_dir, exist_ok=True)
    for ext in ("png", "eps"):
        p = os.path.join(out_dir, f"Final_LARAQ2_pipeline.{ext}")
        fig.savefig(p, format=ext, dpi=300, bbox_inches="tight", facecolor="white")
        print(f"  [saved] {p}")
    plt.close(fig)


# ---------------------------------------------------------------- RD figures
def rd_envelope(pts, idx):
    """Rate-optimal envelope: at each rate keep the best SNR, then keep only
    points that beat every cheaper one.

    Two steps, and the first is not optional.  Rates are rounded in the CSVs, so
    different depths and coders land on the SAME recorded rate with different
    SNR -- LARA-Q2 has 7.94 dB and 13.24 dB both at 0.0321 bps.  Filtering on
    "beats the previous point" alone keeps both and draws a vertical segment at
    a single x, which is not a trade-off curve.  Collapsing equal rates first
    makes rate and SNR both strictly increasing.

    Filtering here, rather than at load time, is what makes every series
    comparable: a per-series special case is exactly how the LARA series came to
    be drawn at their CR=100 slice while the baselines were drawn in full.
    """
    best = {}
    for p in pts:
        r = p[idx]
        if not (r > 0) or p[2] != p[2]:        # drop non-positive and NaN
            continue
        if r not in best or p[2] > best[r][2]:
            best[r] = p
    out = []
    for r in sorted(best):
        if not out or best[r][2] > out[-1][2]:
            out.append(best[r])
    return out


def draw(ax, data, title, which, connect=True):
    idx = {"payload": 0, "total": 1}[which]
    any_pts = False
    for label, pts in data.items():
        usable = sorted((p for p in pts if p[idx] > 0 and p[2] == p[2]), key=lambda p: p[idx])
        if not usable:
            continue
        env = rd_envelope(usable, idx)
        any_pts = True
        st = STYLE.get(label, dict(color="#4B5563", marker="o"))
        # the measured cloud, faint, so the reader can see what the envelope was
        # taken from -- otherwise each method looks like it was measured at only
        # a handful of rates.
        # STYLE carries color / marker / zorder; the cloud plot sets its own
        # marker and zorder, so those keys must not be splatted a second time.
        cloud = {k: v for k, v in st.items() if k not in ("marker", "zorder")}
        ax.plot([p[idx] for p in usable], [p[2] for p in usable], linestyle="none",
                marker=".", markersize=1.7, alpha=0.09, zorder=1, **cloud)
        # zorder comes from STYLE, as it did before this function gained the
        # envelope -- setting it here as well collides with **st.
        ax.plot([p[idx] for p in env], [p[2] for p in env],
                linestyle="-" if connect else "none", label=label,
                markersize=4.2, linewidth=1.4, alpha=0.95, **st)
    ax.set_xscale("log")
    ax.set_xlabel("coded bits per sample" +
                  ("   (payload only)" if which == "payload"
                   else "   (side information amortised)"), fontsize=9.5)
    ax.set_ylabel("mean per-trace SNR (dB)", fontsize=9.5)
    ax.set_title(title, fontsize=10.5, fontweight="bold")
    ax.grid(True, which="both", alpha=0.3, linewidth=0.6)
    ax.axvline(SOURCE_BPS, color="#111827", linestyle=":", linewidth=1.1, zorder=1)
    if any_pts:
        # lower right, not lower left: every curve now rises left-to-right, so
        # the lower-right corner is the only region no series reaches.  At lower
        # left the legend sat on top of the LARA-Q2 line.
        ax.legend(fontsize=7.5, loc="lower right", framealpha=0.92)
    return any_pts


def load_rd_data(rate_dir, wav_dir, arch3_dir):
    data = {}

    # Every series is accumulated the SAME way -- all of its (payload, total,
    # snr) rows, unfiltered -- and the rate-optimal envelope is taken once, in
    # draw(), where the data is actually plotted.
    #
    # This block used to key rows by depth and keep the MINIMUM
    # bps_total_amortised at each depth.  Total rate falls monotonically as CR
    # rises, so that rule selected the CR=100 model at every depth: LARA-Q2 was
    # drawn as a flat 13 -> 16 dB line spanning 0.03-0.11 bps, while the
    # baselines, which were never collapsed, kept all 70 rows and were joined
    # into a sawtooth.  The proposed method was plotted at its worst slice and
    # its competitors in full.  Its true envelope is 13.3 -> 38.6 dB over 35
    # points, from the same CSVs.
    for series in ARCH2_SERIES + ARCH1_SERIES:
        pts = _pts(glob_rows(rate_dir, f"{series}_d*_lam*.csv"))
        if pts:
            data[series] = pts

    for path in sorted(glob.glob(os.path.join(rate_dir, "baseline_*.csv"))):
        name = _series_from_baseline_path(path)
        if name is None:
            continue
        pts = _pts(read_csv(path))
        if pts:
            data.setdefault(MODEL_LABEL.get(name, name), []).extend(pts)

    pts = _pts(glob_rows(arch3_dir, "arch3_*.csv"))
    if pts:
        data["Arch3Classical"] = pts

    for path in sorted(glob.glob(os.path.join(wav_dir, "wavelet_*_equal_rate.csv"))):
        key = os.path.basename(path).split("_")[1]
        label = {"db4": "Wavelet_DB4", "sym8": "Wavelet_SYM8",
                 "coif3": "Wavelet_COIF3"}.get(key, key)
        pts = _pts(read_csv(path))
        if pts:
            data[label] = pts

    for k in data:
        data[k].sort()
    return data


def make_rd_figures(rate_dir, wav_dir, arch3_dir, amortize=200, out_dir=None):
    out_dir = out_dir or FIG_DIR
    data = load_rd_data(rate_dir, wav_dir, arch3_dir)
    if not data:
        print("  no input CSVs found; skipping the rate-distortion figure")
        return {}
    for k, v in sorted(data.items()):
        print(f"    {k:22s} {len(v):3d} points")
    fig, axes = plt.subplots(1, 2, figsize=(13.2, 5.3))
    draw(axes[0], data, "(a) payload only", "payload")
    draw(axes[1], data, f"(b) side information amortised over {amortize} traces",
         "total", connect=False)
    fig.suptitle("Reconstruction SNR against coded rate — INSTANCE local earthquakes "
                 "(Z, 1500 samples)", fontsize=12, fontweight="bold")
    fig.text(0.5, 0.022, ARCH_NOTE, ha="center", fontsize=8, color="#4B5563",
             style="italic")
    fig.text(0.5, -0.012,
             "learned cloud: λ ∈ {1e-5, 1e-4, 1e-3}; envelope = best point per rate"
             f"   |   {SEGMENT_MODE}   |   n_eval={N_EVAL}   |   "
             f"protocol {PROTOCOL_HASH}",
             ha="center", fontsize=7, color="#4B5563")
    fig.tight_layout(rect=(0, 0.055, 1, 0.94))
    for ext in ("png", "eps"):
        p = os.path.join(out_dir, f"laraq_rd_curves.{ext}")
        fig.savefig(p, format=ext, dpi=300, bbox_inches="tight", facecolor="white")
        print(f"  [saved] {p}")
    plt.close(fig)
    return data


def make_snr_figure(rate_dir, legacy_dir=None, out_dir=None):
    out_dir = out_dir or FIG_DIR
    legacy_dir = legacy_dir or LEGACY_DIR
    data = {}
    for series in ARCH2_SERIES + ARCH1_SERIES:
        paths = sorted(glob.glob(os.path.join(rate_dir, f"{series}_d*_lam*.csv")))
        if not paths:
            print(f"  [warn] no rate CSV found for {series}")
        best = {}
        for path in paths:
            rows = read_csv(path)
            if not rows:
                continue
            if series in LEARNED_SERIES:
                rows = [r for r in rows
                        if fnum(r.get("lam")) is not None
                        and abs(fnum(r.get("lam")) - LAM_HEADLINE) < 1e-15]
            if not rows:
                continue
            cr = fnum(rows[0].get("cr_implied")) or fnum(rows[0].get("cr"))
            top = max((fnum(r.get("snr_db"), -1e9) for r in rows), default=None)
            if cr and top is not None and top > -1e8:
                cur = best.get(round(cr, 3))
                if cur is None or top > cur:
                    best[round(cr, 3)] = top
        if best:
            data[series] = sorted(best.items())
    for path in sorted(glob.glob(os.path.join(rate_dir, "baseline_*.csv"))):
        name = _series_from_baseline_path(path)
        if name is None:
            continue
        rows = read_csv(path)
        if not rows:
            continue
        cr = fnum(rows[0].get("cr"))
        top = max((fnum(r.get("snr_db"), -1e9) for r in rows), default=None)
        if cr and top is not None and top > -1e8:
            data.setdefault(MODEL_LABEL.get(name, name), []).append((cr, top))

    # Legacy Final_LARA wavelet: top-K retention at nominal CR.  It has no
    # measured bps, so it belongs here and ONLY here.  The bit-accurate
    # wavelet codec is loaded by make_measured_cr_figure/load_rd_data.
    for path in sorted(glob.glob(os.path.join(legacy_dir,
                                              "wavelet_*_legacy_cr.csv"))):
        rows = read_csv(path)
        if not rows:
            continue
        label = rows[0].get("series") or (
            "Wavelet_" + os.path.basename(path).split("_")[1].upper())
        pts = []
        for r in rows:
            cr, snr = fnum(r.get("cr")), fnum(r.get("snr_db"))
            if cr is not None and snr is not None:
                pts.append((cr, snr))
        if pts:
            data[label] = sorted(pts)

    data = {k: sorted(v) for k, v in data.items() if v}
    if not data:
        print("  no SNR inputs found; skipping the SNR figure")
        return {}
    fig, ax = plt.subplots(figsize=(9.0, 5.2))
    for label, pts in data.items():
        sel = [p for p in pts if 1.5 <= p[0] <= 130]
        if not sel:
            continue
        ax.plot([p[0] for p in sel], [p[1] for p in sel], label=label, linewidth=1.7,
                markersize=5.2, alpha=0.95,
                **STYLE.get(label, dict(color="#4B5563", marker="o")))
    ax.set_xscale("log")
    ticks = [2, 3, 5, 10, 15, 20, 30, 50, 60, 100]
    ax.set_xticks(ticks); ax.set_xticklabels([str(t) for t in ticks]); ax.minorticks_off()
    ax.set_xlabel("nominal compression ratio", fontsize=10)
    ax.set_ylabel(f"mean per-trace SNR (dB), best depth for learned, "
                  f"λ={LAM_HEADLINE:g}", fontsize=10)
    ax.set_title("SNR against nominal CR (headline λ; learned models quantised)",
                 fontsize=11, fontweight="bold")
    ax.grid(True, which="major", alpha=0.3, linewidth=0.6)
    ax.legend(fontsize=7.5, loc="best", framealpha=0.92)
    fig.text(0.01, 0.010, ARCH_NOTE, ha="left", fontsize=6.8,
             color="#4B5563", style="italic")
    fig.text(0.99, -0.014,
             f"{SEGMENT_MODE}  |  P_OFFSET={P_OFFSET}  |  n_eval={N_EVAL}  |  "
             f"protocol {PROTOCOL_HASH}",
             ha="right", fontsize=6.5, color="#4B5563")
    fig.text(0.5, -0.040,
             "Wavelet_DB4 / Wavelet_SYM8 / Wavelet_COIF3 are the legacy "
             "Final_LARA top-K retention curves at nominal CR.  Positions, "
             "signs and quantisation are NOT charged; they are not codec rates.",
             ha="center", fontsize=6.7, color="#7C2D12", style="italic")
    fig.tight_layout(rect=(0, 0.07, 1, 1))
    for ext in ("png", "eps"):
        p = os.path.join(out_dir, f"snr_vs_cr.{ext}")
        fig.savefig(p, format=ext, dpi=300, bbox_inches="tight", facecolor="white")
        print(f"  [saved] {p}")
    plt.close(fig)
    return data


def _cr_envelope(points):
    """Pareto frontier on a source-CR axis.

    CR rises as rate falls, so the frontier is built from the highest CR down:
    keep a point only when its SNR beats every cheaper (higher-CR) point.  The
    result is returned in ascending CR for plotting.
    """
    best = {}
    for cr, snr in points:
        if cr > 0 and np.isfinite(snr):
            best[cr] = max(best.get(cr, -1e9), snr)
    out, best_snr = [], -1e9
    for cr in sorted(best, reverse=True):
        if best[cr] > best_snr:
            out.append((cr, best[cr]))
            best_snr = best[cr]
    return sorted(out)


def make_measured_cr_figure(rate_dir, wav_dir, arch3_dir, out_dir=None):
    """SNR against measured source CR = 32 / bps_total_amortised.

    This is the only SNR figure that contains the bit-accurate wavelet codec and
    Arch3Classical.  The legacy top-K wavelet has no bps and is deliberately
    absent; it lives on snr_vs_cr.
    """
    out_dir = out_dir or FIG_DIR
    raw = load_rd_data(rate_dir, wav_dir, arch3_dir)
    data = {}
    for label, pts in raw.items():
        conv = [(SOURCE_BPS / tot, snr) for _, tot, snr in pts
                if tot and tot > 0 and np.isfinite(snr)]
        if conv:
            data[label] = conv
    if not data:
        print("  no measured-rate inputs found; skipping the measured-CR figure")
        return {}

    fig, ax = plt.subplots(figsize=(9.4, 5.4))
    for label, pts in sorted(data.items()):
        st = STYLE.get(label, dict(color="#4B5563", marker="o"))
        env = _cr_envelope(pts)
        if not env:
            continue
        ax.plot([p[0] for p in pts], [p[1] for p in pts], linestyle="none",
                marker=".", markersize=1.5, alpha=0.08, color=st.get("color"))
        ax.plot([p[0] for p in env], [p[1] for p in env], linestyle="-",
                label=label, markersize=4.8, linewidth=1.5, alpha=0.95, **st)
    ax.set_xscale("log")
    ticks = [2, 3, 5, 10, 20, 30, 50, 100, 200, 500, 1000]
    ax.set_xticks(ticks)
    ax.set_xticklabels([str(t) for t in ticks])
    ax.minorticks_off()
    ax.set_xlabel("measured source CR   (32 / bps_total_amortised)", fontsize=10)
    ax.set_ylabel("mean per-trace SNR (dB)", fontsize=10)
    ax.set_title("SNR against measured source CR (side information amortised)",
                 fontsize=11, fontweight="bold")
    ax.grid(True, which="major", alpha=0.3, linewidth=0.6)
    ax.legend(fontsize=7.4, loc="best", framealpha=0.92)
    fig.text(0.5, 0.008,
             "Wavelet_DB4 / Wavelet_SYM8 / Wavelet_COIF3 here are the "
             "bit-accurate codec (positions, signs, quantisation coded), not the "
             "legacy retention curves.",
             ha="center", fontsize=6.8, color="#4B5563", style="italic")
    fig.text(0.99, -0.018,
             f"{SEGMENT_MODE}  |  P_OFFSET={P_OFFSET}  |  n_eval={N_EVAL}  |  "
             f"protocol {PROTOCOL_HASH}",
             ha="right", fontsize=6.5, color="#4B5563")
    fig.tight_layout(rect=(0, 0.06, 1, 1))
    for ext in ("png", "eps"):
        p = os.path.join(out_dir, f"snr_vs_measured_cr.{ext}")
        fig.savefig(p, format=ext, dpi=300, bbox_inches="tight", facecolor="white")
        print(f"  [saved] {p}")
    plt.close(fig)
    return data


def make_all_figures():
    try:
        make_pipeline_figure()
    except Exception as e:
        print("  block diagram skipped:", e)
    make_rd_figures(RATE_DIR, WAV_DIR, ARCH3_DIR, amortize=N_EVAL)
    make_snr_figure(RATE_DIR, LEGACY_DIR)
    make_measured_cr_figure(RATE_DIR, WAV_DIR, ARCH3_DIR)


## Summary table and acceptance record


In [ ]:
"""Summary table and acceptance record.

The table is not a substitute for the figures: it is the one-screen check that
every series produced rows, that the clamped-symbol counts are negligible, and
that every CSV carries the current INSTANCE protocol.

Guards that matter here:

* Only rows whose ``protocol_hash`` matches the current run are aggregated.
  A CSV left by an older data definition is ignored and counted as stale; it is
  never silently mixed into the summary.
* ``COMPLETE`` requires the full run grid (or the six-checkpoint pilot), a
  protocol-sized calibration and scored set, complete per-architecture row sets,
  the bit-accurate wavelet and arch-3 arms, the legacy Final_LARA wavelet rows,
  AND the P/S coverage gate.  A low-count, partial, mixed or mis-windowed run is
  written for inspection but can never be called COMPLETE.
* A pilot writes ``acceptance_pilot.json``; a complete pilot with a good window
  reaches ``PILOT``, while a failed window reaches ``PILOT_INVALID_WINDOW``.
  Other failures keep their usual status.  A pilot never overwrites the full
  ``acceptance.json``.
"""


def _read_rows(path):
    if not os.path.isfile(path):
        return []
    with open(path, newline="", encoding="utf-8") as fh:
        return list(csv.DictReader(fh))


def _current_rows(path):
    """Return (rows for this protocol, number of stale rows ignored)."""
    rows = _read_rows(path)
    cur = [r for r in rows if r.get("protocol_hash") == PROTOCOL_HASH]
    return cur, len(rows) - len(cur)


def _f(x, default=float("nan")):
    try:
        v = float(x)
        return v if np.isfinite(v) else default
    except (TypeError, ValueError):
        return default


def _expected_checkpoint_row_count(series):
    """Rows the study measurement writes for one checkpoint.

    Arch 2: fixed + adaptive + learned per depth; baselines: fixed + adaptive
    per depth; arch 1: one top-k row per depth.  A short CSV must not be
    averaged into a COMPLETE summary just because it has one row per CR.
    """
    if series in BASELINE_NAMES:
        return len(DEPTHS) * 2
    if series in ARCH1_SERIES:
        return len(DEPTHS)
    return len(DEPTHS) * 3


def write_summary():
    order = list(ARCH2_SERIES) + (list(ARCH1_SERIES) if DO_ARCH1 else []) + list(BASELINE_NAMES)
    buckets = {}
    all_rate_rows = []
    measured_tags = set()
    stale_ignored = 0
    incomplete_checkpoints = []

    for it in CKPTS:
        rows, stale = _current_rows(it["rate"])
        stale_ignored += stale
        if not rows:
            continue
        measured_tags.add(it["tag"])
        all_rate_rows.extend(rows)
        buckets.setdefault((it["series"], it["cr"]), []).extend(rows)
        want = _expected_checkpoint_row_count(it["series"])
        if len(rows) != want:
            incomplete_checkpoints.append({
                "tag": it["tag"], "rows": len(rows), "expected_rows": want})

    # A current-hash file must also agree on the protocol fields.  This catches
    # a hand-edited or partially stamped CSV instead of averaging it in.
    bad_rows = []
    for r in all_rate_rows:
        if (r.get("segment_mode") != SEGMENT_MODE
                or int(_f(r.get("n_calib_used"), -1)) != N_CALIB
                or int(_f(r.get("n_eval_used"), -1)) != N_EVAL):
            bad_rows.append(r)
            if len(bad_rows) >= 5:
                break
    if bad_rows:
        raise RuntimeError(
            "current protocol_hash rows disagree on segment_mode / counts: "
            + "; ".join(
                f"{r.get('series', '?')} {r.get('segment_mode', '?')} "
                f"{r.get('n_calib_used', '?')}/{r.get('n_eval_used', '?')}"
                for r in bad_rows))

    summary = []
    for (series, cr), rows in sorted(
            buckets.items(), key=lambda kv: (order.index(kv[0][0])
                                             if kv[0][0] in order else 99,
                                             kv[0][1])):
        # Headline lambda when it exists; otherwise the only lambda measured.
        cand = [r for r in rows
                if abs(_f(r.get("lam"), -1.0) - LAM_HEADLINE) < 1e-15]
        if not cand:
            cand = rows
        best = max(cand, key=lambda r: (_f(r.get("snr_db"), -1e9),
                                        -_f(r.get("bps_total_amortised"), 1e9)))
        summary.append({
            "series": series,
            "cr": cr,
            "latent_dim": best.get("latent_dim", ""),
            "lam": best.get("lam", ""),
            "depth": best.get("target_bits", best.get("depth", "")),
            "coder": best.get("coder", ""),
            "bps_payload": best.get("bps_payload", ""),
            "bps_total_amortised": best.get("bps_total_amortised", ""),
            "snr_db": best.get("snr_db", ""),
            "pct_clamped": best.get("pct_clamped", ""),
            "n_clamped": best.get("n_clamped", ""),
            "source": "checkpoint",
            "protocol_hash": PROTOCOL_HASH,
            "segment_mode": SEGMENT_MODE,
            "n_calib_used": N_CALIB,
            "n_eval_used": N_EVAL,
        })

    for w in WAVELETS:
        p = os.path.join(WAV_DIR, f"wavelet_{w}_equal_rate.csv")
        rows, stale = _current_rows(p)
        stale_ignored += stale
        for r in rows:
            summary.append({
                "series": f"Wavelet-{w}", "cr": "NaN",
                "latent_dim": r.get("K", ""), "lam": "",
                "depth": r.get("depth", ""), "coder": "topk+exact-positions",
                "bps_payload": r.get("bps_payload", ""),
                "bps_total_amortised": r.get("bps_total_amortised", ""),
                "snr_db": r.get("snr_db", ""), "pct_clamped": "",
                "n_clamped": "", "source": "wavelet",
                "protocol_hash": PROTOCOL_HASH,
                "segment_mode": SEGMENT_MODE,
                "n_calib_used": N_CALIB,
                "n_eval_used": N_EVAL,
            })

    p3 = os.path.join(ARCH3_DIR, "arch3_rates.csv")
    rows3, stale3 = _current_rows(p3)
    stale_ignored += stale3
    for r in rows3:
        summary.append({
            "series": "Arch3Classical", "cr": "NaN",
            "latent_dim": r.get("K", ""), "lam": "",
            "depth": r.get("depth", ""), "coder": r.get("coder", "adaptive"),
            "bps_payload": r.get("bps_payload", ""),
            "bps_total_amortised": r.get("bps_total_amortised", ""),
            "snr_db": r.get("snr_db", ""), "pct_clamped": "",
            "n_clamped": "", "source": "arch3",
            "protocol_hash": PROTOCOL_HASH,
            "segment_mode": SEGMENT_MODE,
            "n_calib_used": N_CALIB,
            "n_eval_used": N_EVAL,
        })

    # Legacy Final_LARA wavelet: deliberately NOT appended to summary.csv.
    # It has nominal CR and SNR but no coded bps, so mixing it into the
    # summary would invite a rate comparison it cannot support.  It is still
    # required for the snr_vs_cr figure and therefore for COMPLETE.
    legacy_files, legacy_rows, legacy_bad = [], 0, []
    for w in WAVELETS:
        lp = os.path.join(LEGACY_DIR, f"wavelet_{w}_legacy_cr.csv")
        lrows, lstale = _current_rows(lp)
        stale_ignored += lstale
        legacy_files.append(os.path.basename(lp))
        legacy_rows += len(lrows)
        have_crs = sorted(_f(r.get("cr")) for r in lrows)
        want_crs = sorted(float(x) for x in LEGACY_WAVELET_CRS)
        if (len(lrows) != len(LEGACY_WAVELET_CRS)
                or len(have_crs) != len(want_crs)
                or any(not np.isfinite(a) for a in have_crs)
                or any(abs(a - b) > 1e-9 for a, b in zip(have_crs, want_crs))):
            legacy_bad.append({"file": os.path.basename(lp),
                               "rows": len(lrows),
                               "expected_rows": len(LEGACY_WAVELET_CRS),
                               "crs": have_crs})
    legacy_ok = (not DO_LEGACY_WAVELET) or (
        not legacy_bad and legacy_rows == len(WAVELETS) * len(LEGACY_WAVELET_CRS))

    summary_cols = [
        "series", "cr", "latent_dim", "lam", "depth", "coder", "bps_payload",
        "bps_total_amortised", "snr_db", "pct_clamped", "n_clamped", "source",
        "protocol_hash", "segment_mode", "n_calib_used", "n_eval_used",
    ]
    df = pd.DataFrame(summary, columns=summary_cols)
    p = os.path.join(OUT_DIR, "summary.csv")
    df.to_csv(p, index=False, encoding="utf-8")
    show = [c for c in ("series", "cr", "depth", "coder", "bps_total_amortised",
                        "snr_db", "pct_clamped") if c in df.columns]
    print("\n### summary")
    print(df[show].to_string(index=False))
    print(f"\n[saved] {p}  ({len(df)} rows)")

    # ------------------------------------------------------------ guards
    missing_measurements = [it["tag"] for it in CKPTS
                            if it["tag"] not in measured_tags]
    protocol_hashes = sorted({str(r["protocol_hash"]) for r in summary})
    segment_modes = sorted({str(r["segment_mode"]) for r in summary})
    protocol_ok = (not summary) or (
        len(protocol_hashes) == 1
        and protocol_hashes[0] == PROTOCOL_HASH
        and len(segment_modes) == 1
        and segment_modes[0] == SEGMENT_MODE)
    if summary and not protocol_ok:
        raise RuntimeError(
            "summary would mix protocols: "
            f"hashes={protocol_hashes}, segment_modes={segment_modes}; "
            f"current hash={PROTOCOL_HASH}")

    # ------------------------------------------------------------ acceptance
    max_clamp = 0.0
    for r in all_rate_rows:
        v = _f(r.get("pct_clamped"), 0.0)
        if np.isfinite(v):
            max_clamp = max(max_clamp, v)

    per_series = {}
    for r in summary:
        per_series[r["series"]] = per_series.get(r["series"], 0) + 1
    figs = sorted(glob.glob(os.path.join(FIG_DIR, "**", "*.png"),
                            recursive=True))
    _ext = globals().get("EXTENDED") or {}

    run_expected = list(globals().get("RUN_EXPECTED", EXPECTED))
    expected_pairs = {(it["series"], int(it["cr"])) for it in run_expected}
    expected_summary_rows = (
        len(expected_pairs)
        + len(WAVELETS) * len(WAVE_TARGETS_BPS)
        + (len(ARCH3_TARGETS_BPS) if DO_ARCH3 else 0))
    summary_rows_ok = (len(summary) == expected_summary_rows)
    checkpoint_rows_ok = not incomplete_checkpoints
    window_ok = bool(globals().get("WINDOW_OK", False))
    window_stats = globals().get("WINDOW_STATS", {})

    counts_ok = (N_CALIB >= MIN_CALIB_VALID and N_EVAL >= MIN_EVAL_VALID)
    grid_ok = (len(CKPTS) == len(run_expected)
               and not FAILED and not MISSING and not MEASURE_FAILED
               and not missing_measurements and summary_rows_ok
               and checkpoint_rows_ok and legacy_ok)
    if QUICK:
        if not window_ok:
            status = "INVALID_WINDOW"
        elif not (grid_ok and protocol_ok):
            status = "PARTIAL"
        else:
            status = "QUICK"
    elif PILOT:
        if not window_ok:
            status = "PILOT_INVALID_WINDOW"
        elif not protocol_ok:
            status = "INVALID_MIXED_PROTOCOL"
        elif not counts_ok:
            status = "INVALID_LOW_COUNT"
        elif not grid_ok:
            status = "PARTIAL"
        else:
            status = "PILOT"
    else:
        if not window_ok:
            status = "INVALID_WINDOW"
        elif not protocol_ok:
            status = "INVALID_MIXED_PROTOCOL"
        elif not counts_ok:
            status = "INVALID_LOW_COUNT"
        elif not grid_ok:
            status = "PARTIAL"
        else:
            status = "COMPLETE"

    acceptance = {
        "generated": time.strftime("%Y-%m-%d %H:%M:%S"),
        "status": status,
        "acceptance_file": ACCEPTANCE_FILE,
        "pilot": bool(PILOT),
        "pilot_specs": PILOT_SPECS,
        "quick": bool(QUICK),
        "protocol_hash": PROTOCOL_HASH,
        "protocol": PROTOCOL,
        "protocol_ok": bool(protocol_ok),
        "counts_ok": bool(counts_ok),
        "window_ok": bool(window_ok),
        "window_stats": window_stats,
        "p_offset": P_OFFSET,
        "p_coverage_min_pct": P_COVERAGE_MIN,
        "s_coverage_min_pct": S_COVERAGE_MIN,
        "p_coverage_min_observed_pct": window_stats.get(
            "p_coverage_min_observed_pct"),
        "s_coverage_min_observed_pct": window_stats.get(
            "s_coverage_min_observed_pct"),
        "checkpoints_expected": len(run_expected),
        "checkpoints_expected_full_grid": len(EXPECTED_FULL),
        "checkpoints_found": len(FOUND),
        "checkpoints_loaded": len(CKPTS),
        "checkpoints_missing": len(MISSING),
        "checkpoints_load_failed": len(FAILED),
        "checkpoints_measure_failed": len(MEASURE_FAILED),
        "missing_measurements": missing_measurements,
        "checkpoint_rows_ok": bool(checkpoint_rows_ok),
        "incomplete_checkpoints": incomplete_checkpoints[:20],
        "summary_rows": len(summary),
        "summary_rows_expected": expected_summary_rows,
        "summary_rows_ok": bool(summary_rows_ok),
        "summary_rows_per_series": per_series,
        "summary_protocol_hashes": protocol_hashes,
        "summary_segment_modes": segment_modes,
        "stale_rows_ignored": stale_ignored,
        "max_pct_clamped": round(max_clamp, 6),
        "clamp_status": "OK" if max_clamp < 0.1 else "CHECK",
        "segment_mode": SEGMENT_MODE,
        "n_calib_used": N_CALIB,
        "n_eval_used": N_EVAL,
        "n_calib_target": N_CALIB_TARGET,
        "n_eval_target": N_EVAL_TARGET,
        "min_calib_valid": MIN_CALIB_VALID,
        "min_eval_valid": MIN_EVAL_VALID,
        "low_count_policy": LOW_COUNT_POLICY,
        "legacy_wavelet": {
            "enabled": bool(DO_LEGACY_WAVELET),
            "ok": bool(legacy_ok),
            "files": legacy_files,
            "rows": int(legacy_rows),
            "rows_expected": int(len(WAVELETS) * len(LEGACY_WAVELET_CRS)),
            "bad": legacy_bad,
            "nominal_crs": LEGACY_WAVELET_CRS,
            "note": "top-K retention at nominal CR; no position/sign/quantisation cost",
        },
        "figures": [os.path.relpath(x, FIG_DIR).replace(os.sep, "/")
                    for x in figs],
        "extended_diagnostics": {
            "ran": bool(_ext) and not _ext.get("disabled", False),
            "disabled": bool(_ext.get("disabled", False)) if _ext else False,
            "ok": bool(_ext.get("ok", False)) if _ext else False,
            "diagnostic_rows": _ext.get("diagnostic_rows", 0) if _ext else 0,
            "failures": _ext.get("failures", []) if _ext else [],
        },
        "wavelets": WAVELETS,
        "wave_targets_bps": WAVE_TARGETS_BPS,
        "arch3_targets_bps": ARCH3_TARGETS_BPS,
        "depths": DEPTHS,
        "device": str(DEVICE),
    }
    acceptance_path = os.path.join(OUT_DIR, ACCEPTANCE_FILE)
    with open(acceptance_path, "w", encoding="utf-8") as fh:
        json.dump(acceptance, fh, indent=1)

    print("\n" + "=" * 78)
    print("ACCEPTANCE")
    print(f"  status                {acceptance['status']}")
    print(f"  mode                  {'QUICK' if QUICK else ('PILOT' if PILOT else 'FULL')}")
    print(f"  acceptance file       {ACCEPTANCE_FILE}")
    print(f"  protocol              {PROTOCOL_HASH}  ({SEGMENT_MODE})")
    print(f"  checkpoints           {len(CKPTS)} loaded / {len(run_expected)} "
          f"expected for this run  (full grid {len(EXPECTED_FULL)})")
    print(f"                        {len(MISSING)} missing, {len(FAILED)} load "
          f"failures, {len(MEASURE_FAILED)} measure failures, "
          f"{len(missing_measurements)} without current rows, "
          f"{len(incomplete_checkpoints)} incomplete row sets")
    print(f"  window gate           {'PASS' if window_ok else 'FAIL'}"
          f"  P>={P_COVERAGE_MIN:g}% (obs "
          f"{window_stats.get('p_coverage_min_observed_pct')}%)  "
          f"S>={S_COVERAGE_MIN:g}% (obs "
          f"{window_stats.get('s_coverage_min_observed_pct')}%)")
    print(f"  summary rows          {len(summary)} / {expected_summary_rows} expected"
          f"  (stale rows ignored: {stale_ignored})")
    print(f"  legacy wavelet        {legacy_rows} / "
          f"{len(WAVELETS) * len(LEGACY_WAVELET_CRS)} rows  "
          f"({'OK' if legacy_ok else 'MISSING/INCOMPLETE'})")
    print(f"  max clamped symbols   {max_clamp:.6f} %   ({acceptance['clamp_status']})")
    print(f"  calibration / scored  {N_CALIB} / {N_EVAL}"
          f"  (targets {N_CALIB_TARGET} / {N_EVAL_TARGET}, {LOW_COUNT_POLICY})")
    print(f"  counts valid          {counts_ok}"
          f"  (minimum {MIN_CALIB_VALID} / {MIN_EVAL_VALID})")
    print(f"  segment rule          {SEGMENT_MODE}  P_OFFSET={P_OFFSET}")
    print(f"  figures               {len(figs)}")
    if status not in ("COMPLETE", "QUICK", "PILOT"):
        print("  NOTE                  this run is NOT a valid comparison; "
              "see status above")
    print("=" * 78)
    return acceptance


## Extended diagnostics: learning curves, reconstruction, and parameter analysis


These diagnostics adapt the useful part of the
`Visualisation, learning curves & parameter analysis` section from
`Final_LARA_27092026.ipynb`. They are not imported from that notebook; the code
is inlined so the validation notebook remains self-contained.

**Two kinds of curve are kept separate.**

- **Training curves** are read from `ckpts_baselines/history_<tag>.json`. They
  show the 110 checkpoints being trained on **STEAD**. The validation notebook
  does not train on INSTANCE, so it cannot produce INSTANCE learning curves.
- **Measurement curves** use the per-checkpoint INSTANCE CSVs already written
  by this run: SNR, coded rate, depth, lambda and clamped-symbol percentage.

The reconstruction examples are the only additional inference pass. They reload
a small, named subset of checkpoints (not all 110), recalibrate Delta on the
INSTANCE calibration split with `set_calibrated_delta`, quantize, decode, and
plot the first few evaluation traces. Each waveform figure marks P and S at
their positions **inside the P-aligned window**; the per-trace CSV records
`window_start`, `p_rel`, `s_rel` and `s_inside`. They do not change the
measurement protocol or any CSV used by the study comparison. The diagnostic
CSVs carry the same `protocol_hash`, `segment_mode`, `p_offset`, `n_calib_used`
and `n_eval_used` fields as the measurement CSVs; stale-protocol rate rows are
ignored. The separate per-sample frequency-spectrum plot from `Final_LARA` is
merged into the three-column waveform figure (time, residual, spectrum) instead
of duplicating it.

Set `LARA_EXTENDED_DIAGNOSTICS=0` to skip this section, or
`LARA_RECONSTRUCTION_DIAGNOSTICS=0` to keep the parameter/learning-curve
analysis but skip the waveform pass.


In [ ]:
"""Extended diagnostics: learning curves, reconstruction, and parameter analysis.

This cell ports the useful part of the ``Visualisation, learning curves &
parameter analysis`` section from ``Final_LARA_27092026.ipynb`` and adapts it to
the validation result schema.  It does not import that notebook at run time.

Two distinct kinds of curve are kept separate:

* **Training curves** come from ``ckpts_baselines/history_<tag>.json``.  They show
  the checkpoints being trained on STEAD.  The validation notebook does not
  train on INSTANCE, so it cannot produce INSTANCE learning curves.
* **Measurement curves** come from the per-checkpoint INSTANCE CSVs already
  written by this run: SNR, coded rate, depth, lambda and clamp percentage.

The reconstruction examples are the only part that performs extra inference.
They reload a small, named subset of checkpoints (not all 110), recalibrate
Delta on the INSTANCE calibration split with the study's
``set_calibrated_delta``, quantize, decode, and plot the first few evaluation
traces.  The measurement pass itself is not changed.
"""
import csv
import glob
import json
import math
import os
import traceback

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt


DIAG_DIR = os.path.join(OUT_DIR, "diagnostics")
LC_DIR = os.path.join(FIG_DIR, "learning_curves")
PARAM_DIR = os.path.join(FIG_DIR, "model_analysis")
REC_DIR = os.path.join(FIG_DIR, "reconstruction")
for _d in (DIAG_DIR, LC_DIR, PARAM_DIR, REC_DIR):
    os.makedirs(_d, exist_ok=True)

# A fixed measurement slice keeps the diagnostic table comparable.  If a
# checkpoint does not contain this depth or coder, the closest available row is
# used and the chosen depth/coder is recorded in the table.
DIAG_DEPTH = 5
DIAG_CODER = "learned"
DIAG_TRACES = 6
DIAG_CRS = [20, 100]
DIAG_LAMBDA_WIDTH = 75
DO_EXTENDED_DIAGNOSTICS = os.environ.get("LARA_EXTENDED_DIAGNOSTICS", "1") != "0"
DO_RECONSTRUCTION_DIAGNOSTICS = os.environ.get(
    "LARA_RECONSTRUCTION_DIAGNOSTICS", "1") != "0"

DIAG_COLORS = {
    "LARA-Q2": "#1A56DB",
    "LARA-Q1": "#D97706",
    "Light_HARA-Q1": "#15803D",
    "GeneralizedAutoencoder": "#5B21B6",
    "AE_PureConcat": "#9D174D",
}
DIAG_MARKERS = {
    "LARA-Q2": "o",
    "LARA-Q1": "s",
    "Light_HARA-Q1": "^",
    "GeneralizedAutoencoder": "D",
    "AE_PureConcat": "v",
}
DIAG_LABELS = dict(globals().get("MODEL_LABEL", {}))
for _s in LEARNED_SERIES + BASELINE_NAMES:
    DIAG_LABELS.setdefault(_s, _s)


def _dfnum(v, default=float("nan")):
    try:
        x = float(v)
        return x if np.isfinite(x) else default
    except (TypeError, ValueError):
        return default


def _diag_slug(text):
    return (str(text).replace(" ", "_").replace("/", "_")
            .replace("\\", "_").replace("(", "").replace(")", ""))


def _diag_save(fig, base):
    for ext in ("png", "eps"):
        fig.savefig(base + "." + ext, dpi=300, bbox_inches="tight",
                    facecolor="white")
    plt.close(fig)


def _history_path(it):
    return os.path.join(CKPT_DIR, f"history_{it['tag']}.json")


def load_history(it):
    """Read one training-history JSON without failing the whole diagnostic."""
    p = _history_path(it)
    if not os.path.isfile(p):
        return []
    try:
        with open(p, encoding="utf-8") as fh:
            hist = json.load(fh)
        if not isinstance(hist, list):
            return []
        return [h for h in hist if isinstance(h, dict) and "epoch" in h]
    except Exception:
        return []


def _history_summary(it):
    hist = load_history(it)
    out = {
        "history_epochs": 0,
        "best_val_epoch": "",
        "final_train_mse": np.nan,
        "final_val_mse": np.nan,
        "final_val_loss": np.nan,
        "final_val_bps": np.nan,
        "best_val_mse": np.nan,
    }
    if not hist:
        return out
    epochs = [int(_dfnum(h.get("epoch"), i + 1)) for i, h in enumerate(hist)]
    out["history_epochs"] = len(hist)
    out["final_train_mse"] = _dfnum(hist[-1].get("train_mse"))
    out["final_val_mse"] = _dfnum(hist[-1].get("val_mse"))
    out["final_val_loss"] = _dfnum(hist[-1].get("val_loss"))
    out["final_val_bps"] = _dfnum(hist[-1].get("val_bps"))
    vals = [_dfnum(h.get("val_mse")) for h in hist]
    good = [i for i, v in enumerate(vals) if np.isfinite(v)]
    if good:
        j = min(good, key=lambda i: vals[i])
        out["best_val_epoch"] = epochs[j]
        out["best_val_mse"] = vals[j]
    return out


def _rate_rows(it):
    p = it.get("rate")
    if not p or not os.path.isfile(p):
        return []
    try:
        with open(p, newline="", encoding="utf-8") as fh:
            rows = list(csv.DictReader(fh))
        return [r for r in rows if r.get("protocol_hash") == PROTOCOL_HASH]
    except Exception:
        return []


def _row_depth(row):
    for key in ("target_bits", "depth"):
        v = row.get(key)
        if v not in (None, ""):
            d = _dfnum(v)
            if np.isfinite(d):
                return int(round(d))
    return None


def _pick_measured(it, depth=DIAG_DEPTH, coder=DIAG_CODER):
    """Pick one measured row per checkpoint, preferring depth/coder."""
    rows = _rate_rows(it)
    if not rows:
        return {}
    same = [r for r in rows if _row_depth(r) == int(depth)]
    if not same:
        same = rows
    for want in (coder, "learned", "adaptive", "fixed", "topk+exact-positions"):
        cand = [r for r in same if (r.get("coder") or "") == want]
        if cand:
            same = cand
            break
    return max(same, key=lambda r: _dfnum(r.get("snr_db"), -1e9))


def build_diagnostic_table():
    """One row per checkpoint: parameters, history, measured INSTANCE result."""
    rows = []
    for it in CKPTS:
        hs = _history_summary(it)
        m = _pick_measured(it)
        params = int(it.get("params") or 0)
        size = _dfnum(it.get("model_size_mb"))
        if not np.isfinite(size):
            size = params * 4.0 / (1024 ** 2)
        rows.append({
            "series": it["series"],
            "cr": int(it["cr"]),
            "d": int(it["d"]),
            "lam": _dfnum(it.get("lam"), 0.0),
            "arch": int((it.get("meta") or {}).get("arch") or it.get("arch") or 0),
            "params": params,
            "params_m": params / 1e6,
            "model_size_mb": size,
            "checkpoint_epoch": (int(_dfnum(it.get("epoch"), 0))
                                 if it.get("epoch") else np.nan),
            **hs,
            "measured_depth": _row_depth(m) if m else "",
            "measured_coder": (m.get("coder") or "") if m else "",
            "measured_bps": _dfnum(m.get("bps_total_amortised")) if m else np.nan,
            "measured_payload_bps": _dfnum(m.get("bps_payload")) if m else np.nan,
            "measured_compression": _dfnum(m.get("compression_vs_source")) if m else np.nan,
            "measured_snr_db": _dfnum(m.get("snr_db")) if m else np.nan,
            "measured_clamp_pct": _dfnum(m.get("pct_clamped")) if m else np.nan,
            "protocol_hash": PROTOCOL_HASH,
            "segment_mode": SEGMENT_MODE,
            "n_calib_used": N_CALIB,
            "n_eval_used": N_EVAL,
        })
    df = pd.DataFrame(rows)
    p = os.path.join(DIAG_DIR, "checkpoint_analysis.csv")
    df.to_csv(p, index=False, encoding="utf-8")
    print(f"  [saved] {p}  ({len(df)} rows)")
    return df


def _headline(df):
    """Learned rows at the headline lambda plus the two baselines."""
    if df.empty:
        return df
    learned = df[(df["series"].isin(LEARNED_SERIES))
                 & np.isclose(df["lam"].astype(float), LAM_HEADLINE)]
    base = df[df["series"].isin(BASELINE_NAMES)]
    return pd.concat([learned, base], ignore_index=True)


def _series_order(df):
    order = list(LEARNED_SERIES) + list(BASELINE_NAMES)
    return [s for s in order if s in set(df["series"])]


def _epoch_arrays(hist):
    ep = np.asarray([_dfnum(h.get("epoch"), i + 1) for i, h in enumerate(hist)])
    def arr(key):
        return np.asarray([_dfnum(h.get(key)) for h in hist])
    return ep, arr


def _style_axes(ax, title, xlabel, ylabel):
    ax.set_title(title, fontsize=10.5, fontweight="bold")
    ax.set_xlabel(xlabel, fontsize=9)
    ax.set_ylabel(ylabel, fontsize=9)
    ax.grid(True, alpha=0.25)
    ax.tick_params(labelsize=8)


def plot_training_history(it):
    """Training and validation curves for one checkpoint, with epoch marker."""
    hist = load_history(it)
    if not hist:
        return None
    ep, arr = _epoch_arrays(hist)
    png_base = os.path.join(
        LC_DIR, f"{it['tag']}_training_history")
    fig, axes = plt.subplots(1, 2, figsize=(12.4, 4.2))

    ax = axes[0]
    ax.plot(ep, arr("train_mse"), "o-", color="#1A56DB", ms=2.4, lw=1.3,
            label="train MSE")
    ax.plot(ep, arr("val_mse"), "o-", color="#C2410C", ms=2.4, lw=1.3,
            label="validation MSE")
    ax.plot(ep, arr("val_loss"), "-", color="#111827", lw=1.1, alpha=0.75,
            label="validation loss")
    ax.set_yscale("log")
    _style_axes(ax, "MSE / loss", "Epoch", "Value (log scale)")
    ax.legend(fontsize=8, loc="best")

    ax = axes[1]
    ax.plot(ep, arr("train_bps"), "o-", color="#1A56DB", ms=2.4, lw=1.3,
            label="train bps")
    ax.plot(ep, arr("val_bps"), "o-", color="#C2410C", ms=2.4, lw=1.3,
            label="validation bps")
    _style_axes(ax, "Rate proxy", "Epoch", "Bits per sample")
    ax.legend(fontsize=8, loc="best")

    epoch = int(_dfnum(it.get("epoch"), 0)) if it.get("epoch") else 0
    if epoch and ep.min() <= epoch <= ep.max():
        for ax in axes:
            ax.axvline(epoch, color="#15803D", lw=1.2, ls="--", alpha=0.85)
        axes[0].annotate(f"selected epoch {epoch}", xy=(epoch, 0.04),
                         xycoords=("data", "axes fraction"), fontsize=8,
                         color="#15803D", ha="left", va="bottom", rotation=90)

    fig.suptitle(
        f"{it['series']}  d={it['d']}  CR={it['cr']}  "
        f"lambda={_dfnum(it.get('lam'), 0.0):g}  —  STEAD training history",
        fontsize=12, fontweight="bold")
    fig.tight_layout(rect=(0, 0, 1, 0.93))
    _diag_save(fig, png_base)
    return png_base + ".png"


def plot_learning_overview():
    """Faceted training curves for every width at the headline lambda."""
    made = []
    for series in LEARNED_SERIES:
        items = [it for it in CKPTS
                 if it["series"] == series
                 and np.isclose(float(it.get("lam") or 0.0), LAM_HEADLINE)]
        items = sorted(items, key=lambda x: WIDTHS.index(x["d"])
                       if x["d"] in WIDTHS else 999)
        if not items:
            continue
        n = len(items)
        ncol = 5 if n > 5 else max(1, n)
        nrow = int(math.ceil(n / ncol))
        fig, axes = plt.subplots(nrow, ncol, figsize=(3.5 * ncol, 2.8 * nrow),
                                 squeeze=False, sharex=True)
        for ax, it in zip(axes.ravel(), items):
            hist = load_history(it)
            if not hist:
                ax.text(0.5, 0.5, "no history", ha="center", va="center",
                        transform=ax.transAxes, fontsize=8)
                ax.set_axis_off()
                continue
            ep, arr = _epoch_arrays(hist)
            ax.plot(ep, arr("train_mse"), color="#93C5FD", lw=1.1,
                    label="train")
            ax.plot(ep, arr("val_mse"), color=DIAG_COLORS.get(series, "#1A56DB"),
                    lw=1.4, label="val")
            epoch = int(_dfnum(it.get("epoch"), 0)) if it.get("epoch") else 0
            if epoch and ep.min() <= epoch <= ep.max():
                ax.axvline(epoch, color="#15803D", lw=1.0, ls="--")
            ax.set_yscale("log")
            _style_axes(ax, f"d={it['d']}  (CR {it['cr']})", "Epoch", "MSE")
            ax.legend(fontsize=6.5, loc="best")
        for ax in axes.ravel()[n:]:
            ax.set_axis_off()
        fig.suptitle(
            f"{DIAG_LABELS.get(series, series)} — STEAD training curves at "
            f"lambda={LAM_HEADLINE:g} (train vs validation MSE)",
            fontsize=12, fontweight="bold")
        fig.tight_layout(rect=(0, 0, 1, 0.94))
        base = os.path.join(LC_DIR, f"learning_curves_{_diag_slug(series)}")
        _diag_save(fig, base)
        made.append(base + ".png")
    return made


def plot_learning_baselines():
    """Training curves for the two non-LARA autoencoders, one panel per CR."""
    made = []
    for series in BASELINE_NAMES:
        items = sorted([it for it in CKPTS if it["series"] == series],
                       key=lambda x: x["cr"])
        if not items:
            continue
        n = len(items)
        ncol = 5 if n > 5 else max(1, n)
        nrow = int(math.ceil(n / ncol))
        fig, axes = plt.subplots(nrow, ncol, figsize=(3.5 * ncol, 2.8 * nrow),
                                 squeeze=False, sharex=True)
        for ax, it in zip(axes.ravel(), items):
            hist = load_history(it)
            if not hist:
                ax.set_axis_off()
                continue
            ep, arr = _epoch_arrays(hist)
            ax.plot(ep, arr("train_mse"), color="#C4B5FD", lw=1.1, label="train")
            ax.plot(ep, arr("val_mse"), color=DIAG_COLORS.get(series, "#5B21B6"),
                    lw=1.4, label="val")
            ax.set_yscale("log")
            _style_axes(ax, f"CR {it['cr']}  (d={it['d']})", "Epoch", "MSE")
            ax.legend(fontsize=6.5, loc="best")
        for ax in axes.ravel()[n:]:
            ax.set_axis_off()
        fig.suptitle(
            f"{DIAG_LABELS.get(series, series)} — STEAD training curves "
            "(train vs validation MSE)", fontsize=12, fontweight="bold")
        fig.tight_layout(rect=(0, 0, 1, 0.94))
        base = os.path.join(LC_DIR, f"learning_curves_{_diag_slug(series)}")
        _diag_save(fig, base)
        made.append(base + ".png")
    return made


def plot_learning_lambda():
    """Validation MSE for the three lambdas at a representative width."""
    made = []
    for series in LEARNED_SERIES:
        items = [it for it in CKPTS
                 if it["series"] == series and it["d"] == DIAG_LAMBDA_WIDTH]
        if not items:
            continue
        items = sorted(items, key=lambda x: float(x.get("lam") or 0.0))
        fig, axes = plt.subplots(1, len(items), figsize=(3.8 * len(items), 3.4),
                                 squeeze=False)
        for ax, it in zip(axes.ravel(), items):
            hist = load_history(it)
            if not hist:
                ax.set_axis_off()
                continue
            ep, arr = _epoch_arrays(hist)
            ax.plot(ep, arr("train_mse"), color="#93C5FD", lw=1.0, label="train")
            ax.plot(ep, arr("val_mse"), color=DIAG_COLORS.get(series, "#1A56DB"),
                    lw=1.5, label="val")
            ax.set_yscale("log")
            _style_axes(ax, f"lambda={float(it.get('lam') or 0):g}", "Epoch", "MSE")
            ax.legend(fontsize=7, loc="best")
        fig.suptitle(
            f"{DIAG_LABELS.get(series, series)} — lambda sensitivity at "
            f"d={DIAG_LAMBDA_WIDTH} (STEAD training)",
            fontsize=12, fontweight="bold")
        fig.tight_layout(rect=(0, 0, 1, 0.92))
        base = os.path.join(LC_DIR, f"learning_curves_lambda_{_diag_slug(series)}")
        _diag_save(fig, base)
        made.append(base + ".png")
    return made


def plot_training_summary(df):
    """Final training loss, rate, and convergence against CR."""
    sub = _headline(df)
    if sub.empty:
        return None
    fig, axes = plt.subplots(2, 2, figsize=(13.2, 8.0))
    axes = axes.ravel()
    for series in _series_order(sub):
        d = sub[sub["series"] == series].sort_values("cr")
        if d.empty:
            continue
        kw = dict(color=DIAG_COLORS.get(series, "#4B5563"),
                  marker=DIAG_MARKERS.get(series, "o"), ms=5, lw=1.5,
                  label=DIAG_LABELS.get(series, series))
        axes[0].plot(d["cr"], d["final_val_mse"], **kw)
        axes[1].plot(d["cr"], d["final_val_bps"], **kw)
        axes[2].plot(d["cr"], d["history_epochs"], **kw)
        axes[3].plot(d["cr"], d["checkpoint_epoch"], **kw)
    axes[0].set_yscale("log")
    _style_axes(axes[0], "Final validation MSE", "CR", "MSE (log)")
    _style_axes(axes[1], "Final validation rate proxy", "CR", "bits / sample")
    _style_axes(axes[2], "History length", "CR", "epochs")
    _style_axes(axes[3], "Selected checkpoint epoch", "CR", "epoch")
    for ax in axes:
        ax.set_xscale("log")
        ax.legend(fontsize=7, ncol=1, loc="best")
    fig.suptitle(
        "Training summary — histories are STEAD training, not INSTANCE "
        "retraining", fontsize=12, fontweight="bold")
    fig.tight_layout(rect=(0, 0, 1, 0.94))
    base = os.path.join(LC_DIR, "training_summary")
    _diag_save(fig, base)
    return base + ".png"


def plot_parameter_comparison(df):
    """Parameter count and float32 model size against CR."""
    sub = _headline(df).drop_duplicates(["series", "cr"])
    if sub.empty:
        return None
    fig, axes = plt.subplots(1, 2, figsize=(13.0, 5.0))
    for series in _series_order(sub):
        d = sub[sub["series"] == series].sort_values("cr")
        kw = dict(color=DIAG_COLORS.get(series, "#4B5563"),
                  marker=DIAG_MARKERS.get(series, "o"), ms=6, lw=1.6,
                  label=DIAG_LABELS.get(series, series))
        axes[0].plot(d["cr"], d["params_m"], **kw)
        axes[1].plot(d["cr"], d["model_size_mb"], **kw)
    for ax in axes:
        ax.set_xscale("log")
        ax.legend(fontsize=8)
    _style_axes(axes[0], "Learnable parameters", "CR", "Millions of parameters")
    _style_axes(axes[1], "Model size (float32 parameters + buffers)", "CR", "MB")
    fig.suptitle("Parameter analysis across the checkpoint grid",
                 fontsize=12, fontweight="bold")
    fig.tight_layout(rect=(0, 0, 1, 0.93))
    base = os.path.join(PARAM_DIR, "parameter_count_comparison")
    _diag_save(fig, base)
    return base + ".png"


def plot_parameter_vs_performance(df):
    """Measured INSTANCE performance against parameter count and size."""
    sub = _headline(df)
    sub = sub[np.isfinite(sub["measured_snr_db"])]
    if sub.empty:
        return None
    fig, axes = plt.subplots(2, 2, figsize=(13.2, 9.0))
    axes = axes.ravel()
    panels = [
        ("params_m", "SNR vs parameters", "Parameters (M)", "SNR (dB)"),
        ("params_m", "Coded rate vs parameters", "Parameters (M)", "Total bps"),
        ("params_m", "Compression vs parameters", "Parameters (M)",
         "Compression vs source"),
        ("model_size_mb", "SNR vs model size", "Model size (MB)", "SNR (dB)"),
    ]
    ycols = ["measured_snr_db", "measured_bps", "measured_compression",
             "measured_snr_db"]
    for ax, (xcol, title, xlabel, ylabel), ycol in zip(axes, panels, ycols):
        for series in _series_order(sub):
            d = sub[sub["series"] == series].sort_values(xcol)
            if d.empty:
                continue
            ax.scatter(d[xcol], d[ycol], s=42, alpha=0.8,
                       color=DIAG_COLORS.get(series, "#4B5563"),
                       marker=DIAG_MARKERS.get(series, "o"),
                       label=DIAG_LABELS.get(series, series), edgecolor="white",
                       linewidth=0.4)
        _style_axes(ax, title, xlabel, ylabel)
        ax.legend(fontsize=7, loc="best")
    fig.suptitle(
        f"INSTANCE measurement at depth {DIAG_DEPTH} "
        f"({DIAG_CODER} coder where available) — one point per checkpoint",
        fontsize=12, fontweight="bold")
    fig.tight_layout(rect=(0, 0, 1, 0.94))
    base = os.path.join(PARAM_DIR, "parameter_vs_performance")
    _diag_save(fig, base)
    return base + ".png"


def _preferred_rows(it):
    """One coder per depth for a checkpoint.

    The measurement deliberately writes fixed/adaptive/learned rows for the
    same latents.  Aggregating all three would triple-count a checkpoint and
    mix coder overheads.  Pick the same preference order as the diagnostic
    table, independently at each depth.
    """
    by_depth = {}
    for r in _rate_rows(it):
        d = _row_depth(r)
        if d is None:
            continue
        by_depth.setdefault(d, []).append(r)
    out = []
    for d in sorted(by_depth):
        same = by_depth[d]
        for want in (DIAG_CODER, "learned", "adaptive", "fixed",
                     "topk+exact-positions"):
            cand = [r for r in same if (r.get("coder") or "") == want]
            if cand:
                same = cand
                break
        best = max(same, key=lambda r: _dfnum(r.get("snr_db"), -1e9))
        out.append(best)
    return out


def _all_rate_rows():
    rows = []
    for it in CKPTS:
        for r in _preferred_rows(it):
            r = dict(r)
            r["_series"] = it["series"]
            r["_lam"] = float(it.get("lam") or 0.0)
            r["_cr"] = int(it["cr"])
            r["_d"] = int(it["d"])
            rows.append(r)
    return rows


def _aggregate_depth(rows, series):
    """Median/IQR over widths at each depth for one series."""
    out = {}
    for r in rows:
        if r.get("_series") != series:
            continue
        lam = float(r.get("_lam") or 0.0)
        if series in LEARNED_SERIES and abs(lam - LAM_HEADLINE) > 1e-15:
            continue
        depth = _row_depth(r)
        if depth is None:
            continue
        coder = r.get("coder") or ""
        if coder not in (DIAG_CODER, "learned", "adaptive", "fixed",
                         "topk+exact-positions"):
            continue
        out.setdefault(depth, []).append(r)
    agg = {}
    for depth, rs in out.items():
        snr = np.asarray([_dfnum(r.get("snr_db")) for r in rs])
        bps = np.asarray([_dfnum(r.get("bps_total_amortised")) for r in rs])
        clamp = np.asarray([_dfnum(r.get("pct_clamped")) for r in rs])
        snr, bps, clamp = snr[np.isfinite(snr)], bps[np.isfinite(bps)], clamp[np.isfinite(clamp)]
        if snr.size == 0:
            continue
        agg[depth] = {
            "snr_med": float(np.median(snr)),
            "snr_q25": float(np.percentile(snr, 25)),
            "snr_q75": float(np.percentile(snr, 75)),
            "bps_med": float(np.median(bps)) if bps.size else np.nan,
            "clamp_med": float(np.median(clamp)) if clamp.size else np.nan,
            "clamp_max": float(np.max(clamp)) if clamp.size else np.nan,
        }
    return agg


def plot_depth_sensitivity():
    """SNR, rate, and clamping against target depth, aggregated over widths."""
    rows = _all_rate_rows()
    if not rows:
        return None
    present = {r["_series"] for r in rows}
    series_list = [s for s in (LEARNED_SERIES + BASELINE_NAMES)
                   if s in present]
    fig, axes = plt.subplots(1, 3, figsize=(15.0, 4.6))
    for series in series_list:
        agg = _aggregate_depth(rows, series)
        if not agg:
            continue
        ds = sorted(agg)
        col = DIAG_COLORS.get(series, "#4B5563")
        marker = DIAG_MARKERS.get(series, "o")
        label = DIAG_LABELS.get(series, series)
        snr = np.asarray([agg[d]["snr_med"] for d in ds])
        q25 = np.asarray([agg[d]["snr_q25"] for d in ds])
        q75 = np.asarray([agg[d]["snr_q75"] for d in ds])
        bps = np.asarray([agg[d]["bps_med"] for d in ds])
        clamp = np.asarray([agg[d]["clamp_med"] for d in ds])
        axes[0].plot(ds, snr, marker=marker, color=col, lw=1.6, ms=5,
                     label=label)
        axes[0].fill_between(ds, q25, q75, color=col, alpha=0.12)
        axes[1].plot(ds, bps, marker=marker, color=col, lw=1.6, ms=5,
                     label=label)
        axes[2].plot(ds, clamp, marker=marker, color=col, lw=1.6, ms=5,
                     label=label)
    _style_axes(axes[0], "SNR vs target depth", "Target bits", "SNR (dB)")
    _style_axes(axes[1], "Coded rate vs target depth", "Target bits",
                "Total bps (amortised)")
    _style_axes(axes[2], "Clamped-symbol rate vs target depth", "Target bits",
                "Clamped (%)")
    axes[2].set_ylim(bottom=0)
    for ax in axes:
        ax.legend(fontsize=7)
    fig.suptitle(
        "Depth sensitivity — median across widths; band is the interquartile "
        "range across widths", fontsize=12, fontweight="bold")
    fig.tight_layout(rect=(0, 0, 1, 0.92))
    base = os.path.join(PARAM_DIR, "depth_sensitivity")
    _diag_save(fig, base)
    return base + ".png"


def plot_lambda_sensitivity(df):
    """Measured SNR and coded rate against lambda at one CR."""
    sub = df[df["series"].isin(LEARNED_SERIES)].copy()
    if sub.empty:
        return None
    # Pick the available CR closest to 20 so this remains useful under QUICK.
    cr_avail = sorted(sub["cr"].unique(), key=lambda x: abs(x - 20))
    if not cr_avail:
        return None
    cr = cr_avail[0]
    sub = sub[sub["cr"] == cr]
    fig, axes = plt.subplots(1, 2, figsize=(12.2, 4.4))
    for series in _series_order(sub):
        d = sub[sub["series"] == series].sort_values("lam")
        if d.empty:
            continue
        kw = dict(color=DIAG_COLORS.get(series, "#4B5563"),
                  marker=DIAG_MARKERS.get(series, "o"), ms=5, lw=1.6,
                  label=DIAG_LABELS.get(series, series))
        axes[0].plot(d["lam"], d["measured_snr_db"], **kw)
        axes[1].plot(d["lam"], d["measured_bps"], **kw)
    for ax in axes:
        ax.set_xscale("log")
        ax.legend(fontsize=8)
    _style_axes(axes[0], f"Measured SNR at CR {cr}", "lambda (log)",
                "SNR (dB)")
    _style_axes(axes[1], f"Measured rate at CR {cr}", "lambda (log)",
                "Total bps (amortised)")
    fig.suptitle(
        f"Lambda sensitivity at depth {DIAG_DEPTH} — INSTANCE measurement",
        fontsize=12, fontweight="bold")
    fig.tight_layout(rect=(0, 0, 1, 0.92))
    base = os.path.join(PARAM_DIR, "lambda_sensitivity")
    _diag_save(fig, base)
    return base + ".png"


def plot_clamp_diagnostic():
    """All measured clamp percentages, by series and target depth."""
    rows = _all_rate_rows()
    if not rows:
        return None
    fig, ax = plt.subplots(figsize=(9.2, 5.0))
    for series in (LEARNED_SERIES + BASELINE_NAMES):
        rs = [r for r in rows if r.get("_series") == series]
        if not rs:
            continue
        depth = np.asarray([_row_depth(r) if _row_depth(r) is not None else np.nan
                            for r in rs], dtype=float)
        clamp = np.asarray([_dfnum(r.get("pct_clamped")) for r in rs])
        ok = np.isfinite(depth) & np.isfinite(clamp)
        if not ok.any():
            continue
        ax.scatter(depth[ok], clamp[ok], s=22, alpha=0.55,
                   color=DIAG_COLORS.get(series, "#4B5563"),
                   marker=DIAG_MARKERS.get(series, "o"),
                   label=DIAG_LABELS.get(series, series), edgecolor="none")
    ax.set_ylim(bottom=0)
    _style_axes(ax, "Clamped symbols across the whole measurement grid",
                "Target bits", "Clamped (%)")
    ax.legend(fontsize=7, ncol=2)
    fig.tight_layout()
    base = os.path.join(PARAM_DIR, "clamp_diagnostic")
    _diag_save(fig, base)
    return base + ".png"


# ------------------------------------------------------------------ reconstruction
def _load_diag_model(it):
    blob = torch.load(it["path"], map_location=DEVICE, weights_only=False)
    model = build_model(it["series"], it["cr"]).to(DEVICE)
    model.load_state_dict(blob["state_dict"], strict=True)
    model.eval()
    return model


def _reconstruct(it, depth=DIAG_DEPTH, n_traces=DIAG_TRACES):
    """Quantized reconstruction through the same code path as measurement.

    Returns original (N, 1500), reconstructed (N, 1500), actual depth, and the
    clamped-symbol count for architecture 2 / baselines.
    """
    n = min(int(n_traces), len(eval_np), len(calib_ds))
    if n <= 0:
        raise RuntimeError("no INSTANCE evaluation traces available")
    x_eval = np.asarray(eval_np[:n], dtype=np.float32)
    x_t = torch.as_tensor(x_eval, dtype=torch.float32, device=DEVICE)
    model = _load_diag_model(it)
    z_calib = _calib_latents(model, calib_ds, DEVICE)
    delta, actual_depth, _ = model.set_calibrated_delta(
        z_calib, target_bits=int(depth))
    delta = float(delta)
    arch = int((it.get("meta") or {}).get("arch") or 0)
    n_clamped = None
    n_symbols = 0

    if arch == 1:
        # Architecture 1 sends d quantized values plus their positions.
        with torch.no_grad():
            f_eval = model.encode_coarse(x_t)
        n_slots = int(f_eval.shape[1])
        recs = []
        for t in range(n):
            with torch.no_grad():
                f = f_eval[t:t + 1]
                fm, idx = model.sparse(f)
                q_all = model.quantizer.indices(fm.squeeze(0)).cpu().numpy().astype(np.int64)
                sel = np.sort(idx.squeeze(0).cpu().numpy())
                qv = q_all[sel]
                full = np.zeros(n_slots, dtype=np.float32)
                full[sel] = qv * delta
                y = model.decode_latent(
                    torch.as_tensor(full, device=DEVICE).unsqueeze(0))
            recs.append(y.cpu().numpy()[0, 0])
            n_symbols += int(qv.size)
        recon = np.asarray(recs)
    else:
        with torch.no_grad():
            z_eval = model.encode_coarse(x_t)
            idx_raw = model.quantizer.indices(z_eval).cpu().numpy().astype(np.int64)
            idx, n_clamped, _, _ = clamp_to_alphabet(idx_raw, int(depth))
            n_symbols = int(idx.size)
            z_q = torch.as_tensor(idx, dtype=torch.float32, device=DEVICE) * delta
            y = model.decode_latent(z_q).cpu().numpy()
        recon = y[:, 0, :]

    del model
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return x_eval[:, 0, :], np.asarray(recon), actual_depth, n_clamped, n_symbols


def _per_trace_metrics(original, reconstructed):
    rows = []
    for i in range(original.shape[0]):
        a = np.asarray(original[i], dtype=float).reshape(-1)
        b = np.asarray(reconstructed[i], dtype=float).reshape(-1)
        mse = float(np.mean((a - b) ** 2))
        power = float(np.mean(a ** 2))
        snr = 10.0 * math.log10(power / (mse + 1e-12)) if mse > 0 else float("inf")
        psnr = 20.0 * math.log10(1.0 / math.sqrt(mse)) if mse > 0 else float("inf")
        if np.std(a) > 0 and np.std(b) > 0:
            corr = float(np.corrcoef(a, b)[0, 1])
        else:
            corr = 0.0
        rows.append({"trace": i + 1, "snr_db": snr, "mse": mse,
                     "psnr_db": psnr, "correlation": corr})
    return rows


def plot_waveforms_comprehensive(original, reconstructed, title, base,
                                 arrivals=None):
    """Time, residual, and frequency view of six traces, with P/S markers."""
    n = min(DIAG_TRACES, len(original))
    fig, axes = plt.subplots(n, 3, figsize=(16.0, 2.55 * n), squeeze=False,
                             sharex="col")
    for i in range(n):
        a = np.asarray(original[i]).reshape(-1)
        b = np.asarray(reconstructed[i]).reshape(-1)
        m = min(len(a), len(b))
        a, b = a[:m], b[:m]
        axes[i, 0].plot(a, color="#1A56DB", lw=1.2, label="original")
        axes[i, 0].plot(b, color="#C2410C", lw=1.2, label="reconstructed")
        axes[i, 0].set_ylabel("Amplitude", fontsize=8)
        axes[i, 0].legend(fontsize=7, loc="best")
        axes[i, 0].grid(True, alpha=0.22)
        if i == 0:
            axes[i, 0].set_title("Time domain", fontsize=10, fontweight="bold")

        axes[i, 1].plot(a - b, color="#111827", lw=1.0)
        axes[i, 1].axhline(0, color="#9CA3AF", lw=0.8)
        axes[i, 1].set_ylabel("Residual", fontsize=8)
        axes[i, 1].grid(True, alpha=0.22)
        if i == 0:
            axes[i, 1].set_title("Original minus reconstruction", fontsize=10,
                                 fontweight="bold")

        arr = arrivals[i] if (arrivals is not None and i < len(arrivals)) else None
        if arr is not None:
            for ax in (axes[i, 0], axes[i, 1]):
                for key, col in (("p_rel", "#7C3AED"), ("s_rel", "#0F766E")):
                    v = arr.get(key)
                    if v is not None and np.isfinite(v) and 0 <= v < m:
                        ax.axvline(float(v), color=col, lw=1.0, alpha=0.8)
            pr, sr = arr.get("p_rel"), arr.get("s_rel")
            axes[i, 0].text(
                0.01, 0.96,
                f"P rel {pr:.0f}" if pr is not None and np.isfinite(pr)
                else "P rel NA",
                transform=axes[i, 0].transAxes, fontsize=6.5,
                color="#7C3AED", va="top")
            axes[i, 0].text(
                0.01, 0.86,
                (f"S rel {sr:.0f}" if sr is not None and np.isfinite(sr)
                 else "S rel NA"),
                transform=axes[i, 0].transAxes, fontsize=6.5,
                color="#0F766E", va="top")

        freq = np.fft.rfftfreq(m, d=1.0)
        axes[i, 2].semilogy(freq, np.abs(np.fft.rfft(a)) + 1e-12,
                            color="#1A56DB", lw=1.1, label="original")
        axes[i, 2].semilogy(freq, np.abs(np.fft.rfft(b)) + 1e-12,
                            color="#C2410C", lw=1.1, label="reconstructed")
        axes[i, 2].set_ylabel("Magnitude", fontsize=8)
        axes[i, 2].legend(fontsize=7, loc="best")
        axes[i, 2].grid(True, alpha=0.22, which="both")
        if i == 0:
            axes[i, 2].set_title("Frequency spectrum", fontsize=10,
                                 fontweight="bold")

        axes[i, 0].text(0.01, 0.02, f"trace {i + 1}", transform=axes[i, 0].transAxes,
                        fontsize=7, color="#4B5563", va="bottom")
    for ax in axes[-1]:
        ax.set_xlabel("Sample" if ax is not axes[-1][2] else "Normalized frequency",
                      fontsize=8)
    fig.suptitle(title, fontsize=12, fontweight="bold")
    fig.tight_layout(rect=(0, 0, 1, 0.94))
    _diag_save(fig, base)
    return base + ".png"


def plot_per_trace_snr(per_trace):
    if not per_trace:
        return None
    df = pd.DataFrame(per_trace)
    labels = list(dict.fromkeys(df["case"]))
    data = [df[df["case"] == c]["snr_db"].to_numpy() for c in labels]
    fig, ax = plt.subplots(figsize=(10.5, 4.8))
    ax.boxplot(data, showfliers=True, patch_artist=True,
               boxprops=dict(facecolor="#DBEAFE", edgecolor="#1A56DB"),
               medianprops=dict(color="#C2410C", lw=1.5))
    ax.set_xticks(range(1, len(labels) + 1))
    ax.set_xticklabels(labels, rotation=30, ha="right", fontsize=8)
    ax.set_ylabel("Per-trace SNR (dB)")
    ax.set_title("Reconstruction SNR distribution for the plotted cases",
                 fontsize=12, fontweight="bold")
    ax.grid(True, alpha=0.25, axis="y")
    fig.tight_layout()
    base = os.path.join(REC_DIR, "per_trace_snr")
    _diag_save(fig, base)
    return base + ".png"


def _case_checkpoints(df):
    """Small named set: each series at CR 20 and CR 100 when present."""
    cases = []
    for series in _series_order(df):
        d = df[df["series"] == series]
        if series in LEARNED_SERIES:
            d = d[np.isclose(d["lam"].astype(float), LAM_HEADLINE)]
        for target in DIAG_CRS:
            if d.empty:
                continue
            cr = min(d["cr"].unique(), key=lambda x: abs(int(x) - target))
            hit = d[d["cr"] == cr]
            if hit.empty:
                continue
            row = hit.iloc[0]
            it = next((x for x in CKPTS
                       if x["series"] == row["series"] and x["cr"] == int(row["cr"])
                       and abs(float(x.get("lam") or 0.0)
                               - float(row["lam"])) < 1e-15), None)
            if it is not None:
                cases.append(it)
    # Deduplicate (QUICK can map both targets to the same CR).
    seen, out = set(), []
    for it in cases:
        key = (it["series"], it["cr"], float(it.get("lam") or 0.0))
        if key not in seen:
            seen.add(key)
            out.append(it)
    return out


def make_reconstruction_diagnostics(df):
    if not DO_RECONSTRUCTION_DIAGNOSTICS:
        print("  reconstruction diagnostics disabled")
        return []
    cases = _case_checkpoints(df)
    if not cases:
        print("  no checkpoint available for reconstruction diagnostics")
        return []
    made, per_trace = [], []
    for it in cases:
        try:
            original, recon, actual_depth, n_clamped, n_symbols = _reconstruct(it)
            metrics = _per_trace_metrics(original, recon)
            case = (f"{it['series']} CR{it['cr']} d{it['d']} "
                    f"lam{float(it.get('lam') or 0):g}")
            clamped_pct = (100.0 * n_clamped / n_symbols
                           if (n_clamped is not None and n_symbols) else np.nan)
            arrivals = eval_trace_meta[:len(metrics)]
            for i, row in enumerate(metrics):
                am = arrivals[i] if i < len(arrivals) else {}
                row.update({
                    "case": case,
                    "series": it["series"], "cr": int(it["cr"]),
                    "d": int(it["d"]), "lam": float(it.get("lam") or 0.0),
                    "depth": int(actual_depth),
                    "n_clamped": n_clamped,
                    "n_symbols": int(n_symbols),
                    "clamped_pct": clamped_pct,
                    "window_start": am.get("start", ""),
                    "p_rel": am.get("p_rel", ""),
                    "s_rel": am.get("s_rel", ""),
                    "s_inside": am.get("s_inside", ""),
                    "protocol_hash": PROTOCOL_HASH,
                    "segment_mode": SEGMENT_MODE,
                    "n_calib_used": N_CALIB,
                    "n_eval_used": N_EVAL,
                })
                per_trace.append(row)
            title = (f"{DIAG_LABELS.get(it['series'], it['series'])}  "
                     f"CR={it['cr']}  depth={actual_depth}  "
                     f"lambda={float(it.get('lam') or 0):g}  —  P-aligned "
                     f"INSTANCE eval traces")
            base = os.path.join(
                REC_DIR,
                f"{_diag_slug(it['series'])}_cr{it['cr']}_d{it['d']}_"
                f"lam{float(it.get('lam') or 0):g}_waveforms")
            made.append(plot_waveforms_comprehensive(
                original, recon, title, base, arrivals=arrivals))
            clamp_txt = ("n/a" if n_clamped is None
                         else f"{clamped_pct:.4f}%")
            print(f"  [reconstruction] {case}  "
                  f"mean SNR {np.mean([m['snr_db'] for m in metrics]):.2f} dB  "
                  f"clamped {clamp_txt}")
        except Exception as e:
            print(f"  [reconstruction FAIL] {it['series']} CR={it['cr']}: {e}")
            traceback.print_exc(limit=1)
    if per_trace:
        p = os.path.join(DIAG_DIR, "per_trace_metrics.csv")
        pd.DataFrame(per_trace).to_csv(p, index=False, encoding="utf-8")
        print(f"  [saved] {p}  ({len(per_trace)} rows)")
        made.append(plot_per_trace_snr(per_trace))
    return [p for p in made if p]


def _count_figures():
    return sorted(glob.glob(os.path.join(FIG_DIR, "**", "*.png"),
                            recursive=True))


def make_extended_diagnostics():
    """Run the adapted Final_LARA analysis section, never aborting the run."""
    if not DO_EXTENDED_DIAGNOSTICS:
        print("\n### extended diagnostics disabled")
        return {"ok": False, "disabled": True, "failures": []}
    print("\n### extended diagnostics")
    t0 = time.time()
    failures = []
    made = []
    df = pd.DataFrame()
    try:
        df = build_diagnostic_table()
    except Exception as e:
        failures.append(("build_diagnostic_table", repr(e)))
        print("  [FAIL] diagnostic table:", e)
        traceback.print_exc(limit=2)

    # Learning curves from the training histories.
    try:
        if not df.empty:
            made += [p for p in (
                plot_training_summary(df),
                plot_parameter_comparison(df),
                plot_parameter_vs_performance(df),
                plot_lambda_sensitivity(df),
            ) if p]
    except Exception as e:
        failures.append(("parameter_analysis", repr(e)))
        print("  [FAIL] parameter analysis:", e)
        traceback.print_exc(limit=2)

    for fn, name in ((plot_learning_overview, "learning_overview"),
                     (plot_learning_baselines, "learning_baselines"),
                     (plot_learning_lambda, "learning_lambda"),
                     (plot_depth_sensitivity, "depth_sensitivity"),
                     (plot_clamp_diagnostic, "clamp_diagnostic")):
        try:
            if name in ("learning_overview", "learning_baselines",
                        "learning_lambda"):
                made += fn() or []
            else:
                p = fn()
                if p:
                    made.append(p)
        except Exception as e:
            failures.append((name, repr(e)))
            print(f"  [FAIL] {name}:", e)
            traceback.print_exc(limit=2)

    # Selected per-checkpoint training-history figures (not all 110).
    try:
        if not df.empty:
            for series in LEARNED_SERIES:
                d = df[(df["series"] == series)
                       & np.isclose(df["lam"].astype(float), LAM_HEADLINE)]
                if d.empty:
                    continue
                # One wide and one narrow checkpoint per series.
                d = d.sort_values("d")
                for pick in {d.iloc[0]["d"], d.iloc[-1]["d"]}:
                    row = d[d["d"] == pick].iloc[0]
                    it = next((x for x in CKPTS
                               if x["series"] == series
                               and x["cr"] == int(row["cr"])
                               and abs(float(x.get("lam") or 0.0)
                                       - float(row["lam"])) < 1e-15), None)
                    if it is not None:
                        p = plot_training_history(it)
                        if p:
                            made.append(p)
    except Exception as e:
        failures.append(("selected_training_histories", repr(e)))
        print("  [FAIL] selected training histories:", e)
        traceback.print_exc(limit=2)

    # Reconstructions are the only extra inference pass.
    try:
        if not df.empty:
            made += make_reconstruction_diagnostics(df)
    except Exception as e:
        failures.append(("reconstruction", repr(e)))
        print("  [FAIL] reconstruction diagnostics:", e)
        traceback.print_exc(limit=2)

    figs = _count_figures()
    actual_recon_traces = int(min(DIAG_TRACES, len(eval_np), len(calib_ds)))
    manifest = {
        "generated": time.strftime("%Y-%m-%d %H:%M:%S"),
        "ok": not failures,
        "diagnostic_rows": int(len(df)),
        "figures_total": len(figs),
        "figures": [os.path.relpath(p, FIG_DIR).replace(os.sep, "/")
                    for p in figs],
        "diagnostic_csv": os.path.join(DIAG_DIR, "checkpoint_analysis.csv"),
        "per_trace_csv": os.path.join(DIAG_DIR, "per_trace_metrics.csv")
                         if os.path.isfile(os.path.join(DIAG_DIR,
                                                        "per_trace_metrics.csv"))
                         else None,
        "depth": DIAG_DEPTH,
        "coder": DIAG_CODER,
        "reconstruction_traces": actual_recon_traces,
        "reconstruction_traces_requested": DIAG_TRACES,
        "reconstruction_traces_used": actual_recon_traces,
        "reconstruction_crs": DIAG_CRS,
        "protocol_hash": PROTOCOL_HASH,
        "segment_mode": SEGMENT_MODE,
        "p_offset": P_OFFSET,
        "pilot": bool(PILOT),
        "window_ok": bool(globals().get("WINDOW_OK", False)),
        "window_stats": globals().get("WINDOW_STATS", {}),
        "n_calib_used": N_CALIB,
        "n_eval_used": N_EVAL,
        "failures": failures,
        "elapsed_min": round((time.time() - t0) / 60.0, 2),
    }
    p = os.path.join(DIAG_DIR, "diagnostics_manifest.json")
    with open(p, "w", encoding="utf-8") as fh:
        json.dump(manifest, fh, indent=1)
    print(f"  [saved] {p}")
    print(f"  diagnostic rows {len(df)}  figures {len(figs)}  "
          f"failures {len(failures)}  elapsed {manifest['elapsed_min']:.1f} min")
    return manifest


## Run the validation


In [ ]:
"""Run the whole validation and write the figures and acceptance record.

The measurement is resumable at the CSV level: rerunning this cell skips a
checkpoint only when its rate CSV carries the current ``protocol_hash``.  A CSV
from a different data definition is re-measured, not reused.  Set
``FORCE = True`` (or ``LARA_FORCE=1``) to redo everything.
"""
RUN_FAILURES = run_measurements()

try:
    make_all_figures()
except Exception as _e:
    print("figures skipped:", _e)
    traceback.print_exc(limit=2)

# Extended analysis: training histories, parameter table, and a small,
# named set of reconstruction examples.  It is deliberately wrapped so a
# diagnostic failure cannot invalidate the measurement run.
try:
    EXTENDED = make_extended_diagnostics()
except Exception as _e:
    EXTENDED = {"ok": False, "diagnostic_rows": 0,
                "failures": [["extended_diagnostics", repr(_e)]]}
    print("extended diagnostics skipped:", _e)
    traceback.print_exc(limit=2)

try:
    ACCEPTANCE = write_summary()
except Exception as _e:
    # Never leave an older COMPLETE acceptance record looking current after a
    # failed summary.  The error record is written before the exception is
    # re-raised, so the output folder cannot be mistaken for a valid run.
    ACCEPTANCE = {
        "generated": time.strftime("%Y-%m-%d %H:%M:%S"),
        "status": "ERROR",
        "error": repr(_e),
        "acceptance_file": ACCEPTANCE_FILE,
        "pilot": bool(PILOT),
        "protocol_hash": PROTOCOL_HASH,
        "segment_mode": SEGMENT_MODE,
        "p_offset": P_OFFSET,
        "window_ok": bool(globals().get("WINDOW_OK", False)),
        "window_stats": globals().get("WINDOW_STATS", {}),
        "n_calib_used": N_CALIB,
        "n_eval_used": N_EVAL,
    }
    with open(os.path.join(OUT_DIR, ACCEPTANCE_FILE), "w",
              encoding="utf-8") as _fh:
        json.dump(ACCEPTANCE, _fh, indent=1)
    print("acceptance ERROR:", _e)
    raise


## Caveats and how to read the numbers

**The checkpoints were trained on STEAD.** This notebook measures them on
INSTANCE; it does not retrain them. A model trained on STEAD is being evaluated
out of domain, which is the point of the validation, but it is not a claim that
the same model would be optimal if trained on INSTANCE.

**Training curves are not INSTANCE learning curves.** The extended diagnostics
read the STEAD training histories in `ckpts_baselines/history_<tag>.json`. They
show how the 110 checkpoints were trained, not how a model learns on INSTANCE.
The INSTANCE curves in this notebook are measurement curves (SNR and coded rate
against compression ratio, depth and lambda), not training curves.

**Δ is measured, not inherited.** The `delta` and `latent_span` columns in the
CSVs are computed from INSTANCE calibration latents at measurement time. If you
compare them with `meta.calib` in a checkpoint you will see different numbers;
that is expected. `meta.calib` was written before training and is not a
measurement-time calibration.

**The window is P-aligned, fixed, and gated.** `start = clip(P - 500, 0,
len(trace) - 1500)`, then 1500 samples, then the study's per-trace min–max. P is
required; S is not filtered. The P/S coverage gate is what prevents another
pre-event run: P must be at least 99 % and S at least 80 % (provisional) on both
the calibration and scored sets. A failed gate is `INVALID_WINDOW`
or `PILOT_INVALID_WINDOW`, never `COMPLETE`.

**The old first-1500 run is pre-event evidence.** On every selected trace its
window ended before P. It is kept unchanged in
`instance_validation_20102026_first1500/`; its Q2/Q1 rankings are not a valid
STEAD-vs-INSTANCE comparison.

**A run can be complete and still invalid for comparison.** If the calibration
or scored set is smaller than the study's 2000 / 200 targets, the 1c policy
still writes every output, but `acceptance.json` says `INVALID_LOW_COUNT`. Do
not interpret that run as a protocol-sized result.

**Stale CSVs cannot leak in.** Every CSV carries a `protocol_hash`. The summary
and figures keep only rows matching the current hash; a CSV left by another
protocol is counted as stale and ignored. The driver re-measures it rather than
skipping on mere file existence.

**No bandpass and no interpolation.** The previous validation applied a 1–40 Hz
bandpass and a pre-P window. Neither is part of this protocol. The only
transform is the fixed P-aligned crop followed by the study's per-trace min–max.

**Architecture 1 is charged exact positions.** `LARA-Q1` and `Light_HARA-Q1`
transmit `d` values plus `log2 C(B*T, d)` position bits. The position coder is
verified by round trip; the small overhead over the combinatorial floor is byte
alignment, not a modelling choice.

**The bit-accurate wavelet arm is fitted on calibration traces, never on the
scored set.** Its band maxima and step sizes come from the first 64 calibration
traces. The scored traces are used only to measure SNR and coded rate.

**The legacy Final_LARA wavelet is idealized.** It keeps the top `K = 1500/CR`
coefficients using the old `abs(c) > sorted_abs[K]` rule, then reconstructs. It
does not encode positions, signs, quantised magnitudes or an entropy stream, so
it has no measured bps. Its `Wavelet_DB4`, `Wavelet_SYM8`, `Wavelet_COIF3`
curves appear only on the nominal-CR figure. They never replace the
bit-accurate wavelet arm and are never plotted against a coded-bps axis.

**The two SNR figures use different x-axes.** `snr_vs_cr.png` uses nominal CR and
contains learned models, baselines and the legacy wavelet. `snr_vs_measured_cr.png`
uses `x = 32 / bps_total_amortised` and contains learned models, baselines, the
bit-accurate wavelet and `Arch3Classical`. Do not mix the two CR definitions.

**CPU is for smoke tests only.** The arithmetic coder is Python and the wavelet
search is a (depth, K) sweep. With `ALLOW_CPU = True` the notebook forces
`QUICK` and prints a warning; the full 110-checkpoint grid is a GPU-host job.

**A missing checkpoint is reported, not hidden.** The inventory builds the
study grid explicitly and verifies each filename against its `meta` (`cr`,
`latent_dim`, `lam`, `arch`) before `strict=True` loading. The acceptance record
reports `found / expected`; a partial folder produces a `PARTIAL` status rather
than a silently shorter comparison.
